# Gemma backbone — V4.2.1 / native o200k

One **20-minute / 1,200-second** run on **one TPU v5e-1**. Use the same
Gemma3_270M transformer backbone with native **tiktoken `o200k_base`**.
Run the notebook top to bottom. No earlier notebook is replaced.

| Setting | Value |
|---|---:|
| Short training batch | **3 ×1,024 =3,072 positions/update** |
| Long training batch | **1 ×2,048 =2,048 positions/update** |
| Batch discovery / ramp / later rechecks | **Disabled** |
| Training budget | **1,200 seconds** |
| Context cycle | **3M short /1M long scored tokens**, repeating |
| Frozen training corpus | **256 MiB** |
| Defined native vocabulary IDs | **200,000** |
| Additional packing PAD/BOS/EOS | **3** |
| Tied embedding/output table | **200,003 ×640** |
| Vocabulary parameters | **128,001,920** |
| Transformer/norm parameters | **100,326,016** |
| Total parameters | **228,327,936** |

Native o200k has **199,998 mergeable tokens**, **2 special tokens** and **19
unused IDs** in its 200,019-ID native address range. Densely map only the
200,000 defined IDs to 1..200000. Model PAD=0, BOS=200001 and EOS=200002.
Use the existing virtual packing tokens and tied output head. The smaller
vocabulary reduces the total by **39,772,800 parameters versus V4**; all
transformer layers and their initialization checks stay the same.

Use [OpenAI's native tiktoken](https://github.com/openai/tiktoken/blob/main/tiktoken_ext/openai_public.py),
pinned to **0.14.0**, with **regex 2026.9.29**. Verify the official o200k rank
artifact and the project's full pattern/ranks/special-token snapshot by SHA256.
This is native byte-level BPE. Raw documents use `encode_ordinary`, so literal
special-token spellings remain ordinary text and no chat template is applied.
The exported `tokenizer/o200k_base.json` contains the native encoding arguments;
it is data for `tiktoken.Encoding`, rather than a Hugging Face tokenizer file.

Reuse the frozen raw corpus and holdouts from Drive. Encode each document
once; reconstruct its exact UTF-8 bytes from native token byte pieces. Attribute
each token its actual byte length, including tokens that split a Unicode
character. Then build the 1,024/2,048 packings sequentially. Native encoding
and packing cache identities are separate from all Gemma caches. Matching
completed o200k caches can be reused on subsequent runs.

Compile **only 3 ×1,024 and 1 ×2,048**, plus the held-out evaluator, before
training. Check the two requested shapes against the existing **85% compiler
memory budget**; a rejected shape stops the run instead of changing the batch.
There are no candidate shapes or training-time compilations. The first update
uses 3 ×1,024. Repeat short-token intervals 0–3M, 4–7M, 8–11M, and long intervals
3–4M, 7–8M, 11–12M, continuing until the timer ends. Switch after the complete
update crossing each boundary, preserving model/Adam state and context cursors.

Keep V4's BF16 computation, FP32 master weights, native dense vocabulary loss,
AdamW **0.0003**, **5** warmup updates, clipping **1.0** and decay **0.01**.
Initialize fresh random weights with the same seed/backbone fingerprint.
Keep initial/final held-out BPB at context 1,024, 16-language and legacy
12-language scores, memory accounting and one mixed post-save profile at the
same two fixed shapes. Missing hardware counters remain explicitly unavailable.
BPB and original bytes/s are useful comparisons across tokenizers; token loss,
token counts and tokens/s reflect different token units.

The environment retains the existing training package pins and adds the two
native tokenizer dependencies in a separate versioned environment. HF_TOKEN
is still used for source data including Stack-smol; no Gemma tokenizer/model
download is required. Save one FP32 weights-only artifact; no restoration check.
No device tests or training benchmarks are run during authoring.


# V4.2.1 engineering contract

- Use native tiktoken 0.14.0 `o200k_base`, with regex 2026.9.29. Verify ranks
  SHA256 `446a9538cb6c348e3516120d7c08b09f57c36495e2acfffe59a5bf8b0cfb1a2d`
  and full exported asset SHA256
  `97973057009b00298ee5c575a23b895c912518ded47f9dc1559032e9f2532187`.
- Densely map the 200000 defined IDs. Exclude the 19 unused native IDs. Add
  protocol PAD/BOS/EOS separately; tied table 200003 ×640, total 228327936.
- Preserve Gemma3_270M transformer layers, native loss, optimizer math, shared
  initialization fingerprint and all prior training package pins.
- Raw text uses encode_ordinary with no automatic special tokens. Verify
  native token-byte concatenation equals original UTF-8. Assign exact native
  token byte lengths; never infer byte counts from character offsets.
- Preserve raw/source identities and encode-once sequential packing. Namespace
  the changed native byte-accounting caches so Gemma token arrays cannot match.
- Fixed short batch 3 at length 1024 and long batch 1 at length 2048, from the
  beginning. Remove batch discovery, ramps and candidate compilation entirely.
- Compile the two exact shapes and the fixed evaluator before training.
  Apply 85% memory admission to those shapes. Never silently substitute batches.
- Train 1200 seconds; repeat 3M-short/1M-long cumulative scored-token intervals.
  Preserve Adam/model state, update counters and independent context row cursors.
- Keep initial/final evaluation, source-byte exposure, memory and profile
  reporting. Profiling uses only the same two shapes and disposable weights.
- Save weights only; no checkpoint restoration. Preserve failed earlier runs.
- Create a new notebook exclusively. CPU tokenizer/data checks and static
  notebook checks are permitted; no accelerator/device tests or model training.


## 1. Connect Drive and reuse the pinned environment
No model update or TPU initialization occurs here. A new run folder is created.


In [ ]:
REQUIREMENTS_INPUT = '\n# This file was autogenerated by uv via the following command:\n#    uv pip compile /tmp/gemma-colab-requirements.in --python-version 3.12 --python-platform x86_64-manylinux_2_35 -o /tmp/gemma-colab-requirements.lock\nabsl-py==2.5.1\n    # via\n    #   array-record\n    #   chex\n    #   clu\n    #   dm-tree\n    #   etils\n    #   gemma\n    #   grain\n    #   kauldron\n    #   keras\n    #   ml-collections\n    #   optax\n    #   orbax-checkpoint\n    #   tensorboard\n    #   tensorflow\n    #   tensorflow-datasets\n    #   tensorflow-metadata\n    #   xmanager\naiofile==3.12.3\n    # via py-key-value-aio\naiofiles==25.1.0\n    # via orbax-checkpoint\naiohappyeyeballs==2.7.1\n    # via aiohttp\naiohttp==3.14.5\n    # via\n    #   aiohttp-retry\n    #   cloud-sql-python-connector\n    #   fsspec\n    #   kubernetes\naiohttp-retry==2.9.1\n    # via kubernetes\naiosignal==1.4.0\n    # via aiohttp\nalembic==1.20.0\n    # via xmanager\naltair==6.3.0\n    # via kauldron\nannotated-types==0.8.0\n    # via pydantic\nanyio==4.15.1\n    # via\n    #   google-genai\n    #   httpx\n    #   httpx2\n    #   mcp\n    #   py-key-value-aio\n    #   sse-starlette\n    #   starlette\n    #   watchfiles\nanywidget==0.11.0\n    # via dialog\narray-record==0.8.6\n    # via\n    #   grain\n    #   tensorflow-datasets\nasttokens==3.0.2\n    # via stack-data\nastunparse==1.6.3\n    # via tensorflow\nasync-generator==1.10\n    # via xmanager\nattrs==26.1.0\n    # via\n    #   aiohttp\n    #   cyclopts\n    #   dm-tree\n    #   jsonschema\n    #   jsonschema-path\n    #   referencing\n    #   xmanager\nauthlib==1.9.1\n    # via fastmcp-slim\nbeartype==0.23.1\n    # via py-key-value-aio\ncachetools==7.2.4\n    # via py-key-value-aio\ncaio==0.12.9\n    # via aiofile\ncertifi==2026.10.19\n    # via\n    #   fastmcp-slim\n    #   google-cloud-aiplatform\n    #   httpcore\n    #   httpx\n    #   kubernetes\n    #   requests\ncffi==2.1.1\n    # via cryptography\ncharset-normalizer==3.5.2\n    # via requests\nchex==0.1.92\n    # via kauldron\nclick==8.5.0\n    # via\n    #   huggingface-hub\n    #   uvicorn\ncloud-sql-python-connector==1.23.0\n    # via xmanager\ncloudpickle==3.1.2\n    # via\n    #   grain\n    #   joblib\nclu==0.0.12\n    # via kauldron\ncomm==0.2.3\n    # via ipywidgets\ncontourpy==1.4.0\n    # via matplotlib\ncryptography==50.0.2\n    # via\n    #   authlib\n    #   cloud-sql-python-connector\n    #   google-auth\n    #   joserfc\n    #   pyjwt\n    #   secretstorage\ncycler==0.12.1\n    # via matplotlib\ncyclopts==5.2.0\n    # via fastmcp-slim\ndatasets==5.1.0\n    # via -r /tmp/gemma-colab-requirements.in\ndialog==1.1.0\n    # via gemma\ndill==0.4.1\n    # via\n    #   datasets\n    #   multiprocess\ndistro==1.9.0\n    # via google-genai\ndm-tree==0.1.10\n    # via tensorflow-datasets\ndnspython==2.9.0\n    # via\n    #   cloud-sql-python-connector\n    #   email-validator\ndocker==7.2.0\n    # via xmanager\ndocstring-parser==0.18.0\n    # via\n    #   cyclopts\n    #   google-cloud-aiplatform\n    #   simple-parsing\ndurationpy==0.11\n    # via kubernetes\neinops==0.8.2\n    # via\n    #   etils\n    #   gemma\n    #   kauldron\nemail-validator==2.3.0\n    # via pydantic\netils==1.14.0\n    # via\n    #   array-record\n    #   clu\n    #   dialog\n    #   gemma\n    #   grain\n    #   kauldron\n    #   orbax-checkpoint\n    #   tensorflow-datasets\n    #   xmanager\nexceptiongroup==1.3.1\n    # via fastmcp-slim\nexecuting==2.3.0\n    # via stack-data\nfastmcp==4.1.3\n    # via gemma\nfastmcp-slim==4.1.3\n    # via fastmcp\nfilelock==4.2.4\n    # via\n    #   datasets\n    #   huggingface-hub\nflatbuffers==25.12.19\n    # via tensorflow\nflax==0.12.10\n    # via\n    #   -r /tmp/gemma-colab-requirements.in\n    #   clu\n    #   gemma\n    #   kauldron\nfonttools==4.67.0\n    # via matplotlib\nfrozenlist==1.8.0\n    # via\n    #   aiohttp\n    #   aiosignal\nfsspec==2026.7.0\n    # via\n    #   datasets\n    #   etils\n    #   huggingface-hub\ngast==0.7.0\n    # via tensorflow\ngemma==4.0.1\n    # via -r /tmp/gemma-colab-requirements.in\ngitdb==4.0.12\n    # via gitpython\ngitpython==3.2.1\n    # via xmanager\ngoogle-api-core==2.42.0\n    # via\n    #   google-api-python-client\n    #   google-cloud-aiplatform\n    #   google-cloud-bigquery\n    #   google-cloud-core\n    #   google-cloud-resource-manager\n    #   google-cloud-sql\n    #   google-cloud-storage\n    #   xmanager\ngoogle-api-python-client==2.201.0\n    # via xmanager\ngoogle-auth==2.62.0\n    # via\n    #   cloud-sql-python-connector\n    #   google-api-core\n    #   google-api-python-client\n    #   google-auth-httplib2\n    #   google-cloud-aiplatform\n    #   google-cloud-bigquery\n    #   google-cloud-core\n    #   google-cloud-resource-manager\n    #   google-cloud-sql\n    #   google-cloud-storage\n    #   google-genai\n    #   xmanager\ngoogle-auth-httplib2==0.4.4\n    # via google-api-python-client\ngoogle-cloud-aiplatform==2.6.0\n    # via xmanager\ngoogle-cloud-bigquery==3.46.2\n    # via google-cloud-aiplatform\ngoogle-cloud-core==2.8.0\n    # via\n    #   google-cloud-bigquery\n    #   google-cloud-storage\ngoogle-cloud-resource-manager==1.19.0\n    # via google-cloud-aiplatform\ngoogle-cloud-sql==0.1.1\n    # via cloud-sql-python-connector\ngoogle-cloud-storage==3.17.1\n    # via\n    #   google-cloud-aiplatform\n    #   xmanager\ngoogle-crc32c==1.9.0\n    # via\n    #   google-cloud-storage\n    #   google-resumable-media\ngoogle-genai==2.32.0\n    # via google-cloud-aiplatform\ngoogle-pasta==0.2.0\n    # via tensorflow\ngoogle-resumable-media==2.11.0\n    # via\n    #   google-cloud-bigquery\n    #   google-cloud-storage\ngoogleapis-common-protos==1.75.5\n    # via\n    #   google-api-core\n    #   grpc-google-iam-v1\n    #   grpcio-status\n    #   tensorflow-metadata\n    #   xmanager\ngrain==0.2.18\n    # via\n    #   gemma\n    #   kauldron\ngraphviz==0.21\n    # via kauldron\ngriffelib==2.4.0\n    # via fastmcp-slim\ngrpc-google-iam-v1==0.14.5\n    # via google-cloud-resource-manager\ngrpcio==1.84.0\n    # via\n    #   google-api-core\n    #   google-cloud-resource-manager\n    #   google-cloud-sql\n    #   googleapis-common-protos\n    #   grpc-google-iam-v1\n    #   grpcio-status\n    #   tensorboard\n    #   tensorflow\n    #   xmanager\ngrpcio-status==1.84.0\n    # via google-api-core\nh11==0.16.0\n    # via\n    #   httpcore\n    #   httpcore2\n    #   uvicorn\nh5py==3.14.0\n    # via\n    #   keras\n    #   tensorflow\nhf-xet==1.7.0\n    # via huggingface-hub\nhttpcore==1.0.9\n    # via httpx\nhttpcore2==2.16.0\n    # via httpx2\nhttplib2==0.32.0\n    # via\n    #   google-api-python-client\n    #   google-auth-httplib2\nhttpx==0.28.1\n    # via google-genai\nhttpx2==2.16.0\n    # via\n    #   fastmcp-slim\n    #   huggingface-hub\n    #   mcp\nhuggingface-hub==2.2.0\n    # via\n    #   -r /tmp/gemma-colab-requirements.in\n    #   datasets\n    #   tokenizers\nhumanize==4.16.0\n    # via\n    #   orbax-checkpoint\n    #   xmanager\nidna==3.20\n    # via\n    #   anyio\n    #   email-validator\n    #   httpx\n    #   httpx2\n    #   requests\n    #   yarl\nimageio==2.38.1\n    # via scikit-image\nimmutabledict==4.3.1\n    # via\n    #   kauldron\n    #   tensorflow-datasets\n    #   xmanager\nipython==9.17.1\n    # via\n    #   ipywidgets\n    #   mediapy\nipython-pygments-lexers==1.1.1\n    # via ipython\nipywidgets==8.1.9\n    # via anywidget\njaraco-classes==3.4.0\n    # via keyring\njaraco-context==6.1.2\n    # via keyring\njaraco-functools==4.6.0\n    # via keyring\njax==0.12.0\n    # via\n    #   -r /tmp/gemma-colab-requirements.in\n    #   chex\n    #   clu\n    #   flax\n    #   gemma\n    #   kauldron\n    #   optax\n    #   orbax-checkpoint\njaxlib==0.12.0\n    # via\n    #   -r /tmp/gemma-colab-requirements.in\n    #   chex\n    #   clu\n    #   jax\n    #   optax\njaxtyping==0.3.11\n    # via\n    #   gemma\n    #   kauldron\njedi==0.20.1\n    # via ipython\njeepney==0.9.0\n    # via\n    #   keyring\n    #   secretstorage\njinja2==3.1.6\n    # via altair\njoblib==1.6.0\n    # via scikit-learn\njoserfc==1.7.7\n    # via\n    #   authlib\n    #   fastmcp-slim\njsonref==1.1.0\n    # via fastmcp-slim\njsonschema==4.26.0\n    # via\n    #   altair\n    #   mcp\njsonschema-path==0.5.0\n    # via fastmcp-slim\njsonschema-specifications==2025.9.1\n    # via jsonschema\njupyterlab-widgets==3.0.17\n    # via ipywidgets\nkauldron==1.4.2\n    # via\n    #   -r /tmp/gemma-colab-requirements.in\n    #   gemma\nkeras==3.15.1\n    # via tensorflow\nkeyring==25.7.0\n    # via py-key-value-aio\nkiwisolver==1.5.1\n    # via matplotlib\nkubernetes==37.0.1\n    # via xmanager\nlark==1.3.1\n    # via\n    #   dialog\n    #   kauldron\nlazy-loader==0.6\n    # via scikit-image\nlibclang==18.1.1\n    # via tensorflow\nlibtpu==0.0.50\n    # via jax\nmako==1.4.3\n    # via alembic\nmarkdown==3.11.2\n    # via tensorboard\nmarkdown-it-py==4.2.0\n    # via rich\nmarkupsafe==3.0.4\n    # via\n    #   jinja2\n    #   mako\n    #   werkzeug\nmatplotlib==3.11.2\n    # via mediapy\nmatplotlib-inline==0.2.2\n    # via ipython\nmcp==2.3.0\n    # via\n    #   dialog\n    #   fastmcp-slim\n    #   gemma\nmcp-types==2.3.0\n    # via\n    #   fastmcp-slim\n    #   mcp\nmdurl==0.1.2\n    # via markdown-it-py\nmediapy==1.2.7\n    # via kauldron\nml-collections==1.1.0\n    # via\n    #   clu\n    #   kauldron\nml-dtypes==0.6.0\n    # via\n    #   jax\n    #   jaxlib\n    #   keras\n    #   tensorflow\n    #   tensorstore\nmore-itertools==11.2.1\n    # via\n    #   jaraco-classes\n    #   jaraco-functools\nmsgpack==1.2.3\n    # via\n    #   flax\n    #   orbax-checkpoint\nmultidict==7.1.0\n    # via\n    #   aiohttp\n    #   yarl\nmultiprocess==0.70.19\n    # via datasets\nnamex==0.1.0\n    # via keras\nnarwhals==2.27.1\n    # via\n    #   altair\n    #   scikit-learn\nnetworkx==3.7\n    # via scikit-image\nnumpy==2.5.4\n    # via\n    #   -r /tmp/gemma-colab-requirements.in\n    #   chex\n    #   clu\n    #   contourpy\n    #   datasets\n    #   dialog\n    #   dm-tree\n    #   etils\n    #   flax\n    #   gemma\n    #   grain\n    #   h5py\n    #   imageio\n    #   jax\n    #   jaxlib\n    #   kauldron\n    #   keras\n    #   matplotlib\n    #   mediapy\n    #   ml-dtypes\n    #   opencv-python\n    #   optax\n    #   orbax-checkpoint\n    #   pandas\n    #   scikit-image\n    #   scikit-learn\n    #   scipy\n    #   tensorboard\n    #   tensorflow\n    #   tensorflow-datasets\n    #   tensorstore\n    #   tifffile\n    #   treescope\noauthlib==4.0.0\n    # via requests-oauthlib\nopenapi-pydantic==0.6.0\n    # via fastmcp-slim\nopencv-python==5.0.0.93\n    # via kauldron\nopentelemetry-api==1.45.1\n    # via\n    #   fastmcp-slim\n    #   google-api-core\n    #   mcp\nopt-einsum==3.4.0\n    # via\n    #   jax\n    #   tensorflow\noptax==0.2.8\n    # via\n    #   -r /tmp/gemma-colab-requirements.in\n    #   flax\n    #   kauldron\noptree==0.20.0\n    # via keras\norbax-checkpoint==0.12.8\n    # via\n    #   flax\n    #   gemma\n    #   kauldron\npackaging==26.3\n    # via\n    #   altair\n    #   clu\n    #   datasets\n    #   fastmcp-slim\n    #   google-cloud-aiplatform\n    #   google-cloud-bigquery\n    #   huggingface-hub\n    #   keras\n    #   lazy-loader\n    #   matplotlib\n    #   scikit-image\n    #   tensorboard\n    #   tensorflow\n    #   wheel\npandas==3.0.6\n    # via\n    #   datasets\n    #   kauldron\nparso==0.8.7\n    # via jedi\npathable==0.6.0\n    # via jsonschema-path\npexpect==4.9.0\n    # via ipython\npillow==13.0.0\n    # via\n    #   dialog\n    #   imageio\n    #   matplotlib\n    #   mediapy\n    #   scikit-image\n    #   tensorboard\nplatformdirs==4.13.5\n    # via fastmcp-slim\nportpicker==1.6.0\n    # via grain\nprometheus-client==0.26.0\n    # via orbax-checkpoint\npromise==2.3\n    # via tensorflow-datasets\nprompt-toolkit==3.0.53\n    # via ipython\npropcache==0.5.4\n    # via\n    #   aiohttp\n    #   yarl\nproto-plus==1.29.0\n    # via\n    #   google-api-core\n    #   google-cloud-aiplatform\n    #   google-cloud-resource-manager\n    #   google-cloud-sql\nprotobuf==6.33.6\n    # via\n    #   google-api-core\n    #   google-cloud-aiplatform\n    #   google-cloud-resource-manager\n    #   google-cloud-sql\n    #   googleapis-common-protos\n    #   grain\n    #   grpc-google-iam-v1\n    #   grpcio-status\n    #   orbax-checkpoint\n    #   proto-plus\n    #   tensorboard\n    #   tensorflow\n    #   tensorflow-datasets\n    #   tensorflow-metadata\n    #   xmanager\npsutil==7.2.2\n    # via\n    #   ipython\n    #   orbax-checkpoint\n    #   portpicker\n    #   tensorflow-datasets\n    #   xmanager\npsygnal==0.16.1\n    # via anywidget\nptyprocess==0.7.0\n    # via pexpect\npure-eval==0.2.4\n    # via stack-data\npy-key-value-aio==0.4.6\n    # via fastmcp-slim\npyarrow==26.0.0\n    # via\n    #   datasets\n    #   tensorflow-datasets\npyasn1==0.6.4\n    # via pyasn1-modules\npyasn1-modules==0.4.2\n    # via google-auth\npycparser==3.11\n    # via cffi\npydantic==2.14.1\n    # via\n    #   fastmcp-slim\n    #   google-cloud-aiplatform\n    #   google-genai\n    #   kubernetes\n    #   mcp\n    #   mcp-types\n    #   openapi-pydantic\n    #   pydantic-settings\npydantic-core==2.50.1\n    # via pydantic\npydantic-settings==2.16.1\n    # via fastmcp-slim\npydub==0.25.1\n    # via dialog\npygments==2.21.0\n    # via\n    #   ipython\n    #   ipython-pygments-lexers\n    #   rich\n    #   rich-rst\npyjwt==2.15.1\n    # via mcp\npyparsing==3.3.3\n    # via\n    #   httplib2\n    #   matplotlib\npyperclip==1.11.0\n    # via fastmcp-slim\npython-dateutil==2.9.0.post0\n    # via\n    #   google-cloud-bigquery\n    #   kubernetes\n    #   matplotlib\n    #   pandas\npython-dotenv==1.2.4\n    # via\n    #   fastmcp-slim\n    #   pydantic-settings\n    #   xmanager\npython-multipart==0.0.32\n    # via\n    #   fastmcp-slim\n    #   mcp\npyyaml==6.0.3\n    # via\n    #   datasets\n    #   fastmcp-slim\n    #   flax\n    #   huggingface-hub\n    #   jsonschema-path\n    #   kubernetes\n    #   ml-collections\n    #   orbax-checkpoint\n    #   xmanager\nreferencing==0.37.0\n    # via\n    #   jsonschema\n    #   jsonschema-path\n    #   jsonschema-specifications\nrequests==2.34.2\n    # via\n    #   cloud-sql-python-connector\n    #   dialog\n    #   docker\n    #   google-api-core\n    #   google-auth\n    #   google-cloud-bigquery\n    #   google-cloud-storage\n    #   google-genai\n    #   jax\n    #   kubernetes\n    #   requests-oauthlib\n    #   tensorflow\n    #   tensorflow-datasets\n    #   xmanager\nrequests-oauthlib==2.0.0\n    # via kubernetes\nrich==15.0.0\n    # via\n    #   cyclopts\n    #   fastmcp-slim\n    #   flax\n    #   keras\n    #   rich-rst\nrich-rst==2.2.0\n    # via cyclopts\nrpds-py==2026.9.1\n    # via\n    #   jsonschema\n    #   referencing\nscikit-image==0.26.0\n    # via kauldron\nscikit-learn==1.9.1\n    # via kauldron\nscipy==1.18.1\n    # via\n    #   jax\n    #   jaxlib\n    #   scikit-image\n    #   scikit-learn\nsecretstorage==3.5.0\n    # via keyring\nsentencepiece==0.2.2\n    # via gemma\nsetuptools==84.2.1\n    # via\n    #   tensorboard\n    #   tensorflow\nsimple-parsing==0.1.9\n    # via tensorflow-datasets\nsimplejson==4.2.0\n    # via orbax-checkpoint\nsix==1.17.0\n    # via\n    #   astunparse\n    #   google-pasta\n    #   promise\n    #   python-dateutil\n    #   tensorflow\nsmmap==5.0.3\n    # via gitdb\nsniffio==1.3.1\n    # via google-genai\nsqlalchemy==2.1.5\n    # via\n    #   alembic\n    #   xmanager\nsqlparse==0.6.0\n    # via xmanager\nsse-starlette==3.5.0\n    # via mcp\nstack-data==0.6.3\n    # via ipython\nstarlette==1.9.0\n    # via\n    #   fastmcp-slim\n    #   mcp\n    #   sse-starlette\ntabulate==0.10.0\n    # via\n    #   kauldron\n    #   xmanager\ntenacity==9.1.4\n    # via google-genai\ntensorboard==2.21.0\n    # via kauldron\ntensorboard-data-server==0.7.2\n    # via tensorboard\ntensorflow==2.21.0\n    # via kauldron\ntensorflow-datasets==4.9.10\n    # via kauldron\ntensorflow-metadata==1.21.0\n    # via tensorflow-datasets\ntensorstore==0.1.85\n    # via\n    #   flax\n    #   orbax-checkpoint\ntermcolor==3.3.0\n    # via\n    #   tensorflow\n    #   tensorflow-datasets\n    #   xmanager\nthreadpoolctl==3.7.0\n    # via scikit-learn\ntifffile==2026.10.15\n    # via scikit-image\ntokenizers==0.23.3\n    # via -r /tmp/gemma-colab-requirements.in\ntoml==0.10.2\n    # via tensorflow-datasets\ntoolz==1.2.0\n    # via chex\ntqdm==4.70.1\n    # via\n    #   datasets\n    #   etils\n    #   huggingface-hub\n    #   kauldron\n    #   tensorflow-datasets\ntraitlets==5.16.1\n    # via\n    #   ipython\n    #   ipywidgets\n    #   matplotlib-inline\ntreescope==0.1.10\n    # via\n    #   flax\n    #   gemma\ntruststore==0.10.5\n    # via\n    #   httpcore2\n    #   httpx2\ntypeguard==4.6.0\n    # via kauldron\ntyping-extensions==4.16.0\n    # via\n    #   aiohttp\n    #   aiosignal\n    #   alembic\n    #   altair\n    #   anyio\n    #   anywidget\n    #   chex\n    #   clu\n    #   etils\n    #   exceptiongroup\n    #   fastmcp-slim\n    #   flax\n    #   google-cloud-aiplatform\n    #   google-genai\n    #   grpcio\n    #   httpx2\n    #   huggingface-hub\n    #   kauldron\n    #   kubernetes\n    #   mcp\n    #   mcp-types\n    #   opentelemetry-api\n    #   optree\n    #   orbax-checkpoint\n    #   py-key-value-aio\n    #   pydantic\n    #   pydantic-core\n    #   pydantic-settings\n    #   referencing\n    #   simple-parsing\n    #   sqlalchemy\n    #   starlette\n    #   tensorflow\n    #   typeguard\n    #   typing-inspection\ntyping-inspection==0.4.4\n    # via\n    #   mcp\n    #   pydantic\n    #   pydantic-settings\nuncalled-for==0.4.1\n    # via fastmcp-slim\nuritemplate==4.2.0\n    # via google-api-python-client\nurllib3==2.8.0\n    # via\n    #   docker\n    #   kubernetes\n    #   requests\nuvicorn==0.54.0\n    # via\n    #   fastmcp-slim\n    #   mcp\nuvloop==0.23.0\n    # via orbax-checkpoint\nwadler-lindig==0.1.7\n    # via jaxtyping\nwatchfiles==1.3.0\n    # via fastmcp-slim\nwcwidth==0.9.3\n    # via prompt-toolkit\nwebsocket-client==1.9.2\n    # via kubernetes\nwebsockets==16.1.1\n    # via\n    #   fastmcp-slim\n    #   google-genai\nwerkzeug==3.1.9\n    # via tensorboard\nwheel==0.48.0\n    # via astunparse\nwidgetsnbextension==4.0.16\n    # via ipywidgets\nwrapt==2.5.1\n    # via\n    #   clu\n    #   dm-tree\n    #   tensorflow\n    #   tensorflow-datasets\nxmanager==0.8.4\n    # via kauldron\nxxhash==4.0.2\n    # via datasets\nyarl==1.25.1\n    # via aiohttp\nzipp==4.1.1\n    # via etils\n\n\ntokamax @ git+https://github.com/openxla/tokamax.git@9ec839edafd361e316265e83d5b74dea01181ed5\n\ntiktoken==0.14.0\nregex==2026.9.29\n'
RESOLVED_DEPENDENCY_LOCK = '# This file was autogenerated by uv via the following command:\n#    uv pip compile /content/gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in --python /content/gemma_colab_pilot/.venv-V3/bin/python --output-file /content/gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.lock\nabsl-py==2.5.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   array-record\n    #   chex\n    #   clu\n    #   dm-tree\n    #   einshape\n    #   etils\n    #   gemma\n    #   grain\n    #   kauldron\n    #   keras\n    #   ml-collections\n    #   optax\n    #   orbax-checkpoint\n    #   qwix\n    #   tensorboard\n    #   tensorflow\n    #   tensorflow-datasets\n    #   tensorflow-metadata\n    #   tokamax\n    #   xmanager\n    #   xprof\naiofile==3.12.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   py-key-value-aio\naiofiles==25.1.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   orbax-checkpoint\naiohappyeyeballs==2.7.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   aiohttp\naiohttp==3.14.5\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   aiohttp-retry\n    #   cloud-sql-python-connector\n    #   fsspec\n    #   gcsfs\n    #   kubernetes\naiohttp-retry==2.9.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kubernetes\naiosignal==1.4.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   aiohttp\nalembic==1.20.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   xmanager\naltair==6.3.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\nannotated-types==0.8.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   pydantic\nanyio==4.15.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-genai\n    #   httpx\n    #   httpx2\n    #   mcp\n    #   py-key-value-aio\n    #   sse-starlette\n    #   starlette\n    #   watchfiles\nanywidget==0.11.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   dialog\narray-record==0.8.6\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   grain\n    #   tensorflow-datasets\nasttokens==3.0.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   stack-data\nastunparse==1.6.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow\nasync-generator==1.10\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   xmanager\nattrs==26.1.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   aiohttp\n    #   cyclopts\n    #   dm-tree\n    #   jsonschema\n    #   jsonschema-path\n    #   referencing\n    #   xmanager\nauthlib==1.9.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\nbeartype==0.23.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   py-key-value-aio\ncachetools==7.2.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   py-key-value-aio\ncaio==0.12.9\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   aiofile\ncertifi==2026.10.19\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\n    #   google-cloud-aiplatform\n    #   httpcore\n    #   httpx\n    #   kubernetes\n    #   requests\ncffi==2.1.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cryptography\ncharset-normalizer==3.5.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   requests\ncheroot==11.1.2\n    # via xprof\nchex==0.1.92\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\nclick==8.5.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   huggingface-hub\n    #   uvicorn\ncloud-sql-python-connector==1.23.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   xmanager\ncloudpickle==3.1.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   grain\n    #   joblib\nclu==0.0.12\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\ncomm==0.2.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipywidgets\ncontourpy==1.4.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   matplotlib\ncryptography==50.0.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   authlib\n    #   cloud-sql-python-connector\n    #   google-auth\n    #   joserfc\n    #   pyjwt\n    #   secretstorage\ncycler==0.12.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   matplotlib\ncyclopts==5.2.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\ndatasets==5.1.0\n    # via -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\ndecorator==5.3.1\n    # via gcsfs\ndialog==1.1.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   gemma\ndill==0.4.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   datasets\n    #   multiprocess\ndistro==1.9.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-genai\ndm-tree==0.1.10\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow-datasets\ndnspython==2.9.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cloud-sql-python-connector\n    #   email-validator\ndocker==7.2.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   xmanager\ndocstring-parser==0.18.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cyclopts\n    #   google-cloud-aiplatform\n    #   simple-parsing\nduckdb==2.0.0\n    # via xprof\ndurationpy==0.11\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kubernetes\neinops==0.8.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   etils\n    #   gemma\n    #   kauldron\neinshape==1.0\n    # via tokamax\nemail-validator==2.3.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   pydantic\netils==1.14.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   array-record\n    #   clu\n    #   dialog\n    #   gemma\n    #   grain\n    #   kauldron\n    #   orbax-checkpoint\n    #   tensorflow-datasets\n    #   xmanager\n    #   xprof\nexceptiongroup==1.3.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\nexecuting==2.3.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   stack-data\nfastmcp==4.1.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   gemma\nfastmcp-slim==4.1.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp\nfilelock==4.2.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   datasets\n    #   huggingface-hub\nfire==0.7.1\n    # via xprof\nflatbuffers==25.12.19\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow\nflax==0.12.10\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   clu\n    #   gemma\n    #   kauldron\n    #   qwix\nfonttools==4.67.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   matplotlib\nfrozenlist==1.8.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   aiohttp\n    #   aiosignal\nfsspec==2026.7.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   datasets\n    #   etils\n    #   gcsfs\n    #   huggingface-hub\n    #   xprof\ngast==0.7.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow\ngcsfs==2026.8.1\n    # via fsspec\ngemma==4.0.1\n    # via -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\ngitdb==4.0.12\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   gitpython\ngitpython==3.2.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   xmanager\ngoogle-api-core==2.42.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-api-python-client\n    #   google-cloud-aiplatform\n    #   google-cloud-bigquery\n    #   google-cloud-core\n    #   google-cloud-resource-manager\n    #   google-cloud-sql\n    #   google-cloud-storage\n    #   google-cloud-storage-control\n    #   xmanager\ngoogle-api-python-client==2.201.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   xmanager\ngoogle-auth==2.62.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cloud-sql-python-connector\n    #   gcsfs\n    #   google-api-core\n    #   google-api-python-client\n    #   google-auth-httplib2\n    #   google-auth-oauthlib\n    #   google-cloud-aiplatform\n    #   google-cloud-bigquery\n    #   google-cloud-core\n    #   google-cloud-resource-manager\n    #   google-cloud-sql\n    #   google-cloud-storage\n    #   google-cloud-storage-control\n    #   google-genai\n    #   xmanager\ngoogle-auth-httplib2==0.4.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-api-python-client\ngoogle-auth-oauthlib==1.5.0\n    # via gcsfs\ngoogle-cloud-aiplatform==2.6.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   xmanager\ngoogle-cloud-bigquery==3.46.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-cloud-aiplatform\ngoogle-cloud-core==2.8.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-cloud-bigquery\n    #   google-cloud-storage\ngoogle-cloud-resource-manager==1.19.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-cloud-aiplatform\ngoogle-cloud-sql==0.1.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cloud-sql-python-connector\ngoogle-cloud-storage==3.17.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   gcsfs\n    #   google-cloud-aiplatform\n    #   xmanager\n    #   xprof\ngoogle-cloud-storage-control==1.16.0\n    # via gcsfs\ngoogle-crc32c==1.9.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-cloud-storage\n    #   google-resumable-media\ngoogle-genai==2.32.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-cloud-aiplatform\ngoogle-pasta==0.2.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow\ngoogle-resumable-media==2.11.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-cloud-bigquery\n    #   google-cloud-storage\ngoogleapis-common-protos==1.75.5\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-api-core\n    #   grpc-google-iam-v1\n    #   grpcio-status\n    #   tensorflow-metadata\n    #   xmanager\ngrain==0.2.18\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   gemma\n    #   kauldron\ngraphviz==0.21\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\ngriffelib==2.4.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\ngrpc-google-iam-v1==0.14.5\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-cloud-resource-manager\n    #   google-cloud-storage-control\ngrpcio==1.84.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-api-core\n    #   google-cloud-resource-manager\n    #   google-cloud-sql\n    #   google-cloud-storage-control\n    #   googleapis-common-protos\n    #   grpc-google-iam-v1\n    #   grpcio-status\n    #   tensorboard\n    #   tensorflow\n    #   xmanager\ngrpcio-status==1.84.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-api-core\ngviz-api==1.10.0\n    # via xprof\nh11==0.16.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   httpcore\n    #   httpcore2\n    #   uvicorn\nh5py==3.14.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   keras\n    #   tensorflow\nhf-xet==1.7.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   huggingface-hub\nhttpcore==1.0.9\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   httpx\nhttpcore2==2.16.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   httpx2\nhttplib2==0.32.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-api-python-client\n    #   google-auth-httplib2\nhttpx==0.28.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-genai\nhttpx2==2.16.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\n    #   huggingface-hub\n    #   mcp\nhuggingface-hub==2.2.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   datasets\n    #   tokenizers\nhumanize==4.16.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   orbax-checkpoint\n    #   xmanager\nidna==3.20\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   anyio\n    #   email-validator\n    #   httpx\n    #   httpx2\n    #   requests\n    #   yarl\nimageio==2.38.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   scikit-image\nimmutabledict==4.3.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\n    #   tensorflow-datasets\n    #   tokamax\n    #   xmanager\nipython==9.17.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipywidgets\n    #   mediapy\nipython-pygments-lexers==1.1.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipython\nipywidgets==8.1.9\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   anywidget\njaraco-classes==3.4.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   keyring\njaraco-context==6.1.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   keyring\njaraco-functools==4.6.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cheroot\n    #   keyring\njax==0.12.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   chex\n    #   clu\n    #   flax\n    #   gemma\n    #   kauldron\n    #   optax\n    #   orbax-checkpoint\n    #   qwix\n    #   tokamax\njaxlib==0.12.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   chex\n    #   clu\n    #   jax\n    #   optax\n    #   qwix\n    #   tokamax\njaxtyping==0.3.11\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   gemma\n    #   kauldron\n    #   tokamax\njedi==0.20.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipython\njeepney==0.9.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   keyring\n    #   secretstorage\njinja2==3.1.6\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   altair\njoblib==1.6.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   scikit-learn\njoserfc==1.7.7\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   authlib\n    #   fastmcp-slim\njsonref==1.1.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\njsonschema==4.26.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   altair\n    #   mcp\njsonschema-path==0.5.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\njsonschema-specifications==2025.9.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   jsonschema\njupyterlab-widgets==3.0.17\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipywidgets\nkauldron==1.4.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   gemma\nkeras==3.15.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow\nkeyring==25.7.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   py-key-value-aio\nkiwisolver==1.5.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   matplotlib\nkubernetes==37.0.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   xmanager\nlark==1.3.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   dialog\n    #   kauldron\nlazy-loader==0.6\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   scikit-image\nlibclang==18.1.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow\nlibtpu==0.0.50\n    # via -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\nmako==1.4.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   alembic\nmarkdown==3.11.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorboard\nmarkdown-it-py==4.2.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   rich\nmarkupsafe==3.0.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   jinja2\n    #   mako\n    #   werkzeug\nmatplotlib==3.11.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   mediapy\nmatplotlib-inline==0.2.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipython\nmcp==2.3.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   dialog\n    #   fastmcp-slim\n    #   gemma\nmcp-types==2.3.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\n    #   mcp\nmdurl==0.1.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   markdown-it-py\nmediapy==1.2.7\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\nml-collections==1.1.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   clu\n    #   kauldron\nml-dtypes==0.6.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   jax\n    #   jaxlib\n    #   keras\n    #   tensorflow\n    #   tensorstore\n    #   xprof\nmore-itertools==11.2.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cheroot\n    #   jaraco-classes\n    #   jaraco-functools\nmsgpack==1.2.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   flax\n    #   orbax-checkpoint\nmultidict==7.1.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   aiohttp\n    #   yarl\nmultiprocess==0.70.19\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   datasets\nnamex==0.1.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   keras\nnarwhals==2.27.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   altair\n    #   scikit-learn\nnetworkx==3.7\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   scikit-image\nnumpy==2.5.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   chex\n    #   clu\n    #   contourpy\n    #   datasets\n    #   dialog\n    #   dm-tree\n    #   einshape\n    #   etils\n    #   flax\n    #   gemma\n    #   grain\n    #   h5py\n    #   imageio\n    #   jax\n    #   jaxlib\n    #   kauldron\n    #   keras\n    #   matplotlib\n    #   mediapy\n    #   ml-dtypes\n    #   opencv-python\n    #   optax\n    #   orbax-checkpoint\n    #   pandas\n    #   qwix\n    #   scikit-image\n    #   scikit-learn\n    #   scipy\n    #   tensorboard\n    #   tensorboardx\n    #   tensorflow\n    #   tensorflow-datasets\n    #   tensorstore\n    #   tifffile\n    #   treescope\n    #   xprof\noauthlib==4.0.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   requests-oauthlib\nopenapi-pydantic==0.6.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\nopencv-python==5.0.0.93\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\nopentelemetry-api==1.45.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\n    #   google-api-core\n    #   mcp\nopt-einsum==3.4.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   jax\n    #   qwix\n    #   tensorflow\noptax==0.2.8\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   flax\n    #   kauldron\noptree==0.20.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   keras\norbax-checkpoint==0.12.8\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   flax\n    #   gemma\n    #   kauldron\npackaging==26.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   altair\n    #   clu\n    #   datasets\n    #   fastmcp-slim\n    #   google-cloud-aiplatform\n    #   google-cloud-bigquery\n    #   huggingface-hub\n    #   keras\n    #   lazy-loader\n    #   matplotlib\n    #   scikit-image\n    #   tensorboard\n    #   tensorboardx\n    #   tensorflow\n    #   wheel\npandas==3.0.6\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   datasets\n    #   kauldron\nparso==0.8.7\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   jedi\npathable==0.6.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   jsonschema-path\npexpect==4.9.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipython\npillow==13.0.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   dialog\n    #   imageio\n    #   matplotlib\n    #   mediapy\n    #   scikit-image\n    #   tensorboard\nplatformdirs==4.13.5\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\nportpicker==1.6.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   grain\nprometheus-client==0.26.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   orbax-checkpoint\npromise==2.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow-datasets\nprompt-toolkit==3.0.53\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipython\npropcache==0.5.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   aiohttp\n    #   yarl\nproto-plus==1.29.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-api-core\n    #   google-cloud-aiplatform\n    #   google-cloud-resource-manager\n    #   google-cloud-sql\n    #   google-cloud-storage-control\nprotobuf==6.33.6\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-api-core\n    #   google-cloud-aiplatform\n    #   google-cloud-resource-manager\n    #   google-cloud-sql\n    #   google-cloud-storage-control\n    #   googleapis-common-protos\n    #   grain\n    #   grpc-google-iam-v1\n    #   grpcio-status\n    #   orbax-checkpoint\n    #   proto-plus\n    #   tensorboard\n    #   tensorboardx\n    #   tensorflow\n    #   tensorflow-datasets\n    #   tensorflow-metadata\n    #   xmanager\n    #   xprof\npsutil==7.2.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipython\n    #   orbax-checkpoint\n    #   portpicker\n    #   tensorflow-datasets\n    #   xmanager\npsygnal==0.16.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   anywidget\nptyprocess==0.7.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   pexpect\npure-eval==0.2.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   stack-data\npy-key-value-aio==0.4.6\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\npyarrow==26.0.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   datasets\n    #   tensorflow-datasets\npyasn1==0.6.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   pyasn1-modules\npyasn1-modules==0.4.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-auth\npycparser==3.11\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cffi\npydantic==2.14.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\n    #   google-cloud-aiplatform\n    #   google-genai\n    #   kubernetes\n    #   mcp\n    #   mcp-types\n    #   openapi-pydantic\n    #   pydantic-settings\n    #   tokamax\npydantic-core==2.50.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   pydantic\npydantic-settings==2.16.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\npydub==0.25.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   dialog\npygments==2.21.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipython\n    #   ipython-pygments-lexers\n    #   rich\n    #   rich-rst\npyjwt==2.15.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   mcp\npyparsing==3.3.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   httplib2\n    #   matplotlib\npyperclip==1.11.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\npython-dateutil==2.9.0.post0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-cloud-bigquery\n    #   kubernetes\n    #   matplotlib\n    #   pandas\npython-dotenv==1.2.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\n    #   pydantic-settings\n    #   xmanager\npython-multipart==0.0.32\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\n    #   mcp\npyyaml==6.0.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   datasets\n    #   fastmcp-slim\n    #   flax\n    #   huggingface-hub\n    #   jsonschema-path\n    #   kubernetes\n    #   ml-collections\n    #   orbax-checkpoint\n    #   xmanager\nqwix==0.1.8\n    # via tokamax\nreferencing==0.37.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   jsonschema\n    #   jsonschema-path\n    #   jsonschema-specifications\nrequests==2.34.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cloud-sql-python-connector\n    #   dialog\n    #   docker\n    #   gcsfs\n    #   google-api-core\n    #   google-auth\n    #   google-cloud-bigquery\n    #   google-cloud-storage\n    #   google-genai\n    #   kubernetes\n    #   requests-oauthlib\n    #   tensorflow\n    #   tensorflow-datasets\n    #   xmanager\nrequests-oauthlib==2.0.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-auth-oauthlib\n    #   kubernetes\nrich==15.0.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cyclopts\n    #   fastmcp-slim\n    #   flax\n    #   keras\n    #   rich-rst\nrich-rst==2.2.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   cyclopts\nrpds-py==2026.9.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   jsonschema\n    #   referencing\nscikit-image==0.26.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\nscikit-learn==1.9.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\nscipy==1.18.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   jax\n    #   jaxlib\n    #   scikit-image\n    #   scikit-learn\nsecretstorage==3.5.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   keyring\nsentencepiece==0.2.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   gemma\nsetuptools==84.2.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorboard\n    #   tensorflow\nsimple-parsing==0.1.9\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow-datasets\nsimplejson==4.2.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   orbax-checkpoint\nsix==1.17.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   astunparse\n    #   google-pasta\n    #   gviz-api\n    #   promise\n    #   python-dateutil\n    #   tensorflow\n    #   xprof\nsmmap==5.0.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   gitdb\nsniffio==1.3.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-genai\nsqlalchemy==2.1.5\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   alembic\n    #   xmanager\nsqlparse==0.6.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   xmanager\nsse-starlette==3.5.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   mcp\nstack-data==0.6.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipython\nstarlette==1.9.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\n    #   mcp\n    #   sse-starlette\ntabulate==0.10.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\n    #   xmanager\ntenacity==9.1.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-genai\ntensorboard==2.21.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\ntensorboard-data-server==0.7.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorboard\ntensorboardx==2.6.5\n    # via tokamax\ntensorflow==2.21.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\ntensorflow-datasets==4.9.10\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\ntensorflow-metadata==1.21.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow-datasets\ntensorstore==0.1.85\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   flax\n    #   orbax-checkpoint\ntermcolor==3.3.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fire\n    #   tensorflow\n    #   tensorflow-datasets\n    #   xmanager\nthreadpoolctl==3.7.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   scikit-learn\ntifffile==2026.10.15\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   scikit-image\ntokamax @ git+https://github.com/openxla/tokamax.git@9ec839edafd361e316265e83d5b74dea01181ed5\n    # via -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\ntokenizers==0.23.3\n    # via -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\ntoml==0.10.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorflow-datasets\ntoolz==1.2.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   chex\ntqdm==4.70.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   datasets\n    #   etils\n    #   huggingface-hub\n    #   kauldron\n    #   tensorflow-datasets\n    #   tokamax\ntraitlets==5.16.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipython\n    #   ipywidgets\n    #   matplotlib-inline\ntreescope==0.1.10\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   flax\n    #   gemma\ntruststore==0.10.5\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   httpcore2\n    #   httpx2\ntypeguard==4.6.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\ntyping-extensions==4.16.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   aiohttp\n    #   aiosignal\n    #   alembic\n    #   altair\n    #   anyio\n    #   anywidget\n    #   chex\n    #   clu\n    #   etils\n    #   exceptiongroup\n    #   fastmcp-slim\n    #   flax\n    #   google-cloud-aiplatform\n    #   google-genai\n    #   grpcio\n    #   httpx2\n    #   huggingface-hub\n    #   kauldron\n    #   kubernetes\n    #   mcp\n    #   mcp-types\n    #   opentelemetry-api\n    #   optree\n    #   orbax-checkpoint\n    #   py-key-value-aio\n    #   pydantic\n    #   pydantic-core\n    #   pydantic-settings\n    #   referencing\n    #   simple-parsing\n    #   sqlalchemy\n    #   starlette\n    #   tensorflow\n    #   typeguard\n    #   typing-inspection\ntyping-inspection==0.4.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   mcp\n    #   pydantic\n    #   pydantic-settings\nuncalled-for==0.4.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\nuritemplate==4.2.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   google-api-python-client\nurllib3==2.8.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   docker\n    #   kubernetes\n    #   requests\n    #   xprof\nuvicorn==0.54.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\n    #   mcp\nuvloop==0.23.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   orbax-checkpoint\nwadler-lindig==0.1.7\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   jaxtyping\nwatchfiles==1.3.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\nwcwidth==0.9.3\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   prompt-toolkit\nwebsocket-client==1.9.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kubernetes\nwebsockets==16.1.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   fastmcp-slim\n    #   google-genai\nwerkzeug==3.1.9\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   tensorboard\n    #   xprof\nwheel==0.48.0\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   astunparse\nwidgetsnbextension==4.0.16\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   ipywidgets\nwrapt==2.5.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   clu\n    #   dm-tree\n    #   tensorflow\n    #   tensorflow-datasets\nxmanager==0.8.4\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   kauldron\nxprof==2.24.0\n    # via tokamax\nxxhash==4.0.2\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   datasets\nyarl==1.25.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   aiohttp\nzipp==4.1.1\n    # via\n    #   -r gemma_colab_pilot/20261021T185542Z-6a2ec9f2/requirements.in\n    #   etils\n\ntiktoken==0.14.0\nregex==2026.9.29\n'
import os
import hashlib
import sys
import json
import time
import uuid
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone

NOTEBOOK_START_MONOTONIC = time.monotonic()
BASE = Path("/content/gemma_colab_pilot")
BASE.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
WORK = BASE / RUN_ID
WORK.mkdir(exist_ok=False)
if shutil.disk_usage(BASE).free < 4 * 1024**3:
    raise RuntimeError("This weights-only pilot needs at least 4 GiB free local disk.")
from google.colab import drive
drive.mount("/content/drive")
OUTPUT = Path("/content/drive/MyDrive/ShiningGem/colab-v5e1/V4.2.1") / RUN_ID
OUTPUT.mkdir(parents=True, exist_ok=False)
DEPENDENCY_CACHE = OUTPUT.parent.parent / "V3-dependencies"
DEPENDENCY_CACHE.mkdir(parents=True, exist_ok=True)

def setup_command(command, log_name, environment=None):
    with (OUTPUT / log_name).open("x") as log:
        process = subprocess.Popen(command, env=environment, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        status = process.wait()
    if status:
        raise RuntimeError(f"Setup failed (exit {status}); error above and in {OUTPUT / log_name}.")

INPUT_PATH = WORK / "requirements.in"
LOCK_PATH = WORK / "requirements.lock"
INPUT_PATH.write_text(REQUIREMENTS_INPUT)
LOCK_PATH.write_text(RESOLVED_DEPENDENCY_LOCK)
LOCK_DIGEST = hashlib.sha256(LOCK_PATH.read_bytes()).hexdigest()
INPUT_DIGEST = hashlib.sha256(INPUT_PATH.read_bytes()).hexdigest()
TOOLING = BASE / ".bootstrap"
if not (TOOLING / "uv").is_dir():
    setup_command([sys.executable, "-m", "pip", "install", "--quiet", "--target", str(TOOLING),
                   "uv==0.11.15"], "setup-bootstrap.log")
uv_env = dict(os.environ, PYTHONPATH=str(TOOLING))
uv = [sys.executable, "-m", "uv"]
PYTHON = BASE / ".venv-V4_2_1" / "bin" / "python"
if not PYTHON.exists():
    setup_command([*uv, "venv", "--python", "3.12.15", "--managed-python",
                   str(PYTHON.parent.parent)], "setup-python.log", uv_env)
ENV_MARKER = PYTHON.parent.parent / ".v4_2_1-lock-sha256"
ENVIRONMENT_REUSED = ENV_MARKER.exists() and ENV_MARKER.read_text().strip() == LOCK_DIGEST
if ENVIRONMENT_REUSED:
    print("Reusing matching V4.2.1 environment; no package resolution or install.", flush=True)
else:
    print("Installing the V3.4 pins plus native tiktoken/regex pins; no dependency resolution step.", flush=True)
    setup_command([*uv, "pip", "install", "--quiet", "--python", str(PYTHON),
                   "-r", str(LOCK_PATH)], "setup-install.log", uv_env)
    ENV_MARKER.write_text(LOCK_DIGEST + "\n")
PERSISTENT_JAX_CACHE = OUTPUT.parent.parent / "V3-cache" / "jax"
if PERSISTENT_JAX_CACHE.exists():
    shutil.copytree(PERSISTENT_JAX_CACHE, BASE / "jax-cache-V3", dirs_exist_ok=True)
SETUP_SECONDS = time.monotonic() - NOTEBOOK_START_MONOTONIC
shutil.copyfile(INPUT_PATH, OUTPUT / "requirements.in")
shutil.copyfile(LOCK_PATH, OUTPUT / "requirements.lock")
(OUTPUT / "dependency_source.json").write_text(json.dumps({
    "requirements_input_sha256": INPUT_DIGEST, "requirements_lock_sha256": LOCK_DIGEST,
    "source": "successful V3.4 report lock plus tiktoken==0.14.0 and regex==2026.9.29", "environment_reused": ENVIRONMENT_REUSED,
    "dependency_resolution_performed": False, "bootstrap_uv": "0.11.15",
    "tokamax_execution": False, "splash_execution": False}, indent=2))
print(f"Environment ready in {SETUP_SECONDS:.1f} seconds.")
print("Results:", OUTPUT)


## 2. Native o200k tokenizer and source-data access
Native `tiktoken` loads `o200k_base`. HF_TOKEN is for corpus access, including Stack-smol; the tokenizer itself needs no Hugging Face model access.


In [ ]:
from google.colab import userdata
from getpass import getpass

TOKENIZER_REFERENCE = "o200k_base"
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face token for corpus access, including Stack-smol: ")
if not HF_TOKEN.strip():
    raise ValueError("HF_TOKEN is required for the corpus sources; o200k itself is public.")


## 3. Write the embedded program
The next cell writes source and configuration; it does not execute training.


In [ ]:
PROGRAM_SOURCE = '"""Gemma Colab V4.2.1: one fixed multilingual tokenizer-screening pilot on one v5e TPU.\n"""\nfrom __future__ import annotations\nimport argparse\nimport dataclasses\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport sys\nimport time\nimport traceback\nimport zipfile\nimport numpy as np\n\nSTACK = "bigcode/the-stack-smol"\n\n\n\n\n\n\n\n\nENCODING_SCHEMA = "v421-native-byte-document-encoding-1"\n\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text())\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\\n")\n    temporary.replace(path)\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef chunk_document(ids, sequence_length, bos_id, eos_id, pad_id=0):\n    """Every target appears once; consecutive chunks overlap by one input token."""\n    if pad_id in ids:\n        raise ValueError("Document encodes a padding ID; cannot silently mask real text.")\n    sequence = [bos_id, *ids, eos_id]\n    for start in range(0, len(sequence) - 1, sequence_length):\n        window = sequence[start:start + sequence_length + 1]\n        count = len(window) - 1\n        tokens = np.full(sequence_length, pad_id, np.int32)\n        targets = np.full(sequence_length, pad_id, np.int32)\n        mask = np.zeros(sequence_length, np.bool_)\n        tokens[:count] = window[:-1]\n        targets[:count] = window[1:]\n        mask[:count] = True\n        yield tokens, targets, mask\n\n\ndef accelerator_imports(cpu_test=False):\n    global jax, jnp, optax, struct, serialization, traverse_util\n    os.environ["JAX_PLATFORMS"] = "cpu" if cpu_test else "tpu"\n    import jax\n    import jax.numpy as jnp\n    import optax\n    from flax import struct, serialization, traverse_util\n    return jax\n\n\ndef preflight(cpu_test=False):\n    accelerator_imports(cpu_test)\n    devices = jax.devices()\n    if len(devices) != 1:\n        raise RuntimeError(f"This notebook requires exactly ONE v5e chip; found {len(devices)} devices: {devices}. Select the v5e-1 TPU runtime in Colab.")\n    device = devices[0]\n    if not cpu_test:\n        kind = device.device_kind.lower().replace(" ", "")\n        if device.platform != "tpu" or not any(label in kind for label in ("v5lite", "v5e")):\n            raise RuntimeError(f"Expected a v5e-1 TPU, found {device}. Change the Colab runtime hardware accelerator.")\n    return {"platform": device.platform, "device_kind": device.device_kind,\n            "device_count": len(devices), "cpu_test": cpu_test,\n            "python": sys.version, "jax": jax.__version__,\n            "packages": {d.metadata["Name"]: d.version for d in importlib.metadata.distributions()}}\n\n\ndef numpy_leaf(value):\n    if isinstance(value, jax.Array):\n        return np.asarray(jax.device_get(value))\n    return np.asarray(value)\n\n\ndef archive_tree(state_dict):\n    leaves = []\n    def encode(node):\n        if isinstance(node, dict):\n            return {key: encode(value) for key, value in node.items()}\n        name = f"arrays/{len(leaves):05d}.npy"\n        leaves.append((name, node))\n        return {"__array__": name}\n    return encode(state_dict), leaves\n\n\ndef save_archive(path, state, metadata):\n    """Write one array at a time; never materialize a full host checkpoint copy."""\n    tree, leaves = archive_tree(serialization.to_state_dict(state))\n    temporary = Path(str(path) + ".partial")\n    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as archive:\n        for name, leaf in leaves:\n            array = numpy_leaf(leaf)\n            with archive.open(name, "w", force_zip64=True) as stream:\n                np.lib.format.write_array(stream, array, allow_pickle=False)\n            del array\n        archive.writestr("checkpoint.json", json.dumps({"tree": tree, "metadata": metadata}, allow_nan=False))\n    temporary.replace(path)\n    return len(leaves)\n\n\nMODEL_HELPERS = \'@struct.dataclass\\nclass PilotState:\\n    step: jax.Array\\n    params: object\\n    opt_state: object\\n    rng: jax.Array\\n    data_position: jax.Array\\n    short_rows: jax.Array\\n    long_rows: jax.Array\\n\'\n\ndef pack_chunks(arrays, seed):\n    """Best-fit decreasing, preserving each original chunk\'s complete context."""\n    tokens, targets, masks = (arrays[key] for key in ("tokens", "targets", "mask"))\n    count, length = tokens.shape\n    lengths = masks.sum(axis=1).astype(int)\n    if np.any(lengths <= 0) or not np.array_equal(masks, np.arange(length)[None, :] < lengths[:, None]):\n        raise ValueError("Input chunks must have a nonempty contiguous prefix of scored targets.")\n    shuffled = np.random.default_rng(seed).permutation(count)\n    ordered = shuffled[np.argsort(-lengths[shuffled], kind="stable")]\n    buckets = [[] for _ in range(length + 1)]\n    bins, assignments = [], []\n    for source in ordered:\n        size = int(lengths[source])\n        free = next((room for room in range(size, length + 1) if buckets[room]), None)\n        if free is None:\n            index = len(bins)\n            bins.append(0)\n            free = length\n        else:\n            index = buckets[free].pop()\n        start = bins[index]\n        bins[index] += size\n        buckets[free - size].append(index)\n        assignments.append([int(source), index, start, size])\n    shape = (len(bins), length)\n    packed = {"tokens": np.zeros(shape, np.int32), "targets": np.zeros(shape, np.int32),\n              "mask": np.zeros(shape, np.bool_), "positions": np.zeros(shape, np.int32),\n              "segment_ids": np.full(shape, -1, np.int32)}\n    seen = np.zeros(count, np.int32)\n    for source, index, start, size in assignments:\n        sl = slice(start, start + size)\n        packed["tokens"][index, sl] = tokens[source, :size]\n        packed["targets"][index, sl] = targets[source, :size]\n        packed["mask"][index, sl] = True\n        packed["positions"][index, sl] = np.arange(size)\n        packed["segment_ids"][index, sl] = source\n        seen[source] += 1\n    # Verify content and positions before discarding the unpacked training arrays.\n    for source, index, start, size in assignments:\n        sl = slice(start, start + size)\n        if not (np.array_equal(packed["tokens"][index, sl], tokens[source, :size])\n                and np.array_equal(packed["targets"][index, sl], targets[source, :size])\n                and np.array_equal(packed["positions"][index, sl], np.arange(size))):\n            raise ValueError("Packing changed a source chunk.")\n    if not np.all(seen == 1) or int(packed["mask"].sum()) != int(lengths.sum()):\n        raise ValueError("Packing duplicated or dropped source targets.")\n    report = {"version": PACKING_VERSION, "source_chunks": count, "packed_sequences": len(bins),\n              "scored_tokens": int(lengths.sum()), "positions": int(np.prod(shape)),\n              "occupancy": float(lengths.sum() / np.prod(shape)),\n              "targets_preserved": True, "context_preserved": True,\n              "assignments_source_bin_start_length": assignments}\n    return packed, report\n\n\ndef packed_attention_mask(segment_ids):\n    length = segment_ids.shape[-1]\n    same_segment = segment_ids[:, :, None] == segment_ids[:, None, :]\n    causal = jnp.arange(length)[:, None] >= jnp.arange(length)[None, :]\n    valid = segment_ids >= 0\n    allowed = same_segment & causal[None, :, :] & valid[:, :, None] & valid[:, None, :]\n    # Padding queries attend to themselves so every softmax has at least one key.\n    return allowed | ((~valid)[:, :, None] & jnp.eye(length, dtype=jnp.bool_)[None, :, :])\n\n\ndef batch_training_arrays(arrays, batch_size, seed):\n    order = np.random.default_rng(seed).permutation(len(arrays["tokens"]))\n    padded = (-len(order)) % batch_size\n    result = {}\n    for key, values in arrays.items():\n        values = values[order]\n        if padded:\n            values = np.pad(values, ((0, padded), (0, 0)), constant_values=-1 if key == "segment_ids" else 0)\n        result[key] = values.reshape((-1, batch_size, values.shape[-1]))\n    return result, hashlib.sha256(order.astype("<i8").tobytes()).hexdigest()\n\n\n\n"""V3 extensions, concatenated with base.py by build_notebook.py; no local execution."""\nimport functools\nfrom collections import Counter, defaultdict\nfrom urllib.parse import urlsplit, urlunsplit\n\nPROTOCOL = "V39-expanded-byte-mix-1"\nPACKING_VERSION = "V3-intact-chunks-1"\nLANGUAGES = ["eng_Latn", "fra_Latn", "spa_Latn", "por_Latn", "cmn_Hani", "arb_Arab",\n    "hin_Deva", "swh_Latn", "zul_Latn", "xho_Latn", "tsn_Latn", "sot_Latn",\n    "rus_Cyrl", "jpn_Jpan", "deu_Latn", "ibo_Latn"]\nWIKI_CODES = ["en", "fr", "es", "pt", "zh", "ar", "hi", "sw", "zu", "xh", "tn", "st", "ru", "ja", "de", "ig"]\nDOMAINS = ["pdf_edu", "dclm", "web_edu", "multilingual_web", "math", "code_python",\n    "code_java", "code_cpp", "code_javascript", "code_rust", "code_html", "translations", "wiki_holdout"]\nSMOL = "HuggingFaceFW/finepdfs_edu_50BT-dclm_30BT-fineweb_edu_20BT-shuffled"\nSOURCE_REPOS = [SMOL, "HuggingFaceFW/fineweb-2", "HuggingFaceTB/finemath",\n    "codeparrot/codeparrot-clean", "HuggingFaceFW/finetranslations", STACK, "HuggingFaceFW/finewiki"]\nINPUT_KEYS = ("tokens", "targets", "mask", "positions", "segment_ids", "text_mask", "byte_weights", "language_ids", "domain_ids")\nEXTRA_KEYS = ("text_mask", "byte_weights", "language_ids", "domain_ids")\n\n\ndef recipe():\n    buckets = [dict(key=name, repo=SMOL, subset=None, language="eng_Latn", domain=name, bp=bp)\n        for name, bp in (("pdf_edu", 2500), ("dclm", 1500), ("web_edu", 1000))]\n    shares = [250, 200, 150, 350, 350, 350, 200, 100, 100, 100, 80, 250, 250, 170, 100]\n    buckets += [dict(key="fw2_" + lang, repo=SOURCE_REPOS[1], subset=lang, language=lang,\n        domain="multilingual_web", bp=bp) for lang, bp in zip(LANGUAGES[1:], shares)]\n    buckets.append(dict(key="math", repo=SOURCE_REPOS[2], subset="finemath-3plus",\n        language="eng_Latn", domain="math", bp=500))\n    buckets.append(dict(key="code_python", repo=SOURCE_REPOS[3], subset=None,\n        language="eng_Latn", domain="code_python", bp=300))\n    for key, subset, bp in (("java", "java", 200), ("cpp", "c++", 200),\n            ("javascript", "javascript", 150), ("rust", "rust", 100), ("html", "html", 50)):\n        buckets.append(dict(key="code_" + key, repo=STACK, subset=subset,\n            language="eng_Latn", domain="code_" + key, bp=bp))\n    buckets += [dict(key="translations_" + lang, repo=SOURCE_REPOS[4], subset=lang,\n        language=lang, domain="translations", bp=125)\n        for lang in ("fra_Latn", "arb_Arab", "hin_Deva", "swh_Latn")]\n    assert sum(bucket["bp"] for bucket in buckets) == 10000\n    return {"protocol": PROTOCOL, "training_bytes_target": 256 * 1024**2,\n        "in_mix_eval_bytes_target": 1024**2, "wiki_eval_bytes_per_language": 64 * 1024,\n        "max_document_span_bytes": 32 * 1024, "byte_quota_tolerance": 3,\n        "source_weight_unit": "original UTF-8 bytes", "buckets": buckets,\n        "split_rule": "SHA256(source group) modulo 10: 0 held out, 1..9 train",\n        "sampling_scope": "bounded deterministic stream prefixes; not uniform full-corpus sampling",\n        "code_language_note": "Code has its own six domain labels; eng_Latn is an accounting placeholder for code, not detected natural-language identity."}\n\n\ndef protocol_fingerprint():\n    return hashlib.sha256(json.dumps(recipe(), sort_keys=True).encode()).hexdigest()\n\n\ndef canonical_group(row, text):\n    metadata = row.get("metadata")\n    url = row.get("url") or (metadata.get("url") if isinstance(metadata, dict) else None)\n    if url:\n        parsed = urlsplit(str(url))\n        return "url:" + urlunsplit(("", parsed.netloc.lower().removeprefix("www."), parsed.path.rstrip("/"), parsed.query, ""))\n    if row.get("repo_name") or row.get("repository_name"):\n        return "code-repo:" + str(row.get("repo_name") or row["repository_name"])\n    return "id:" + str(row.get("id") or row.get("blob_id") or hashlib.sha256(text.encode()).hexdigest())\n\n\ndef utf8_prefix(text, limit):\n    return text.encode("utf-8")[:limit].decode("utf-8", errors="ignore")\n\n\ndef read_jsonl(path):\n    with Path(path).open(encoding="utf-8") as stream:\n        return [json.loads(line) for line in stream]\n\n\ndef verify_raw(directory):\n    manifest = read_json(directory / "manifest.json")\n    if manifest["recipe_sha256"] != protocol_fingerprint():\n        raise ValueError("The shared raw corpus belongs to another V3 protocol.")\n    for name, digest in manifest["files"].items():\n        if sha256_file(directory / name) != digest:\n            raise ValueError(f"Raw corpus checksum differs: {name}")\n    return manifest\n\n\n\n"""V4.2.1 parallel source preparation. Embedded in the notebook; not run locally."""\nfrom concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED\nimport threading\n\nDOWNLOAD_WORKERS = 6\nPREFIX_CACHE_VERSION = "v39-expanded-source-prefix-1"\nPREFIX_FACTOR = 1.25\nPREFIX_PART_BYTES = 4 * 1024**2\nPREFIX_JOB_LIMIT_BYTES = 256 * 1024**2\n\n\ndef immutable_sources(shared):\n    from huggingface_hub import HfApi\n    path = shared / "source-lock.json"\n    if path.exists():\n        return read_json(path)\n    def resolve(repo):\n        return HfApi().dataset_info(repo, revision="main").sha\n    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:\n        revisions = list(pool.map(resolve, SOURCE_REPOS))\n    lock = dict(zip(SOURCE_REPOS, revisions))\n    write_json(path, lock)\n    return lock\n\n\ndef source_jobs(lock):\n    from datasets import get_dataset_config_names\n    rules = recipe()\n    wiki_repo = SOURCE_REPOS[-1]\n    names = set(get_dataset_config_names(wiki_repo, revision=lock[wiki_repo]))\n    wiki = []\n    for language, code in zip(LANGUAGES, WIKI_CODES):\n        subset = next((name for name in (code, code + "wiki") if name in names), None)\n        if subset is None:\n            raise ValueError(f"Pinned FineWiki has no configuration for {language}.")\n        wiki.append({"key": "wiki_" + language, "kind": "wiki", "repo": wiki_repo,\n            "subset": subset, "revision": lock[wiki_repo], "language": language,\n            "targets": {"wiki:wiki_" + language: rules["wiki_eval_bytes_per_language"]}, "row_limit": 100000})\n    grouped = defaultdict(list)\n    for bucket in rules["buckets"]:\n        grouped[(bucket["repo"], bucket["subset"])].append(bucket)\n    training = []\n    for (repo, subset), buckets in grouped.items():\n        targets = {}\n        for bucket in buckets:\n            targets["train:" + bucket["key"]] = round(rules["training_bytes_target"] * bucket["bp"] / 10000)\n            targets["in_mix:" + bucket["key"]] = round(rules["in_mix_eval_bytes_target"] * bucket["bp"] / 10000)\n        training.append({"key": "smol_english" if repo == SMOL else buckets[0]["key"], "kind": "training",\n            "repo": repo, "subset": subset, "revision": lock[repo], "buckets": buckets,\n            "targets": targets, "row_limit": 300000})\n    # Canonical merge order is V3 order, independent of thread completion order.\n    return wiki + training\n\n\ndef candidate_records(job, row):\n    rules = recipe()\n    if job["kind"] == "wiki":\n        texts = [(row.get("text"), job["language"])]\n        bucket_key, domain = job["key"], "wiki_holdout"\n    else:\n        if row.get("early_stop") or row.get("is_truncated"):\n            return []\n        if job["repo"] == SMOL:\n            origin = str(row.get("dataset", ""))\n            bucket_key = "pdf_edu" if "finepdfs" in origin else "web_edu" if "fineweb-edu" in origin else "dclm" if "dclm" in origin else None\n            if bucket_key is None:\n                raise ValueError("Unknown upstream Smol-Data source label: " + origin)\n            bucket = next(bucket for bucket in job["buckets"] if bucket["key"] == bucket_key)\n        else:\n            bucket = job["buckets"][0]\n            bucket_key = bucket["key"]\n        domain = bucket["domain"]\n        if job["repo"] == SOURCE_REPOS[4]:\n            score = row.get("edu_score")\n            if score is None or int(score) < 2:\n                return []\n            texts = [(row.get("og_full_text"), bucket["language"]), (row.get("translated_text"), "eng_Latn")]\n        else:\n            texts = [(row.get("content") if domain.startswith("code_") else row.get("text"), bucket["language"])]\n    group = canonical_group(row, texts[0][0] or "")\n    if job["kind"] != "wiki" and "wikipedia.org" in group:\n        return []\n    split = "wiki" if job["kind"] == "wiki" else "in_mix" if int(hashlib.sha256(group.encode()).hexdigest()[:16], 16) % 10 == 0 else "train"\n    provenance = {key: row.get(key) for key in ("id", "url", "repo_name", "repository_name", "path", "license", "licenses", "lang", "dataset") if row.get(key) is not None}\n    candidates = []\n    for text, language in texts:\n        if not isinstance(text, str) or not text.strip():\n            continue\n        encoded = text.encode("utf-8")\n        span = utf8_prefix(text, rules["max_document_span_bytes"])\n        candidates.append({"text": span, "original_document_sha256": hashlib.sha256(encoded).hexdigest(),\n            "source_was_clipped": span != text, "source_group": group, "source": provenance,\n            "language": language, "domain": domain, "bucket": bucket_key, "split": split,\n            "utf8_bytes": len(span.encode("utf-8"))})\n    return candidates\n\n\ndef materialize_prefix(job, root, factor, report_status, stop, required_strata=None):\n    """Cache deterministic source prefixes in independently committed 4 MiB target batches."""\n    from datasets import load_dataset, DownloadConfig\n    started = time.monotonic()\n    job_hash = hashlib.sha256(json.dumps({"version": PREFIX_CACHE_VERSION, "job": job}, sort_keys=True).encode()).hexdigest()\n    directory = root / job_hash[:24]\n    directory.mkdir(parents=True, exist_ok=True)\n    state_path = directory / "state.json"\n    state = read_json(state_path) if state_path.exists() else {"job_sha256": job_hash, "job": job,\n        "rows_read": 0, "candidate_bytes": {}, "candidate_documents": {}, "parts": [], "exhausted": False}\n    if state["job_sha256"] != job_hash:\n        raise ValueError("Source-prefix cache identity mismatch: " + job["key"])\n    for part in state["parts"]:\n        if sha256_file(directory / part["name"]) != part["sha256"]:\n            raise ValueError("Source-prefix cache checksum mismatch: " + job["key"])\n    original_rows = state["rows_read"]\n    original_parts = len(state["parts"])\n    active = set(job["targets"]) if required_strata is None else set(required_strata)\n    if not active <= set(job["targets"]):\n        raise ValueError("Unknown source-cache stratum requested.")\n    desired = {key: math.ceil(value * factor) for key, value in job["targets"].items() if key in active}\n    cache_limit = prefix_cache_limit(job)\n    def complete():\n        return all(state["candidate_bytes"].get(key, 0) >= quota and state["candidate_documents"].get(key, 0) >= 4\n                   for key, quota in desired.items())\n    def status(stage):\n        report_status(job["key"], {"stage": stage, "rows": state["rows_read"],\n            "candidate_bytes": sum(state["candidate_bytes"].values()), "seconds": time.monotonic() - started})\n    def result():\n        return {"directory": str(directory), "rows_read": state["rows_read"], "candidate_parts": len(state["parts"]),\n            "candidate_bytes": sum(state["candidate_bytes"].values()), "prefix_cache_reused": original_parts > 0,\n            "cache_limit_bytes": cache_limit, "cached_json_bytes": sum(part["bytes"] for part in state["parts"]),\n            "candidate_bytes_by_stratum": dict(state["candidate_bytes"]),\n            "selection_policy": "v391-quota-filtered-prefix", "requested_strata": sorted(active),\n            "new_rows_read": state["rows_read"] - original_rows, "seconds": time.monotonic() - started}\n    if complete() or state["exhausted"]:\n        status("cached")\n        return result()\n    buffer = bytearray()\n    committed_bytes = sum(part["bytes"] for part in state["parts"])\n    last_flush = time.monotonic()\n    def commit():\n        nonlocal buffer, last_flush, committed_bytes\n        if buffer:\n            name = f"part-{len(state[\'parts\']):05d}.jsonl"\n            temporary = directory / (name + ".partial")\n            temporary.write_bytes(buffer)\n            temporary.replace(directory / name)\n            state["parts"].append({"name": name, "bytes": len(buffer), "sha256": hashlib.sha256(buffer).hexdigest()})\n            committed_bytes += len(buffer)\n            buffer = bytearray()\n        # A cursor is committed only after all candidates from its row are committed.\n        write_json(state_path, state)\n        last_flush = time.monotonic()\n    status("opening stream")\n    kwargs = {"name": job["subset"], "split": "train", "streaming": True, "revision": job["revision"],\n              "download_config": DownloadConfig(max_retries=1)}\n    if job["repo"] == SOURCE_REPOS[4]:\n        # Avoid downloading duplicate original/translated chunk-list columns.\n        kwargs["columns"] = ["id", "url", "og_full_text", "translated_text", "og_language", "edu_score", "early_stop"]\n    try:\n        if job["repo"] == STACK:\n            from huggingface_hub import HfApi\n            paths = sorted(path for path in HfApi().list_repo_files(STACK, repo_type="dataset", revision=job["revision"])\n                if path.startswith("data/" + job["subset"] + "/") and path.endswith(".json"))\n            if not paths:\n                raise ValueError("Pinned code language files missing: " + job["subset"])\n            kwargs.pop("name")\n            kwargs.pop("revision")\n            kwargs["data_files"] = {"train": ["hf://datasets/" + STACK + "@" + job["revision"] + "/" + path for path in paths]}\n            stream = load_dataset("json", **kwargs)\n        else:\n            stream = load_dataset(job["repo"], **kwargs)\n        if state["rows_read"]:\n            stream = stream.skip(state["rows_read"])\n        status("reading")\n        for row in stream:\n            if stop.is_set():\n                commit()\n                status("stopped; cache saved")\n                return result()\n            if state["rows_read"] >= job["row_limit"]:\n                state["exhausted"] = True\n                break\n            records = select_prefix_records(candidate_records(job, row), state, desired)\n            encoded_records = [(json.dumps(record, ensure_ascii=False) + "\\n").encode("utf-8") for record in records]\n            projected_json_bytes = committed_bytes + len(buffer) + sum(map(len, encoded_records))\n            if projected_json_bytes > cache_limit:\n                raise RuntimeError(f"Source-prefix disk cache limit reached: {job[\'key\']}; "\n                    f"projected={projected_json_bytes:,}, limit={cache_limit:,} bytes; "\n                    f"candidate UTF-8 bytes by stratum={state[\'candidate_bytes\']}. "\n                    "Completed source batches preserved; no training started.")\n            for record, encoded_record in zip(records, encoded_records):\n                buffer.extend(encoded_record)\n                stratum = record["split"] + ":" + record["bucket"]\n                state["candidate_bytes"][stratum] = state["candidate_bytes"].get(stratum, 0) + record["utf8_bytes"]\n                state["candidate_documents"][stratum] = state["candidate_documents"].get(stratum, 0) + 1\n            state["rows_read"] += 1\n            if len(buffer) >= PREFIX_PART_BYTES or time.monotonic() - last_flush >= 5:\n                commit()\n                status("reading")\n            if complete():\n                break\n        else:\n            state["exhausted"] = True\n        commit()\n        status("ready")\n        return result()\n    except Exception:\n        commit()\n        status("failed; cache saved")\n        raise\n\n\ndef build_raw_corpus(destination, lock):\n    started = time.monotonic()\n    destination.mkdir(parents=True, exist_ok=True)\n    if (destination / "manifest.json").exists():\n        # A previous interrupted run may have completed the final corpus already.\n        manifest = verify_raw(destination)\n        write_json(destination.parent / "parallel_downloads.json", {"version": "V4.2.1",\n            "recovered_complete_staging": True, "max_workers": DOWNLOAD_WORKERS,\n            "peak_active_source_jobs": 0, "source_jobs": 0, "fetch_seconds": 0.,\n            "note": "Verified complete staged corpus recovered without source fetching."})\n        return manifest\n    shared = destination.parent\n    cache_root = shared / PREFIX_CACHE_VERSION\n    cache_root.mkdir(parents=True, exist_ok=True)\n    jobs = source_jobs(lock)\n    status_lock, stop = threading.Lock(), threading.Event()\n    statuses = {job["key"]: {"stage": "queued", "rows": 0, "candidate_bytes": 0, "seconds": 0.} for job in jobs}\n    peak_active = 0\n    def status(key, value):\n        nonlocal peak_active\n        with status_lock:\n            statuses[key] = value\n            active = sum(item["stage"] in ("opening stream", "reading") for item in statuses.values())\n            peak_active = max(peak_active, active)\n    # Start the large English stream and all four translations early; interleave Wiki/FW2 thereafter.\n    front = [job for job in jobs if job["key"] == "smol_english" or job["repo"] == SOURCE_REPOS[4]]\n    wiki_jobs = [job for job in jobs if job["kind"] == "wiki"]\n    other_jobs = [job for job in jobs if job not in front and job["kind"] != "wiki"]\n    scheduled = front[:]\n    for index in range(max(len(wiki_jobs), len(other_jobs))):\n        if index < len(wiki_jobs):\n            scheduled.append(wiki_jobs[index])\n        if index < len(other_jobs):\n            scheduled.append(other_jobs[index])\n    print(f"V4.2.1: {len(jobs)} source jobs; {DOWNLOAD_WORKERS} concurrent workers; completed batches persist in Drive.", flush=True)\n    reports = {}\n    pool = ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS)\n    futures = {pool.submit(materialize_prefix, job, cache_root, PREFIX_FACTOR, status, stop): job for job in scheduled}\n    pending = set(futures)\n    try:\n        while pending:\n            done, pending = wait(pending, timeout=5, return_when=FIRST_COMPLETED)\n            for future in done:\n                job = futures[future]\n                try:\n                    report = future.result()\n                except Exception:\n                    write_json(shared / "parallel_download_failure.json", {"job": job, "error": traceback.format_exc(),\n                        "completed_jobs": reports, "training_started": False})\n                    raise\n                reports[job["key"]] = report\n                print(f"Ready {len(reports)}/{len(jobs)}: {job[\'key\']} ({report[\'seconds\']:.1f}s, reused={report[\'prefix_cache_reused\']})", flush=True)\n            with status_lock:\n                active = [(key, dict(value)) for key, value in statuses.items() if value["stage"] in ("opening stream", "reading")]\n            if pending:\n                details = "; ".join(f"{key}: {value[\'stage\']}, {value[\'rows\']:,} rows, {value[\'candidate_bytes\']/1024**2:.1f} MiB"\n                                    for key, value in active)\n                print(f"Sources ready {len(reports)}/{len(jobs)}; active {len(active)}/{DOWNLOAD_WORKERS}. {details}", flush=True)\n    finally:\n        stop.set()\n        for future in pending:\n            future.cancel()\n        pool.shutdown(wait=True, cancel_futures=True)\n    fetch_seconds = time.monotonic() - started\n\n    # Merge in canonical source order so network completion cannot change the corpus.\n    rules = recipe()\n    seen_hashes, forbidden_groups = set(), set()\n    docs = {"train": [], "in_mix": [], "wiki": []}\n    counts, rejected, summaries = Counter(), Counter(), {}\n    refill_stop = threading.Event()\n    for job in jobs:\n        targets = job["targets"]\n        actual = {key: 0 for key in targets}\n        consumed_parts = 0\n        factor = PREFIX_FACTOR\n        directory = Path(reports[job["key"]]["directory"])\n        while True:\n            state = read_json(directory / "state.json")\n            for part in state["parts"][consumed_parts:]:\n                with (directory / part["name"]).open(encoding="utf-8") as stream:\n                    for line in stream:\n                        candidate = json.loads(line)\n                        split, bucket = candidate["split"], candidate["bucket"]\n                        stratum = split + ":" + bucket\n                        remaining = targets[stratum] - actual[stratum]\n                        if remaining <= rules["byte_quota_tolerance"]:\n                            continue\n                        if split == "train" and candidate["source_group"] in forbidden_groups:\n                            rejected["reserved_evaluation_group"] += 1\n                            continue\n                        span = utf8_prefix(candidate["text"], remaining)\n                        if not span:\n                            continue\n                        digest = hashlib.sha256(span.encode("utf-8")).hexdigest()\n                        full_digest = candidate["original_document_sha256"]\n                        if full_digest in seen_hashes or digest in seen_hashes:\n                            rejected["exact_duplicate"] += 1\n                            continue\n                        seen_hashes.update((full_digest, digest))\n                        if split != "train":\n                            forbidden_groups.add(candidate["source_group"])\n                        length = len(span.encode("utf-8"))\n                        docs[split].append({"text": span, "sha256": digest, "original_document_sha256": full_digest,\n                            "source_group": candidate["source_group"], "source": candidate["source"],\n                            "language": candidate["language"], "domain": candidate["domain"], "bucket": bucket,\n                            "utf8_bytes": length, "source_byte_start": 0, "source_byte_end": length,\n                            "clipped": candidate["source_was_clipped"] or span != candidate["text"]})\n                        actual[stratum] += length\n            consumed_parts = len(state["parts"])\n            if all(targets[key] - actual[key] <= rules["byte_quota_tolerance"] for key in targets):\n                break\n            if state["exhausted"] or state["rows_read"] >= job["row_limit"]:\n                raise RuntimeError(f"Data quota shortfall after deduplication: {job[\'key\']}: {actual}/{targets}. No substitution.")\n            # Continue the same pinned stream when deduplication consumes the buffer.\n            # This is additional data preparation, never another model experiment.\n            factor *= 2\n            print(f"Extending cached prefix after deduplication: {job[\'key\']}", flush=True)\n            prior_rows = state["rows_read"]\n            missing = [key for key in targets if targets[key] - actual[key] > rules["byte_quota_tolerance"]]\n            report = materialize_prefix(job, cache_root, factor, status, refill_stop, required_strata=missing)\n            reports[job["key"]]["refilled_prefix"] = report\n            if report["rows_read"] == prior_rows:\n                raise RuntimeError("No additional source rows available: " + job["key"])\n        if job["kind"] == "wiki":\n            summaries[job["key"]] = {"bytes": actual["wiki:" + job["key"]], "subset": job["subset"]}\n        else:\n            for bucket in job["buckets"]:\n                key = bucket["key"]\n                summaries[key] = {"targets": {split: targets[split + ":" + key] for split in ("train", "in_mix")},\n                    "actual": {split: actual[split + ":" + key] for split in ("train", "in_mix")},\n                    "repository": job["repo"], "subset": job["subset"]}\n        print("Selected byte quotas: " + job["key"] + " " + json.dumps(actual), flush=True)\n    by_bucket = defaultdict(list)\n    for doc in docs["train"]:\n        by_bucket[doc["bucket"]].append(doc)\n    rng = np.random.default_rng(42)\n    weights = {bucket["key"]: bucket["bp"] / 10000 for bucket in rules["buckets"]}\n    for rows in by_bucket.values():\n        rng.shuffle(rows)\n    used, interleaved = Counter(), []\n    while any(by_bucket.values()):\n        key = min((key for key, rows in by_bucket.items() if rows), key=lambda key: used[key] / weights[key])\n        doc = by_bucket[key].pop()\n        interleaved.append(doc)\n        used[key] += doc["utf8_bytes"]\n    docs["train"] = interleaved\n    train_groups = {doc["source_group"] for doc in docs["train"]}\n    eval_groups = {doc["source_group"] for split in ("in_mix", "wiki") for doc in docs[split]}\n    if train_groups & eval_groups:\n        raise ValueError("Source group overlaps training and evaluation.")\n    for split, rows in docs.items():\n        temporary = destination / (split + ".jsonl.partial")\n        with temporary.open("w", encoding="utf-8") as stream:\n            for doc in rows:\n                stream.write(json.dumps(doc, ensure_ascii=False) + "\\n")\n        temporary.replace(destination / (split + ".jsonl"))\n        counts[split] = len(rows)\n    # Preserve the V3 manifest schema so completed V3 raw/token caches remain reusable.\n    manifest = {"protocol": PROTOCOL, "recipe_sha256": protocol_fingerprint(), "source_revisions": lock,\n        "recipe": rules, "buckets": summaries, "rejected": dict(rejected), "document_counts": dict(counts),\n        "original_bytes": {split: sum(doc["utf8_bytes"] for doc in rows) for split, rows in docs.items()},\n        "language_bytes": {split: {lang: sum(doc["utf8_bytes"] for doc in rows if doc["language"] == lang) for lang in LANGUAGES}\n                           for split, rows in docs.items()},\n        "cross_split_source_groups": 0,\n        "deduplication_scope": "global exact source/span hashes and source groups; near duplicates are not exhaustively removed",\n        "files": {split + ".jsonl": sha256_file(destination / (split + ".jsonl")) for split in docs}}\n    write_json(shared / "parallel_downloads.json", {"version": "V4.2.1", "max_workers": DOWNLOAD_WORKERS,\n        "source_jobs": len(jobs), "peak_active_source_jobs": peak_active,\n        "fetch_seconds": fetch_seconds, "prepare_raw_seconds": time.monotonic() - started,\n        "jobs": reports, "merge_order": [job["key"] for job in jobs],\n        "initial_prefix_factor": PREFIX_FACTOR, "minimum_candidate_documents_per_stratum": 4,\n        "max_cached_bytes_per_job": max(prefix_cache_limit(job) for job in jobs),\n        "cached_bytes_limit_by_job": {job["key"]: prefix_cache_limit(job) for job in jobs},\n        "prefix_selection": "stop caching each satisfied stratum; refill deficient strata only",\n        "persistent_batch_target_bytes": PREFIX_PART_BYTES,\n        "selection": "canonical V3 source order; independent of worker completion order",\n        "throughput_speedup_measured": False})\n    # The final manifest is the completion marker; it is committed last.\n    write_json(destination / "manifest.json", manifest)\n    return manifest\n\n\ndef prepare_tokenizer(config, destination):\n    import base64\n    import tiktoken\n    if (config["tokenizer_reference"] != "o200k_base"\n            or config["tokenizer_revision"] != "tiktoken-0.14.0"\n            or config["model_id"] != "google/gemma-3-270m"\n            or importlib.metadata.version("tiktoken") != "0.14.0"):\n        raise ValueError("V4.2.1 requires native tiktoken 0.14.0 o200k_base and the Gemma3_270M backbone.")\n    destination.mkdir(parents=True, exist_ok=True)\n    cache = Path(config["tokenizer_cache_directory"])\n    cache.mkdir(parents=True, exist_ok=True)\n    os.environ["TIKTOKEN_CACHE_DIR"] = str(cache)\n    tokenizer = tiktoken.get_encoding("o200k_base")\n    ranks = tokenizer._mergeable_ranks\n    specials = tokenizer._special_tokens\n    rank_file = b"".join(base64.b64encode(piece) + b" " + str(rank).encode() + b"\\n"\n        for piece, rank in sorted(ranks.items(), key=lambda item: item[1]))\n    if hashlib.sha256(rank_file).hexdigest() != O200K_RANKS_SHA256:\n        raise ValueError("Native o200k_base mergeable ranks differ from the official pinned artifact.")\n    asset = {"pat_str": tokenizer._pat_str,\n        "mergeable_ranks": [[base64.b64encode(piece).decode(), rank] for piece, rank in ranks.items()],\n        "special_tokens": specials}\n    payload = (json.dumps(asset, ensure_ascii=False, separators=(",", ":")) + "\\n").encode("utf-8")\n    if hashlib.sha256(payload).hexdigest() != O200K_ASSET_SHA256:\n        raise ValueError("Native o200k_base pattern, ranks or special-token map differ from the project pin.")\n    (destination / "o200k_base.json").write_bytes(payload)\n    ids = sorted(set(ranks.values()) | set(specials.values()))\n    if len(ranks) != 199998 or len(ids) != 200000 or max(ids) != 200018:\n        raise ValueError("Unexpected o200k_base vocabulary inventory.")\n    mapping = {old: index + 1 for index, old in enumerate(ids)}\n    rows = len(ids) + 3\n    total = 100326016 + rows * 640\n    if rows != 200003 or total != 228327936:\n        raise ValueError("Unexpected model size after native o200k vocabulary adaptation.")\n    info = {"reference": "o200k_base", "revision": "tiktoken-0.14.0",\n        "implementation": "native tiktoken", "tokenizer_sha256": O200K_ASSET_SHA256,\n        "mergeable_ranks_sha256": O200K_RANKS_SHA256, "vocabulary_ids": len(ids),\n        "mergeable_token_ids": len(ranks), "native_special_token_ids": specials,\n        "original_id_space": max(ids) + 1, "unused_native_ids": max(ids) + 1 - len(ids),\n        "model_rows": rows, "pad_id": 0, "bos_id": len(ids) + 1, "eos_id": len(ids) + 2,\n        "id_mapping": "sorted defined native IDs map to 1..N; holes have no model row; PAD=0, BOS=N+1, EOS=N+2",\n        "expected_parameters": total, "model_type": "native_byte_level_BPE",\n        "text_encoding_policy": "encode_ordinary; native special-token spellings remain ordinary text",\n        "byte_accounting": "Exact length of each native token byte piece; handles UTF-8 splits without character offsets"}\n    info["vocabulary_adaptation"] = {\n        "backbone": "Gemma3_270M", "backbone_parameters": 100326016,\n        "hidden_size": 640, "native_tokenizer_ids": len(ids), "protocol_tokens": 3,\n        "embedding_shape": [rows, 640], "embedding_and_head_tied": True,\n        "embedding_parameters": rows * 640, "total_parameters": total,\n        "v4_parameters": 268100736, "parameter_delta_from_v4": total - 268100736,\n        "automatic_native_special_tokens": False,\n        "packing_policy": "Defined native IDs -> 1..200000; PAD=0, BOS=200001, EOS=200002.",\n        "initialization": "Fresh random weights; shared seed/backbone digest; no checkpoint transfer.",\n        "cache_policy": "Reuse frozen raw bytes; native o200k encoding and context packings use separate hash/schema identities."}\n    write_json(destination / "identity.json", info)\n    write_json(destination / "native_ids_in_dense_order.json", ids)\n    print(f"Native o200k_base: {len(ids):,} defined IDs; tied table {rows:,} x 640; model {total:,} parameters.", flush=True)\n    return tokenizer, mapping, info\n\n\n\n\n"""Sequence-aware cache identity and input validation, embedded in the runner."""\n\n\ndef validate_training_config(config):\n    if (config["sequence_length"] != 1024 or config["long_sequence_length"] != 2048\n            or config["local_batch_per_chip"] != 3 or config["long_batch_per_chip"] != 1\n            or config["training_seconds"] != 1200. or config["updates_per_dispatch"] != 16\n            or config["short_phase_tokens"] != 3000000 or config["long_phase_tokens"] != 1000000\n            or config["batch_discovery_enabled"] is not False):\n        raise ValueError("V4.2.1 requires fixed 3x1024 / 1x2048, a 3M/1M token cycle and 1200 seconds.")\n    if not 0 < config["memory_budget_fraction"] < 1:\n        raise ValueError("Memory admission must leave device headroom.")\n\n\ndef tokenization_cache_key(tokenizer_sha256, raw_manifest_sha256, sequence_length):\n    identity = {"schema": "sequence-aware-token-cache-v3-native-tiktoken-bytes", "tokenizer_sha256": tokenizer_sha256,\n        "raw_manifest_sha256": raw_manifest_sha256, "packing_version": PACKING_VERSION,\n        "sequence_length": int(sequence_length)}\n    return hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:24]\n\n\ndef validate_tokenization_manifest(details, identity, raw_digest, sequence_length):\n    expected_key = tokenization_cache_key(identity["tokenizer_sha256"], raw_digest, sequence_length)\n    if (details.get("sequence_length") != sequence_length\n            or details.get("packing_version") != PACKING_VERSION\n            or details.get("cache_key") != expected_key\n            or details.get("raw_manifest_sha256") != raw_digest\n            or details.get("tokenizer", {}).get("tokenizer_sha256") != identity["tokenizer_sha256"]):\n        raise ValueError("Prepared tokenization metadata does not match context, tokenizer, corpus and packing. Existing cache preserved.")\n\n\ndef validate_sequence_arrays(arrays, sequence_length, split):\n    shape = arrays["tokens"].shape\n    if len(shape) != 2 or shape[0] < 1 or shape[1] != sequence_length:\n        raise ValueError(f"{split}: expected nonempty [rows, {sequence_length}] inputs, found {shape}.")\n    if any(arrays[key].shape != shape for key in INPUT_KEYS):\n        raise ValueError(f"{split}: token, mask, position or accounting array shapes disagree.")\n\n\ndef training_specification_matches(config, training):\n    return (config["local_batch_per_chip"] == 3 and config["long_batch_per_chip"] == 1\n        and config["training_seconds"] == 1200. and config["batch_discovery_enabled"] is False\n        and training["short"]["tokens"].shape[1:] == (1024,)\n        and training["long"]["tokens"].shape[1:] == (2048,))\n\n\ndef prepare_cached(config, work, output):\n    validate_training_config(config)\n    started = time.monotonic()\n    work, output = Path(work), Path(output)\n    shared = output.parent.parent / "V3-shared" / protocol_fingerprint()[:16]\n    shared.mkdir(parents=True, exist_ok=True)\n    lock = immutable_sources(shared)\n    raw = shared / "raw"\n    raw_reused = raw.exists()\n    if not raw_reused:\n        check_source_access(lock)\n        staging = shared / "raw.partial"\n        build_raw_corpus(staging, lock)\n        staging.rename(raw)\n    manifest = verify_raw(raw)\n    prepared = work / "prepared"\n    prepared.mkdir(exist_ok=False)\n    if raw_reused:\n        downloads = {"version": "V4.2.1", "raw_cache_reused": True, "source_jobs": 0,\n            "max_workers": DOWNLOAD_WORKERS, "peak_active_source_jobs": 0, "fetch_seconds": 0.}\n    else:\n        downloads = read_json(shared / "parallel_downloads.json")\n        downloads["raw_cache_reused"] = False\n    write_json(output / "parallel_downloads.json", downloads)\n    expected_jobs = len(LANGUAGES) + len({(b["repo"], b["subset"]) for b in recipe()["buckets"]})\n    checks = {"V4.2.1-021_DOWNLOAD_BOUND": downloads["peak_active_source_jobs"] <= 6,\n        "V4.2.1-022_PREFIX_CACHE": raw_reused or all((Path(item["directory"]) / "state.json").exists()\n            for item in downloads.get("jobs", {}).values()),\n        "V4.2.1-023_MERGE_ORDER": raw_reused or downloads["merge_order"][:len(LANGUAGES)] == ["wiki_" + lang for lang in LANGUAGES],\n        "V4.2.1-024_SOURCE_COMPLETE": raw_reused or downloads["source_jobs"] == expected_jobs == len(downloads["jobs"])}\n    write_json(output / "preparation_requirements.json", {key: "PASS" if value else "FAIL" for key, value in checks.items()})\n    if not all(checks.values()):\n        raise ValueError("Expanded data preparation failed validation.")\n    tokenizer, mapping, identity = prepare_tokenizer(config, prepared / "tokenizer")\n    raw_digest = sha256_file(raw / "manifest.json")\n    contexts = {}\n    missing_packing = any(not (shared / "tokenized" / tokenization_cache_key(\n        identity["tokenizer_sha256"], raw_digest, length)).exists() for length in (1024, 2048))\n    encoding_directory = None\n    encoding_report = {"status": "not_needed", "reason": "Both matching packed context caches already exist.",\n        "schema": ENCODING_SCHEMA, "encode_passes_this_run": 0}\n    if missing_packing:\n        encoding_directory, encoding_details, encoding_reused = prepared_encoding(\n            shared, raw, work, tokenizer, mapping, identity, raw_digest)\n        for split, counts in encoding_details["splits"].items():\n            if counts["original_bytes"] != manifest["original_bytes"][split]:\n                raise ValueError("Encoded original byte total differs: " + split)\n        encoding_report = {"status": "available", "schema": ENCODING_SCHEMA,\n            "cache_reused": encoding_reused, "encode_passes_this_run": 0 if encoding_reused else 1,\n            "encoding_manifest": encoding_details}\n        write_json(output / "encoding_manifest.json", encoding_details)\n    write_json(output / "encoding_preparation.json", encoding_report)\n    for length in (1024, 2048):\n        destination = prepared / str(length)\n        destination.mkdir(exist_ok=False)\n        key = tokenization_cache_key(identity["tokenizer_sha256"], raw_digest, length)\n        cache = shared / "tokenized" / key\n        reused = cache.exists()\n        if reused:\n            details = read_json(cache / "tokenization.json")\n            validate_tokenization_manifest(details, identity, raw_digest, length)\n            for name, digest in details["files"].items():\n                if sha256_file(cache / name) != digest:\n                    raise ValueError("Context tokenization cache checksum mismatch: " + name)\n                shutil.copyfile(cache / name, destination / name)\n        else:\n            details = {"tokenizer": identity, "raw_manifest_sha256": raw_digest, "sequence_length": length,\n                "packing_version": PACKING_VERSION, "cache_key": key, "splits": {}, "files": {}}\n            # Evaluate only at the fixed 1024 context. Long packing is training-only.\n            for split in (("train", "in_mix", "wiki") if length == 1024 else ("train",)):\n                arrays, packing = pack_encoded_split(encoding_directory, split, identity, length)\n                validate_sequence_arrays(arrays, length, split)\n                name = split + ".npz"\n                np.savez_compressed(destination / name, **arrays)\n                details["splits"][split] = {k: v for k, v in packing.items() if not k.startswith("assignments_")}\n                details["files"][name] = sha256_file(destination / name)\n                print(f"{length}: {split}: {packing[\'text_tokens\']:,} text targets, occupancy={packing[\'occupancy\']:.2%}", flush=True)\n                del arrays\n            write_json(destination / "tokenization.json", details)\n            temporary = cache.with_name(key + ".partial-" + str(os.getpid()))\n            temporary.mkdir(parents=True, exist_ok=False)\n            for name in [*details["files"], "tokenization.json"]:\n                shutil.copyfile(destination / name, temporary / name)\n            temporary.rename(cache)\n        write_json(destination / "tokenization.json", details)\n        contexts[str(length)] = {"manifest": details, "cache_reused": reused, "cache_key": key}\n        write_json(output / f"tokenization_{length}.json", details)\n    # Top-level metadata remains the fixed-context evaluation metadata.\n    shutil.copyfile(prepared / "1024/tokenization.json", prepared / "tokenization.json")\n    config = {**config, "tokenizer_identity": identity, "shared_directory": str(shared)}\n    write_json(prepared / "resolved_config.json", config)\n    shutil.copyfile(raw / "manifest.json", prepared / "manifest.json")\n    shutil.copyfile(raw / "manifest.json", output / "data_manifest.json")\n    shutil.copyfile(shared / "source-lock.json", output / "source-lock.json")\n    shutil.copytree(prepared / "tokenizer", output / "tokenizer")\n    shutil.copyfile(prepared / "tokenizer/identity.json", output / "tokenizer_identity.json")\n    write_json(output / "vocabulary_adaptation.json", identity["vocabulary_adaptation"])\n    write_json(output / "preparation.json", {"raw_cache_reused": raw_reused, "contexts": contexts,\n        "encoding": encoding_report,\n        "preparation_seconds": time.monotonic() - started, "raw_manifest_sha256": raw_digest,\n        "actual_training_bytes": manifest["original_bytes"]["train"], "target_training_bytes": 256 * 1024**2,\n        "tokenizer_sha256": identity["tokenizer_sha256"], "shared_directory": str(shared)})\n\n\ndef load_model_helpers(cpu_test=False):\n    accelerator_imports(cpu_test)\n    exec(MODEL_HELPERS, globals())\n\n\ndef make_model(config):\n    # Only the tied vocabulary table changes; retain the V4 transformer architecture.\n    from gemma import gm\n    model = gm.nn.Gemma3_270M(dtype=jnp.float32)\n    return model.clone(config=dataclasses.replace(model.config, num_embed=config["tokenizer_identity"]["model_rows"]))\n\n\nPILOT_VERSION = \'V4.2.1\'\n\ndef make_optimizer(config, params):\n    schedule = optax.join_schedules([\n        optax.linear_schedule(0., config["learning_rate"], config["warmup_updates"]),\n        optax.constant_schedule(config["learning_rate"]),\n    ], [config["warmup_updates"]])\n    decay = jax.tree.map(lambda value: value.ndim > 1, params)\n    transform = optax.chain(\n        optax.clip_by_global_norm(config["gradient_clip"]),\n        optax.adamw(schedule, b1=.9, b2=.999, eps=1e-8,\n                    weight_decay=config["weight_decay"], mask=decay))\n    flat = traverse_util.flatten_dict(params)\n    count = sum(value.size for value in flat.values())\n    routing = [{"path": "/".join(path), "shape": list(value.shape),\n                "parameters": value.size, "optimizer": "AdamW",\n                "weight_decay": value.ndim > 1}\n               for path, value in sorted(flat.items())]\n    return transform, {"name": "AdamW", "adamw_parameters": count,\n        "muon_parameters": 0, "learning_rate": config["learning_rate"],\n        "warmup_updates": config["warmup_updates"], "routing": routing}\n\n\ndef readable_evaluation(values, raw_bytes, sequence_length):\n    ln, lb, dn, db, eos = [np.asarray(value) for value in values]\n    if int(round(float(lb.sum()))) != raw_bytes or not np.isfinite(ln).all():\n        raise ValueError("Evaluation original-byte denominator or finite-loss check failed.")\n    languages = {language: {"bpb": float(ln[i] / (lb[i] * math.log(2))), "nll_nats": float(ln[i]),\n                            "original_utf8_bytes": int(lb[i])} for i, language in enumerate(LANGUAGES) if lb[i] > 0}\n    domains = {domain: {"bpb": float(dn[i] / (db[i] * math.log(2))), "original_utf8_bytes": int(db[i])}\n               for i, domain in enumerate(DOMAINS) if db[i] > 0}\n    return {"pooled_bpb": float(ln.sum() / (lb.sum() * math.log(2))),\n        "language_macro_bpb": float(np.mean([value["bpb"] for value in languages.values()])),\n        "legacy_12_language_macro_bpb": float(np.mean([languages[lang]["bpb"] for lang in LANGUAGES[:12] if lang in languages])),\n        "languages": languages, "domains": domains, "original_utf8_bytes": raw_bytes,\n        "eos_nll_nats_separate": float(eos), "scoring": f"all original text targets once; virtual EOS excluded from BPB; fixed {sequence_length}-token chunk context"}\n\n\ndef initial_backbone_digest(params):\n    digest = hashlib.sha256()\n    for path, value in sorted(traverse_util.flatten_dict(params).items()):\n        if path[0] == "embedder":\n            continue\n        digest.update("/".join(path).encode())\n        digest.update(np.asarray(jax.device_get(value)).tobytes())\n    return digest.hexdigest()\n\n\n\n"""Repair the Tokamax loss gradient boundary without changing its TPU math."""\n\n\n\n\n# Shared native-Gemma runtime assembled into both new notebooks.\n# This source is not executed by the artifact builder.\n\nTRAIN_KEYS = ("tokens", "targets", "mask", "positions", "segment_ids")\n\n\ndef token_losses(model, params, batch):\n    compute = jax.tree.map(lambda value: value.astype(jnp.bfloat16), params)\n    logits = model.apply({"params": compute}, tokens=batch["tokens"],\n        positions=batch["positions"],\n        attention_mask=packed_attention_mask(batch["segment_ids"])).logits.astype(jnp.float32)\n    with jax.named_scope("cross_entropy"):\n        selected = jnp.take_along_axis(logits, batch["targets"][..., None], axis=-1)[..., 0]\n        return jax.scipy.special.logsumexp(logits, axis=-1) - selected\n\n\ndef make_training_functions(model, optimizer, batch_size=1, mode="short"):\n    cursor_field = "short_rows" if mode == "short" else "long_rows"\n    def one_step(state, batch):\n        def objective(params):\n            losses = token_losses(model, params, batch)\n            count = jnp.sum(batch["mask"], dtype=jnp.float32)\n            nll = jnp.sum(jnp.where(batch["mask"], losses, 0.))\n            return nll / jnp.maximum(count, 1.), count\n        (loss, count), grads = jax.value_and_grad(objective, has_aux=True)(state.params)\n        norm = optax.global_norm(grads)\n        finite = jnp.isfinite(loss) & jnp.isfinite(norm) & (count > 0)\n        def apply(_):\n            with jax.named_scope("optimizer_update"):\n                delta, new_opt = optimizer.update(grads, state.opt_state, state.params)\n                next_params = optax.apply_updates(state.params, delta)\n            return state.replace(step=state.step + 1, params=next_params, opt_state=new_opt,\n                data_position=state.data_position + 1,\n                **{cursor_field: getattr(state, cursor_field) + batch_size})\n        return jax.lax.cond(finite, apply, lambda _: state, operand=None), (loss, count, norm, finite)\n    def group(state, training, requested):\n        initial = (state, jnp.array(0, jnp.int32), jnp.array(0., jnp.float32),\n                   jnp.array(0, jnp.int32), jnp.array(0., jnp.float32), jnp.array(True))\n        def condition(carry):\n            return (carry[1] < requested) & carry[5]\n        def body(carry):\n            state, steps, nll, count, max_norm, finite = carry\n            indices = (getattr(state, cursor_field) + jnp.arange(batch_size)) % training["tokens"].shape[0]\n            batch = jax.tree.map(lambda values: values[indices], training)\n            state, (loss, tokens, norm, good) = one_step(state, batch)\n            return (state, steps + good.astype(jnp.int32), nll + loss * tokens,\n                count + jnp.where(good, tokens, 0).astype(jnp.int32),\n                jnp.maximum(max_norm, norm), finite & good)\n        state, steps, nll, count, norm, finite = jax.lax.while_loop(condition, body, initial)\n        return state, {"updates": steps, "nll_sum": nll, "tokens": count,\n                       "max_gradient_norm": norm, "finite": finite}\n    return one_step, jax.jit(group, donate_argnums=(0,))\n\n\ndef make_evaluator(model):\n    """One executable for both held-out sets; split identity is runtime data."""\n    def evaluate(params, arrays):\n        initial = (jnp.zeros((2, len(LANGUAGES)), jnp.float32),\n                   jnp.zeros((2, len(LANGUAGES)), jnp.float32),\n                   jnp.zeros((2, len(DOMAINS)), jnp.float32),\n                   jnp.zeros((2, len(DOMAINS)), jnp.float32), jnp.zeros(2, jnp.float32))\n        def body(index, totals):\n            batch = jax.tree.map(lambda values: jax.lax.dynamic_index_in_dim(\n                values, index, keepdims=False), arrays)\n            losses = token_losses(model, params, batch)\n            nll = jnp.where(batch["text_mask"], losses, 0.).reshape(-1)\n            weights = batch["byte_weights"].reshape(-1).astype(jnp.float32)\n            language = batch["language_ids"].reshape(-1) + batch["split_id"] * len(LANGUAGES)\n            domain = batch["domain_ids"].reshape(-1) + batch["split_id"] * len(DOMAINS)\n            ln = jnp.bincount(language, weights=nll, length=2 * len(LANGUAGES)).reshape((2, -1))\n            lb = jnp.bincount(language, weights=weights, length=2 * len(LANGUAGES)).reshape((2, -1))\n            dn = jnp.bincount(domain, weights=nll, length=2 * len(DOMAINS)).reshape((2, -1))\n            db = jnp.bincount(domain, weights=weights, length=2 * len(DOMAINS)).reshape((2, -1))\n            eos = jnp.sum(jnp.where(batch["mask"] & ~batch["text_mask"], losses, 0.))\n            eos = jax.nn.one_hot(batch["split_id"], 2) * eos\n            return tuple(total + value for total, value in zip(totals, (ln, lb, dn, db, eos)))\n        return jax.lax.fori_loop(0, arrays["tokens"].shape[0], body, initial)\n    return jax.jit(evaluate)\n\n\ndef training_prefix_counts(arrays):\n    """CPU preparation only; no per-token source histogram in the training graph."""\n    batches = arrays["tokens"].shape[0]\n    counts = {"tokens": arrays["mask"].sum(axis=(1, 2), dtype=np.int64)}\n    weights = arrays["byte_weights"].reshape(-1)\n    for key, field, classes in (("domain_bytes", "domain_ids", len(DOMAINS)),\n                                 ("language_bytes", "language_ids", len(LANGUAGES))):\n        labels = arrays[field].reshape((batches, -1))\n        indices = (labels + np.arange(batches, dtype=np.int64)[:, None] * classes).reshape(-1)\n        counts[key] = np.bincount(indices, weights=weights,\n            minlength=batches * classes).astype(np.int64).reshape((batches, classes))\n    if int(counts["domain_bytes"].sum()) != int(counts["language_bytes"].sum()):\n        raise ValueError("Prepared source/language byte accounting differs.")\n    return {key: np.concatenate((np.zeros((1, *value.shape[1:]), np.int64),\n                np.cumsum(value, axis=0, dtype=np.int64)), axis=0)\n            for key, value in counts.items()}\n\n\ndef completed_prefix(prefix, updates):\n    batches = prefix.shape[0] - 1\n    passes, remainder = divmod(updates, batches)\n    return passes * prefix[-1] + prefix[remainder]\n\n\ndef verify_native_protocol():\n    started = time.monotonic()\n    segments = jnp.array([[0, 0, 1, 1, -1]], jnp.int32)\n    actual = np.asarray(jax.jit(packed_attention_mask)(segments))[0]\n    expected = np.array([[1, 0, 0, 0, 0], [1, 1, 0, 0, 0], [0, 0, 1, 0, 0],\n                         [0, 0, 1, 1, 0], [0, 0, 0, 0, 1]], np.bool_)\n    passed = bool(np.array_equal(actual, expected))\n    if not passed:\n        raise ValueError("Native packed attention mask failed causal/segment/padding verification.")\n    return {"passed": passed, "seconds": time.monotonic() - started,\n        "optimizer_updates": 0, "attention": "native Gemma", "loss": "native dense exact softmax",\n        "custom_kernel_equivalence": "not applicable; no Splash or Tokamax execution",\n        "mask_verified": passed}\n\n\n"""Training-scoped allocator samples, with explicit peak-counter limitations."""\n\ndef memory_snapshot(stats, elapsed):\n    stats = stats or {}\n    keys = ("bytes_in_use", "bytes_limit", "bytes_reserved", "peak_bytes_in_use", "peak_bytes_reserved")\n    sample = {"training_elapsed_seconds": float(elapsed)}\n    for key in keys:\n        value = stats.get(key)\n        sample[key] = int(value) if isinstance(value, (int, float)) and not isinstance(value, bool) and value >= 0 else None\n    limit = sample["bytes_limit"]\n    for key, result in (("bytes_in_use", "active_percent"), ("bytes_reserved", "reserved_percent")):\n        sample[result] = 100. * sample[key] / limit if limit and sample[key] is not None else None\n    return sample\n\n\ndef summarize_training_memory(before, samples, after, compiler, sampling_seconds):\n    observed = [before, *samples, after]\n    limits = [row["bytes_limit"] for row in observed if row.get("bytes_limit")]\n    limit = limits[-1] if limits else None\n    stable_limit = len(set(limits)) <= 1\n    def maximum(key):\n        values = [row[key] for row in observed if row.get(key) is not None]\n        return max(values) if values else None\n    active = maximum("bytes_in_use")\n    reserved = maximum("bytes_reserved")\n    old_peak, new_peak = before.get("peak_bytes_in_use"), after.get("peak_bytes_in_use")\n    # Only a new lifetime high-water mark, strictly reached within the training\n    # boundaries, establishes the actual allocator peak for that interval.\n    isolated = old_peak is not None and new_peak is not None and new_peak > old_peak\n    peak = new_peak if isolated else None\n    required = ("argument_size_in_bytes", "output_size_in_bytes", "alias_size_in_bytes", "temp_size_in_bytes")\n    estimate = (compiler[required[0]] + compiler[required[1]] - compiler[required[2]] + compiler[required[3]]) \\\n        if compiler and all(key in compiler for key in required) else None\n    def percent(value):\n        return 100. * value / limit if stable_limit and limit and value is not None else None\n    return {\n        "status": "available" if active is not None and limit and stable_limit else "unavailable",\n        "scope": "JAX allocator counters, synchronized training boundaries and approximately 2-second progress samples; before final evaluation and profiling.",\n        "device_limit_bytes": limit, "device_limit_gib": limit / 2**30 if limit else None,\n        "device_limit_stable": stable_limit,\n        "sampled_peak_active_bytes": active, "sampled_peak_active_gib": active / 2**30 if active is not None else None,\n        "sampled_peak_active_percent": percent(active),\n        "sampled_peak_reserved_bytes": reserved, "sampled_peak_reserved_percent": percent(reserved),\n        "training_allocator_peak_bytes": peak, "training_allocator_peak_percent": percent(peak),\n        "training_allocator_peak_status": "isolated_new_high_water_mark" if isolated else "not_isolatable",\n        "lifetime_allocator_peak_before_training_bytes": old_peak,\n        "lifetime_allocator_peak_after_training_bytes": new_peak,\n        "compiled_peak_buffer_estimate_bytes": estimate,\n        "compiled_peak_buffer_estimate_percent": percent(estimate),\n        "compiler_estimate_formula": "arguments + outputs - aliased buffers + temporary buffers; an estimate, not measured device usage",\n        "sampling_seconds_inside_training": sampling_seconds,\n        "sample_count": len(observed), "samples": observed,\n        "limitations": "Synchronized samples can miss temporary peaks. Lifetime counters include earlier phases unless a new peak was reached during training. Allocator counters may not expose every runtime-owned buffer. Reserved pool bytes are not active tensor bytes; compiler estimates are separate.",\n    }\n\n\n"""XProf measured-metric parsing; compiler estimates are reported separately.\n\nEmbedded into runner.py. Pure JSON parsers also support local fixture checks.\n"""\n\ndef _number(value):\n    if isinstance(value, dict):\n        value = value.get("v")\n    if isinstance(value, bool) or value is None:\n        return None\n    try:\n        result = float(str(value).rstrip("%"))\n    except (ValueError, TypeError):\n        return None\n    return result if math.isfinite(result) else None\n\n\ndef _normal_key(key):\n    return "".join(c for c in str(key).lower() if c.isalnum())\n\n\ndef _property_values(data):\n    """Preserve explicit zero; never turn a missing counter into zero."""\n    result = {}\n    def visit(node):\n        if isinstance(node, dict):\n            for key, value in node.items():\n                number = _number(value)\n                if number is not None:\n                    result.setdefault(_normal_key(key), number)\n                visit(value)\n        elif isinstance(node, list):\n            for value in node:\n                visit(value)\n    visit(data)\n    return result\n\n\ndef extract_utilization(overview):\n    props = _property_values(overview)\n    aliases = {\n        "matrix_unit_utilization": ("mxu_utilization_percent", "matrix_unit_utilization_percent"),\n        "memory_bandwidth_utilization": ("hbm_bw_utilization_percent", "hbm_utilization_percent"),\n        "host_idle_time": ("host_idle_time_percent",),\n        "device_idle_time": ("device_idle_time_percent",),\n    }\n    metrics = {}\n    for name, keys in aliases.items():\n        found = next(((key, props[_normal_key(key)]) for key in keys\n                      if _normal_key(key) in props), None)\n        valid = found is not None and 0 <= found[1] <= 100\n        metrics[name] = {"value": found[1] if valid else None, "unit": "percent",\n            "status": "available" if valid else "unavailable",\n            "source": "XProf overview_page",\n            "source_field": found[0] if valid else None,\n            "reason": None if valid else "XProf did not provide a supported percent counter; no FLOP or occupancy proxy used."}\n    return metrics\n\n\ndef _datatable_rows(data):\n    rows = []\n    def visit(node):\n        if isinstance(node, dict):\n            if isinstance(node.get("cols"), list) and isinstance(node.get("rows"), list):\n                columns = [(col.get("id") or col.get("label") or str(i), col.get("label", ""))\n                           for i, col in enumerate(node["cols"])]\n                for row in node["rows"]:\n                    item = {}\n                    for (key, label), cell in zip(columns, row.get("c", [])):\n                        value = cell.get("v") if isinstance(cell, dict) else cell\n                        item[_normal_key(key)] = value\n                        if label:\n                            item.setdefault(_normal_key(label), value)\n                    rows.append(item)\n            for value in node.values():\n                if isinstance(value, (dict, list)):\n                    visit(value)\n        elif isinstance(node, list):\n            for value in node:\n                visit(value)\n    visit(data)\n    return rows\n\n\ndef operation_category(name):\n    text = str(name).lower()\n    if "optimizer_update" in text or "adam" in text or "optax" in text:\n        return "optimizer_updates"\n    if "vocabulary_projection" in text or ("embedder" in text and "dot_general" in text) or "/decode/" in text:\n        return "vocabulary_projection"\n    if "cross_entropy" in text or "logsumexp" in text:\n        return "cross_entropy"\n    if any(key in text for key in ("/mlp/", "feedforward", "gating_einsum", "linear_einsum")):\n        return "mlp"\n    if any(key in text for key in ("/attn/", "q_einsum", "kv_einsum", "attn_vec")):\n        return "attention"\n    return "other_or_fused"\n\n\ndef extract_operation_timings(hlo_stats):\n    """Use exclusive HLO self-time only, never sum overlapping parent scopes."""\n    time_keys = ("selftimeus", "selftimeinus", "totalselftimeus", "totalselftimeinus")\n    name_keys = ("frameworkopname", "frameworkop", "tfopname", "opname", "hloname",\n                 "hloopname", "hloexpression", "hloopexpression", "hlooptext", "expression")\n    rows = []\n    for row in _datatable_rows(hlo_stats):\n        duration = next((_number(row[k]) for k in time_keys if k in row and _number(row[k]) is not None), None)\n        names = [str(row[k]) for k in name_keys if row.get(k)]\n        if duration is None or duration < 0 or not names:\n            continue\n        name = " | ".join(dict.fromkeys(names))\n        rows.append({"name": name, "self_time_us": duration, "category": operation_category(name)})\n    categories = {key: 0.0 for key in ("attention", "vocabulary_projection", "cross_entropy",\n                                      "optimizer_updates", "mlp", "other_or_fused")}\n    for row in rows:\n        categories[row["category"]] += row["self_time_us"]\n    total = sum(categories.values())\n    return {"status": "available" if rows else "unavailable",\n        "reason": None if rows else "XProf did not expose supported exclusive HLO timing columns; inspect the saved HLO/trace artifacts.",\n        "source": "XProf hlo_stats", "unit": "microseconds",\n        "scope": "Summed HLO self-time over profiled device cores; includes forward/backward; excludes idle. Not elapsed wall time.",\n        "classification": "Named HLO metadata; fused/unrecognized operations remain other_or_fused.",\n        "categories": {key: {"self_time_us": value if any(row["category"] == key for row in rows) else None,\n                               "percent_of_accounted_op_time": 100 * value / total\n                                   if total > 0 and any(row["category"] == key for row in rows) else None}\n                       for key, value in categories.items()},\n        "top_operations": sorted(rows, key=lambda row: row["self_time_us"], reverse=True)[:50]}\n\n\ndef _decode_tool_json(payload):\n    if isinstance(payload, bytes):\n        payload = payload.decode("utf-8")\n    return json.loads(payload) if isinstance(payload, str) else payload\n\n\ndef historical_comparison(config, tokenizer_hash, corpus_hash, throughput, bpb):\n    same_data = (tokenizer_hash == "7d4046bf0505a327dd5a0abbb427ecd4fc82f99c2ceaa170bc61ecde12809b0c"\n        and corpus_hash == "74d6d6305b893aa9403e8ac30c10b9d920c23bc7d3c9d020debbd5b3500d707a")\n    return {"reference_version": "V3.5", "reference_tokens_per_second": 45734.29605429257,\n        "reference_finewiki_macro_bpb": 2.9347027242183685,\n        "reference_batch": 2, "reference_sequence_length": 1024,\n        "reference_training_seconds": 60.014611282999795,\n        "current_sequence_length": config["sequence_length"],\n        "same_evaluation_context_length": config["sequence_length"] == 1024,\n        "current_training_contexts": [1024, 2048], "current_corpus": "expanded 256 MiB",\n        "current_batch": config["local_batch_per_chip"], "current_budget_seconds": config["training_seconds"],\n        "same_tokenizer_and_corpus": same_data,\n        "controlled_same_budget_comparison": False,\n        "throughput_ratio_to_reference": throughput / 45734.29605429257 if same_data else None,\n        "bpb_difference_from_reference": bpb - 2.9347027242183685 if same_data else None,\n        "note": "Descriptive historical comparison only. Corpus, language coverage, batch schedule, training context and duration differ. BPB and throughput do not isolate implementation efficiency."}\n\n\ndef extract_roofline_estimates(roofline):\n    """Use the aggregate Program row and its own peak units, never sum rates."""\n    tables = roofline if isinstance(roofline, list) else [roofline]\n    table = next((x for x in tables if isinstance(x, dict) and "cols" in x\n                  and "peak_flop_rate" in x.get("p", {})), None)\n    missing = {"status": "unavailable", "kind": "compiler_cost_model_estimate",\n        "reason": "No supported aggregate Program row and hardware peaks in roofline export."}\n    if table is None:\n        return missing\n    row = next((r for r in _datatable_rows(table) if r.get("category") == "Program"\n                and r.get("operation") == "Program" and r.get("step") == "Total"), None)\n    if row is None:\n        return missing\n    flop = _number(row.get("modelfloprate"))\n    hbm = _number(row.get("hbmbw"))\n    peak_flop = _number(table["p"].get("peak_flop_rate"))\n    peak_hbm = _number(table["p"].get("peak_hbm_bw"))\n    if any(x is None or x <= 0 for x in (peak_flop, peak_hbm)) or flop is None or hbm is None:\n        return missing\n    return {"status": "available", "kind": "compiler_cost_model_estimate", "source": "XProf roofline_model aggregate Program row",\n        "estimated_compute_tflops_per_second": flop / 1000., "peak_compute_tflops_per_second": peak_flop / 1000.,\n        "estimated_compute_percent_of_peak": 100 * flop / peak_flop,\n        "estimated_hbm_gib_per_second": hbm, "peak_hbm_gib_per_second": peak_hbm,\n        "estimated_hbm_percent_of_peak": 100 * hbm / peak_hbm,\n        "estimated_bound_by": row.get("boundby"),\n        "note": "Compiler FLOP/byte costs divided by observed duration. Not hardware-counter measurements or MXU busy time. Estimates may exceed physical peak; never clamp them."}\n\n\ndef raw_counter_evidence(profile_data):\n    rows = []\n    if profile_data is not None:\n        for plane in profile_data.planes:\n            for line in plane.lines:\n                if "counter" in line.name.lower():\n                    rows.append({"plane": plane.name, "line": line.name,\n                                 "event_count": sum(1 for _ in line.events)})\n    return {"tracks": rows, "counter_events": sum(row["event_count"] for row in rows)}\n\n\ndef profiling_requirements(metrics, training_memory):\n    def available(name):\n        return metrics.get(name, {}).get("status") == "available"\n    ops = metrics.get("operation_timings", {})\n    categories = ops.get("categories", {})\n    checks = {\n        "matrix_unit_utilization": available("matrix_unit_utilization"),\n        "memory_bandwidth_utilization": available("memory_bandwidth_utilization"),\n        "host_and_device_idle_time": available("host_idle_time") and available("device_idle_time"),\n        "operation_timings": available("operation_timings") and all(\n            categories.get(key, {}).get("self_time_us") is not None\n            for key in ("attention", "vocabulary_projection", "optimizer_updates")),\n        "training_memory": training_memory.get("status") == "available",\n    }\n    count = sum(checks.values())\n    return {"checks": {k: "PASS" if v else "FAIL" for k, v in checks.items()},\n        "requirements_status": "PASS" if all(checks.values()) else "FAIL",\n        "telemetry_status": "COMPLETE" if all(checks.values()) else "PARTIAL" if count else "UNAVAILABLE",\n        "available_requirement_groups": count, "required_groups": len(checks)}\n\n\ndef summarize_xprof(trace_dir, output, profile_data, completed_updates):\n    files = sorted(Path(trace_dir).rglob("*.xplane.pb")) + sorted(Path(trace_dir).rglob("*.xplane.riegeli"))\n    raw, errors = {}, {}\n    if files:\n        try:\n            from xprof.convert import raw_to_tool_data\n            # Reuse OpStats created by the first conversion; do not rebuild it\n            # four times or generate the redundant op_profile/Perfetto exports.\n            for index, tool in enumerate(("overview_page", "hlo_stats", "roofline_model")):\n                try:\n                    payload, _ = raw_to_tool_data.xspace_to_tool_data(\n                        [str(p) for p in files], tool, {"use_saved_result": index > 0})\n                    if payload is None:\n                        raise ValueError("XProf returned no tool data")\n                    raw[tool] = _decode_tool_json(payload)\n                    write_json(output / "profile_tools" / (tool + ".json"), raw[tool])\n                except Exception as error:\n                    errors[tool] = type(error).__name__ + ": " + str(error)\n        except Exception as error:\n            errors["converter"] = type(error).__name__ + ": " + str(error)\n    else:\n        errors["capture"] = "No raw TPU XPlane files were produced."\n    metrics = extract_utilization(raw.get("overview_page", {}))\n    evidence = {"hardware_counter_message_present": False, "device_timeline_present": False,\n                "device_type": None, "raw_counters": None}\n    try:\n        if profile_data is None and files and files[0].suffix == ".pb":\n            profile_data = jax.profiler.ProfileData.from_file(str(files[0]))\n        evidence["raw_counters"] = raw_counter_evidence(profile_data)\n    except Exception as error:\n        errors["raw_counter_audit"] = type(error).__name__ + ": " + str(error)\n    try:\n        from xprof.protobuf import op_stats_pb2\n        candidates = sorted(Path(trace_dir).rglob("ALL_HOSTS.op_stats*.pb"))\n        if candidates:\n            stats = op_stats_pb2.OpStats()\n            stats.ParseFromString(candidates[-1].read_bytes())\n            evidence["device_type"] = stats.run_environment.device_type\n            tpu = "TPU" in evidence["device_type"].upper()\n            evidence["device_timeline_present"] = tpu and stats.device_op_metrics_db.total_time_ps > 0\n            evidence["hardware_counter_message_present"] = tpu and stats.HasField("performance_counter_result")\n            if evidence["hardware_counter_message_present"]:\n                for name, field in (("matrix_unit_utilization", "matrix_unit_utilization_percent"),\n                                    ("memory_bandwidth_utilization", "hbm_utilization_percent")):\n                    value = float(getattr(stats.performance_counter_result, field))\n                    if math.isfinite(value) and 0 <= value <= 100:\n                        metrics[name].update(value=value, status="available", reason=None,\n                            source="XProf OpStats.performance_counter_result", source_field=field)\n    except Exception as error:\n        errors["counter_evidence"] = type(error).__name__ + ": " + str(error)\n    for name in ("matrix_unit_utilization", "memory_bandwidth_utilization"):\n        if not evidence["hardware_counter_message_present"]:\n            metrics[name].update(value=None, status="unavailable", source_field=None,\n                reason="No supported TPU utilization-counter message was exported. Default zeros are not measurements; inspect raw-counter audit. Runtime cause is not established.")\n    if not evidence["device_timeline_present"]:\n        metrics["device_idle_time"].update(value=None, status="unavailable", source_field=None,\n            reason="No TPU device timeline supports the overview idle percentage.")\n    metrics["device_idle_time"]["scope"] = "XProf device-operation timeline idle; not unused MXU capacity inside active operations."\n    reported_host_idle = metrics["host_idle_time"].get("value")\n    metrics["host_idle_time"].update(value=None, status="unavailable",\n        reason="XProf host-op idle accounting does not establish Python/CPU idle or blocked dispatch time. See separately measured host wait and dispatch gaps.")\n    metrics["host_idle_time"]["xprof_host_op_idle_percent_context_only"] = reported_host_idle\n    ops = extract_operation_timings(raw.get("hlo_stats", {}))\n    for category in ops["categories"].values():\n        duration = category["self_time_us"]\n        category["milliseconds_per_update"] = duration / (1000 * completed_updates) if duration is not None and completed_updates else None\n    metrics["operation_timings"] = ops\n    return {"measurements": metrics, "estimated_utilization": extract_roofline_estimates(raw.get("roofline_model", {})),\n        "counter_evidence": evidence, "extraction_errors": errors,\n        "xplane_files": [str(p.relative_to(trace_dir)) for p in files]}\n\n\ndef record_diagnostic_profile(state, training, group_jit, output, work, dispatches=4, updates_per_dispatch=8):\n    """One bounded capture of the existing executable; official weights stay saved."""\n    begun = time.monotonic()\n    trace_dir = Path(work) / "profile"\n    trace_dir.mkdir(exist_ok=False)\n    report = {"scope": "Post-checkpoint diagnostic, excluded from official training scores/tokens/time. No diagnostic weights saved.",\n        "requested_updates": dispatches * updates_per_dispatch, "completed_updates": 0,\n        "capture_status": "FAIL", "capture_error": None, "diagnostic_weights_saved": False,\n        "host_dispatch_gaps_ms": [], "host_enqueue_ms": [], "host_sync_wait_ms": [],\n        "dispatch_and_device_wait_ms": [], "capture_options": {\n            "host_tracer_level": 2, "python_tracer_level": 0, "enable_hlo_proto": True,\n            "tpu_trace_mode": "TRACE_COMPUTE_AND_SYNC", "tpu_perf_counters": True,\n            "create_perfetto_trace": False}}\n    tracing, previous_end, profile_data = False, None, None\n    try:\n        options = jax.profiler.ProfileOptions()\n        options.host_tracer_level = 2\n        options.python_tracer_level = 0\n        options.enable_hlo_proto = True\n        options.raise_error_on_start_failure = True\n        options.advanced_configuration = {"tpu_trace_mode": "TRACE_COMPUTE_AND_SYNC", "tpu_perf_counters": True}\n        jax.profiler.start_trace(str(trace_dir), create_perfetto_trace=False, profiler_options=options)\n        tracing = True\n        window_start = time.monotonic()\n        for index in range(dispatches):\n            begin = time.monotonic()\n            if previous_end is not None:\n                report["host_dispatch_gaps_ms"].append((begin - previous_end) * 1000)\n            with jax.profiler.StepTraceAnnotation("diagnostic_train", step_num=index):\n                enqueue_start = time.monotonic()\n                pending = group_jit(state, training, jnp.array(updates_per_dispatch, jnp.int32))\n                enqueue_end = time.monotonic()\n                state, values = jax.block_until_ready(pending)\n                end = time.monotonic()\n            report["host_enqueue_ms"].append((enqueue_end - enqueue_start) * 1000)\n            report["host_sync_wait_ms"].append((end - enqueue_end) * 1000)\n            report["dispatch_and_device_wait_ms"].append((end - begin) * 1000)\n            metrics = jax.device_get(values)\n            completed = int(metrics["updates"])\n            report["completed_updates"] += completed\n            if not bool(metrics["finite"]) or completed != updates_per_dispatch:\n                raise FloatingPointError("Nonfinite or incomplete diagnostic dispatch")\n            previous_end = end\n        report["diagnostic_window_seconds"] = time.monotonic() - window_start\n        report["capture_status"] = "PASS"\n    except Exception as error:\n        report["capture_error"] = type(error).__name__ + ": " + str(error)\n    finally:\n        if tracing:\n            try:\n                profile_data = jax.profiler.stop_trace()\n            except Exception as error:\n                report["stop_error"] = type(error).__name__ + ": " + str(error)\n                report["capture_status"] = "FAIL"\n        del state\n    report["capture_seconds"] = time.monotonic() - begun\n    try:\n        report.update(summarize_xprof(trace_dir, output, profile_data, report["completed_updates"]))\n    except Exception as error:\n        report["extraction_errors"] = {"summary": type(error).__name__ + ": " + str(error)}\n        report["measurements"] = extract_utilization({})\n        report["measurements"]["operation_timings"] = extract_operation_timings({})\n        report["estimated_utilization"] = {"status": "unavailable", "reason": "Profile summary failed."}\n    window = report.get("diagnostic_window_seconds", 0.)\n    report["host_timing"] = {\n        "enqueue_wall_ms": sum(report["host_enqueue_ms"]),\n        "synchronization_call_wall_ms": sum(report["host_sync_wait_ms"]),\n        "between_dispatch_wall_ms": sum(report["host_dispatch_gaps_ms"]),\n        "synchronization_call_percent_of_window": 100 * sum(report["host_sync_wait_ms"]) / (1000 * window) if window else None,\n        "note": "Host wall-clock durations around enqueue and synchronous completion APIs, including API overhead. These are not OS CPU-idle percentages; synchronization can overlap TPU execution."}\n    destination = output / "profile"\n    try:\n        shutil.copytree(trace_dir, destination)\n        report["trace_directory"] = str(destination)\n    except Exception as error:\n        report["trace_copy_error"] = type(error).__name__ + ": " + str(error)\n        report["trace_directory"] = str(trace_dir)\n    training_memory = read_json(output / "training_memory.json")\n    report.update(profiling_requirements(report["measurements"], training_memory))\n    if report["capture_status"] != "PASS":\n        report["requirements_status"] = "FAIL"\n    report["seconds_including_export_and_analysis"] = time.monotonic() - begun\n    write_json(output / "profiling.json", report)\n    return report\n\n\n\n\ndef train_pilot(work, output):\n    work, output = Path(work), Path(output)\n    version = PILOT_VERSION\n    with (output / "training.started").open("x") as marker:\n        marker.write(version + ": one training attempt; automatic retry disabled.\\n")\n    write_json(output / "attempt_status.json", {"stage": "pretraining", "optimizer_updates": 0,\n        "timed_training_started": False, "version": version})\n    environment = preflight()\n    write_json(output / "preflight.json", environment)\n    load_model_helpers()\n    data_dir = work / "prepared"\n    config = read_json(data_dir / "resolved_config.json")\n    validate_training_config(config)\n    manifest = verify_raw(Path(config["shared_directory"]) / "raw")\n    tokenization = read_json(data_dir / "tokenization.json")\n    identity = config["tokenizer_identity"]\n    validate_tokenization_manifest(tokenization, identity, sha256_file(data_dir / "manifest.json"), config["sequence_length"])\n    if tokenization["raw_manifest_sha256"] != sha256_file(data_dir / "manifest.json"):\n        raise ValueError("Tokenized and raw corpus manifests differ.")\n    if shutil.disk_usage(work).free < 3 * 1024**3:\n        raise RuntimeError("The weights-only artifact requires at least 3 GiB free local disk.")\n    host, orders, prefixes, context_metadata = {}, {}, {}, {}\n    for mode, length in (("short", 1024), ("long", 2048)):\n        directory = data_dir / str(length)\n        details = read_json(directory / "tokenization.json")\n        validate_tokenization_manifest(details, identity, sha256_file(data_dir / "manifest.json"), length)\n        context_metadata[mode] = details\n        if sha256_file(directory / "train.npz") != details["files"]["train.npz"]:\n            raise ValueError("Training input checksum mismatch: " + mode)\n        with np.load(directory / "train.npz", allow_pickle=False) as archive:\n            arrays = {key: archive[key] for key in INPUT_KEYS}\n        validate_sequence_arrays(arrays, length, mode)\n        if int(arrays["byte_weights"].sum()) != manifest["original_bytes"]["train"]:\n            raise ValueError("Training byte attribution mismatch: " + mode)\n        arrays, orders[mode] = row_training_arrays(arrays, config["seed"])\n        prefixes[mode] = row_prefix_counts(arrays)\n        host[mode] = {key: arrays[key] for key in TRAIN_KEYS}\n        del arrays\n    eval_parts = {}\n    for split in ("in_mix", "wiki"):\n        directory = data_dir / "1024"\n        name = split + ".npz"\n        if sha256_file(directory / name) != tokenization["files"][name]:\n            raise ValueError("Evaluation checksum mismatch: " + split)\n        with np.load(directory / name, allow_pickle=False) as archive:\n            arrays = {key: archive[key] for key in INPUT_KEYS}\n        validate_sequence_arrays(arrays, 1024, split)\n        if int(arrays["byte_weights"].sum()) != manifest["original_bytes"][split]:\n            raise ValueError("Evaluation byte attribution mismatch: " + split)\n        eval_parts[split], _ = batch_training_arrays(arrays, 1, config["seed"])\n    eval_host = {key: np.concatenate((eval_parts["in_mix"][key], eval_parts["wiki"][key]), axis=0)\n        for key in INPUT_KEYS}\n    eval_host["split_id"] = np.concatenate((np.zeros(eval_parts["in_mix"]["tokens"].shape[0], np.int32),\n        np.ones(eval_parts["wiki"]["tokens"].shape[0], np.int32)))\n    input_bytes_by_mode = {mode: sum(value.nbytes for value in values.values()) for mode, values in host.items()}\n    input_bytes = sum(input_bytes_by_mode.values())\n    eval_bytes = sum(value.nbytes for value in eval_host.values())\n    phase = time.monotonic()\n    training, validation = jax.device_put((host, eval_host))\n    jax.block_until_ready((training, validation))\n    input_transfer_seconds = time.monotonic() - phase\n    resident = all(jax.devices()[0] in leaf.devices() for leaf in jax.tree.leaves(training))\n    del host, arrays, eval_parts, eval_host\n    config["other_resident_bytes_short"] = input_bytes_by_mode["long"] + eval_bytes\n    config["other_resident_bytes_long"] = input_bytes_by_mode["short"] + eval_bytes\n    print(f"Both context inputs on TPU: {input_bytes / 1024**2:.1f} MiB; exact CPU row accounting prepared.", flush=True)\n    write_json(output / "training_accounting.json", {"method": "Independent context row cursors, fixed shuffled order; no batch padding or cursor reset",\n        "contexts": {mode: {"rows_per_pass": training[mode]["tokens"].shape[0],\n            "tokens_per_pass": int(prefixes[mode]["tokens"][-1]), "training_input_bytes": input_bytes_by_mode[mode]}\n            for mode in training}, "evaluation_context": 1024,\n        "note": "Both contexts reuse the same frozen text. Byte exposure includes repeats, not unique corpus bytes."})\n    write_json(output / "config.json", config)\n    write_json(output / "environment.json", environment)\n\n    phase = time.monotonic()\n    model = make_model(config)\n    if model.config.final_logit_softcap is not None:\n        raise ValueError("This protocol requires native uncapped Gemma 270M logits.")\n    params = jax.jit(lambda key: model.init(key, tokens=jnp.ones((1, 1), jnp.int32))["params"])(\n        jax.random.PRNGKey(config["seed"]))\n    jax.block_until_ready(params)\n    flat = traverse_util.flatten_dict(params)\n    parameter_count = sum(value.size for value in flat.values())\n    embedding_shape = (identity["model_rows"], 640)\n    tied = (flat[("embedder", "input_embedding")].shape == embedding_shape\n            and sum(value.shape == embedding_shape for value in flat.values()) == 1)\n    if parameter_count != identity["expected_parameters"] or not tied or not all(\n            value.dtype == jnp.float32 for value in flat.values()):\n        raise ValueError("Parameter count, tied table, or FP32 master weights differ.")\n    backbone_digest = initial_backbone_digest(params)\n    backbone_key = hashlib.sha256(json.dumps({"gemma": environment["packages"].get("gemma"),\n        "jax": environment["jax"], "seed": config["seed"], "backbone": "Gemma3_270M"},\n        sort_keys=True).encode()).hexdigest()[:16]\n    fingerprint_path = Path(config["shared_directory"]) / ("backbone-" + backbone_key + ".json")\n    fingerprint = {"sha256": backbone_digest, "transformer_and_norm_parameters": 100326016,\n                   "seed": config["seed"]}\n    if fingerprint_path.exists() and read_json(fingerprint_path) != fingerprint:\n        raise ValueError("Transformer initialization differs from the shared baseline.")\n    write_json(fingerprint_path, fingerprint)\n    write_json(output / "backbone_initialization.json", fingerprint)\n    probe_path = next(path for path, value in sorted(flat.items())\n                      if path[0].startswith("layer_") and value.ndim > 1)\n    def probe(parameters):\n        return np.asarray(jax.device_get(traverse_util.flatten_dict(parameters)[probe_path].reshape(-1)[:64]))\n    initial_probe = probe(params)\n    optimizer, partition = make_optimizer(config, params)\n    routing = partition.pop("routing")\n    routing_ok = (sum(row["parameters"] for row in routing) == parameter_count\n                  and all(row["optimizer"] == config["optimizer_label"] for row in routing)\n                  and partition["muon_parameters"] + partition["adamw_parameters"] == parameter_count)\n    if not routing_ok:\n        raise ValueError("Optimizer routing does not cover every parameter with the requested optimizer.")\n    write_json(output / "optimizer_parameter_routing.json", {"parameters": parameter_count,\n        "optimizer": config["optimizer_label"], "all_parameters_accounted_for": routing_ok,\n        "leaves": routing})\n    write_json(output / "optimizer_partition.json", partition)\n    state = PilotState(jnp.array(0, jnp.int32), params, optimizer.init(params),\n        jax.random.PRNGKey(config["seed"]), jnp.array(0, jnp.int32),\n        jnp.array(0, jnp.int32), jnp.array(0, jnp.int32))\n    del params, flat\n    jax.block_until_ready(state)\n    initialization_seconds = time.monotonic() - phase\n    protocol_report = verify_native_protocol()\n    write_json(output / "kernel_verification.json", protocol_report)\n    functions, analyses, all_analyses, compilation_events = {}, {"short": {}, "long": {}}, {}, []\n    def compile_shape(mode, batch, current_state, scope):\n        if (mode, batch) not in (("short", 3), ("long", 1)) or scope != "before_training":\n            raise ValueError("Only fixed 3x1024 and 1x2048 may compile, before training starts.")\n        key = (mode, batch)\n        if key in functions:\n            return all_analyses[key]\n        begun = time.monotonic()\n        _, group = make_training_functions(model, optimizer, batch, mode)\n        executable = group.lower(current_state, training[mode], jnp.array(0, jnp.int32)).compile()\n        analysis = compiler_memory(executable)\n        functions[key] = executable\n        all_analyses[key] = analysis\n        analyses[mode][batch] = analysis\n        peak = compiler_peak(analysis)\n        other = config["other_resident_bytes_" + mode]\n        compilation_events.append({"mode": mode, "batch": batch, "sequence_length": 1024 if mode == "short" else 2048,\n            "seconds": time.monotonic() - begun, "scope": scope, "official_step_at_compile": int(current_state.step),\n            "compiler_memory": analysis, "compiler_peak_plus_other_resident_bytes": peak + other if peak is not None else None})\n        write_json(output / "compilation_events.json", compilation_events)\n        return analysis\n    print("Compiling fixed 3x1024, fixed 1x2048 and 1024-context evaluator; zero optimizer updates...", flush=True)\n    phase = time.monotonic()\n    for mode, batch in (("short", 3), ("long", 1)):\n        analysis = compile_shape(mode, batch, state, "before_training")\n        limit = (jax.devices()[0].memory_stats() or {}).get("bytes_limit")\n        peak = compiler_peak(analysis)\n        if peak is not None and limit and peak + config["other_resident_bytes_" + mode] > limit * config["memory_budget_fraction"]:\n            raise RuntimeError("Required initial context exceeds the conservative memory budget: " + mode)\n    evaluation_jit = make_evaluator(model)\n    evaluator = evaluation_jit.lower(state.params, validation).compile()\n    state, warmup_metrics = jax.block_until_ready(functions[("short", 3)](state, training["short"], jnp.array(0, jnp.int32)))\n    if int(state.step) != 0 or int(state.short_rows) != 0 or int(state.long_rows) != 0:\n        raise ValueError("Compilation preparation performed an optimizer update.")\n    compilation_seconds = time.monotonic() - phase\n    write_json(output / "dispatch_preparation.json", {"dispatch": "compiled executable",\n        "training_warmup_requested_updates": 0, "training_warmup_completed_updates": int(warmup_metrics["updates"]),\n        "pretraining_optimizer_step": int(state.step), "initial_training_executables": 2,\n        "evaluation_executables": 1, "fixed_training_shapes": [[3, 1024], [1, 2048]], "adaptive_compilation": "disabled; no discovery"})\n    def evaluate(parameters):\n        values = jax.block_until_ready(evaluator(parameters, validation))\n        return {split: readable_evaluation(tuple(value[index] for value in values),\n                    manifest["original_bytes"][split], config["sequence_length"])\n                for index, split in enumerate(("in_mix", "wiki"))}\n    phase = time.monotonic()\n    before = evaluate(state.params)\n    initial_eval_seconds = time.monotonic() - phase\n    write_json(output / "timed_training.started.json", {"version": version,\n        "stage": "timed training", "training_seconds": config["training_seconds"]})\n    write_json(output / "attempt_status.json", {"version": version, "stage": "timed training",\n        "timed_training_started": True, "optimizer_updates": None})\n    print(f"Initial FineWiki macro BPB={before[\'wiki\'][\'language_macro_bpb\']:.5f}. Starting ONE {config[\'training_seconds\']:g}-second {version} run, batch {config[\'local_batch_per_chip\']} x {config[\'sequence_length\']}...", flush=True)\n    state, adaptive = fixed_train(state, training, prefixes, functions, config, output)\n    log_rows = adaptive["rows"]\n    updates, tokens_processed = adaptive["updates"], adaptive["tokens"]\n    training_seconds, last_start = adaptive["training_seconds"], adaptive["last_start"]\n    used_keys = {(row["mode"], row["batch"]) for row in log_rows}\n    used_analyses = [all_analyses[key] for key in used_keys if all_analyses[key] is not None]\n    compiled_memory = max(used_analyses, key=compiler_peak) if used_analyses else None\n    training_memory = summarize_training_memory(adaptive["memory_before"], adaptive["samples"],\n        adaptive["memory_after"], compiled_memory, adaptive["sampling_seconds"])\n    peaks_with_other = [compiler_peak(all_analyses[key]) + config["other_resident_bytes_" + key[0]]\n        for key in used_keys if all_analyses[key] is not None]\n    training_memory["compiled_peak_plus_other_resident_bytes"] = max(peaks_with_other) if peaks_with_other else None\n    limit = training_memory["device_limit_bytes"]\n    training_memory["compiled_peak_plus_other_resident_percent"] = (\n        max(peaks_with_other) * 100 / limit if peaks_with_other and limit else None)\n    training_memory["compiler_estimates_by_shape"] = compilation_events\n    write_json(output / "training_memory.json", training_memory)\n    if not log_rows:\n        raise RuntimeError("No optimizer updates completed.")\n    domain_bytes = sum(completed_prefix(prefixes[mode]["domain_bytes"], rows)\n        for mode, rows in adaptive["rows_used"].items())\n    language_bytes = sum(completed_prefix(prefixes[mode]["language_bytes"], rows)\n        for mode, rows in adaptive["rows_used"].items())\n    expected_tokens = sum(int(completed_prefix(prefixes[mode]["tokens"], rows))\n        for mode, rows in adaptive["rows_used"].items())\n    if (tokens_processed != expected_tokens or int(state.short_rows) != adaptive["rows_used"]["short"]\n            or int(state.long_rows) != adaptive["rows_used"]["long"]):\n        raise ValueError("CPU/device token or context cursor accounting differs.")\n    changed = not np.array_equal(initial_probe, probe(state.params))\n    final_finite = bool(jax.device_get(jax.jit(lambda parameters: jnp.all(jnp.stack([\n        jnp.all(jnp.isfinite(value)) for value in jax.tree.leaves(parameters)])))(state.params)))\n    phase = time.monotonic()\n    after = evaluate(state.params)\n    final_eval_seconds = time.monotonic() - phase\n    memory_stats = jax.devices()[0].memory_stats()\n    phase = time.monotonic()\n    metadata = {"version": version, "protocol": PROTOCOL, "config": config,\n        "parameter_count": parameter_count, "initialization": "random; zero pretrained weights",\n        "backbone_initialization_sha256": backbone_digest, "optimizer": partition,\n        "raw_manifest_sha256": sha256_file(data_dir / "manifest.json"),\n        "tokenization_manifest_sha256": sha256_file(data_dir / "tokenization.json"),\n        "packing_version": PACKING_VERSION, "data_order_sha256": orders,\n        "training_rows_per_context": {mode: values["tokens"].shape[0] for mode, values in training.items()}, "updates": updates,\n        "context_rows_consumed": adaptive["rows_used"], "final_short_batch": adaptive["short_batch"],\n        "final_long_batch": adaptive["long_batch"], "batch_policy": adaptive["batch_policy"],\n        "data_position": int(state.data_position), "checkpoint_fields": ["params"],\n        "checkpoint_kind": "weights_only", "optimizer_resume_supported": False}\n    print("Saving ONE FP32 weights-only artifact; optimizer resume is not included...", flush=True)\n    local_checkpoint = work / "weights.npz"\n    leaves = save_archive(local_checkpoint, {"params": state.params}, metadata)\n    digest = sha256_file(local_checkpoint)\n    partial = output / "weights.npz.partial"\n    shutil.copyfile(local_checkpoint, partial)\n    if sha256_file(partial) != digest:\n        raise ValueError("Drive weights checksum differs from the local artifact.")\n    checkpoint = output / "weights.npz"\n    partial.rename(checkpoint)\n    with zipfile.ZipFile(checkpoint) as archive:\n        saved = json.loads(archive.read("checkpoint.json"))\n        schema_verified = (set(saved["tree"]) == {"params"} and saved["metadata"] == metadata\n            and sum(name.startswith("arrays/") for name in archive.namelist()) == leaves)\n    if not schema_verified:\n        raise ValueError("Saved weights schema or array count differs.")\n    write_json(output / "checkpoint_manifest.json", {"sha256": digest,\n        "bytes": checkpoint.stat().st_size, "array_leaves": leaves,\n        "kind": "weights_only", "optimizer_resume_supported": False,\n        "verification": "local/Drive SHA256 equality plus weights schema and leaf count",\n        "restore_executed": False})\n    checkpoint_seconds = time.monotonic() - phase\n    bytes_processed = int(domain_bytes.sum())\n    prefix_id = config["requirement_prefix"]\n    checks = {\n        "001_ONE_ATTEMPT": (output / "training.started").exists(),\n        "002_ONE_V5E": environment["platform"] == "tpu" and environment["device_count"] == 1,\n        "003_SOURCE_PINS": set(manifest["source_revisions"]) == set(SOURCE_REPOS),\n        "004_FROZEN_CORPUS": tokenization["raw_manifest_sha256"] == sha256_file(data_dir / "manifest.json"),\n        "005_SPLIT_SEPARATION": manifest["cross_split_source_groups"] == 0,\n        "006_LOSSLESS_TOKENIZATION": all(tokenization["splits"][split]["original_bytes"] == manifest["original_bytes"][split]\n                                          for split in ("train", "in_mix", "wiki")),\n        "007_INTACT_PACKING": all(tokenization["splits"][split]["targets_preserved"] and\n            tokenization["splits"][split]["context_preserved"] for split in ("train", "in_mix", "wiki")),\n        "008_NATIVE_ATTENTION_MASK": protocol_report["passed"],\n        "009_RESIDENT_INPUTS": resident,\n        "010_GEMMA_BACKBONE": parameter_count == identity["expected_parameters"] and parameter_count < 300000000 and tied,\n        "011_OPTIMIZER_ROUTING": routing_ok,\n        "012_TIMED_DISPATCH": all(row["start_seconds"] < config["training_seconds"] for row in log_rows),\n        "013_DISPATCH_BOUND": all(1 <= row["updates"] <= 16 for row in log_rows),\n        "014_FINITE_UPDATES": all(row["finite"] for row in log_rows) and final_finite,\n        "015_ZERO_EXTRA_UPDATES": int(state.step) == updates and int(state.data_position) == updates,\n        "016_FIXED_EVALUATION": len(after["wiki"]["languages"]) == len(LANGUAGES) and all(\n            after[split]["original_utf8_bytes"] == before[split]["original_utf8_bytes"] for split in ("in_mix", "wiki")),\n        "017_WEIGHTS_ARTIFACT": schema_verified,\n        "018_CHANGED_WEIGHTS": changed,\n        "019_MINIMUM_UPDATES": updates >= config["minimum_updates"],\n        "020_BYTE_ACCOUNTING": bytes_processed == int(language_bytes.sum()) and bytes_processed > 0 and expected_tokens == tokens_processed,\n        "025_NATIVE_TRAINING": training_specification_matches(config, training),\n        "026_COMBINED_EVALUATOR": validation["split_id"].shape[0] == validation["tokens"].shape[0],\n        "027_CPU_ACCOUNTING": all(not set(EXTRA_KEYS) & set(values) for values in training.values()),\n        "030_FIXED_BATCHES": all(row["batch"] == (3 if row["mode"] == "short" else 1) for row in log_rows),\n        "031_TOKEN_SCHEDULE": all(row["mode"] == context_phase(row["total_tokens"] - row["tokens"], 3) for row in log_rows),\n        "033_NO_DISCOVERY": adaptive["batch_policy"]["discovery_enabled"] is False and not adaptive["checks"],\n        "034_ONLY_REQUESTED_SHAPES": set(functions) == {("short", 3), ("long", 1)},\n        "035_NO_TRAINING_COMPILATION": all(event["scope"] == "before_training" for event in compilation_events),\n        "032_CONTEXT_CURSORS": int(state.short_rows) == adaptive["rows_used"]["short"] and int(state.long_rows) == adaptive["rows_used"]["long"],\n        "028_WEIGHTS_ONLY": saved["metadata"]["optimizer_resume_supported"] is False,\n        "029_NO_RESTORE_CHECK": read_json(output / "checkpoint_manifest.json")["restore_executed"] is False,\n    }\n    checks = {prefix_id + "-" + key: value for key, value in checks.items()}\n    checks.update({key: value == "PASS" for key, value in read_json(output / "preparation_requirements.json").items()})\n    downloads = read_json(output / "parallel_downloads.json")\n    elapsed = time.monotonic() - config["notebook_start_monotonic"]\n    rate = config["compute_units_per_hour"]\n    comparison = historical_comparison(config, identity["tokenizer_sha256"],\n        sha256_file(data_dir / "manifest.json"), tokens_processed / training_seconds,\n        after["wiki"]["language_macro_bpb"])\n    goals = {"throughput_at_least_v35_reference": tokens_processed / training_seconds >= 45734.29605429257,\n        "pretraining_compilation_seconds_at_most_180": compilation_seconds <= 180.,\n        "notebook_seconds_at_most_1800": elapsed <= 1800.,\n        "occupancy_at_least_95_percent": tokens_processed / adaptive["positions"] >= .95}\n    improvement = "PASS" if all(goals.values()) else "FAIL"\n    result = {"version": version, "protocol": PROTOCOL, "run_count": 1, "config": config,\n        "device": environment["device_kind"], "model_parameters": parameter_count, "optimizer": partition,\n        "tokenizer_identity": identity, "vocabulary_adaptation": identity["vocabulary_adaptation"],\n        "source_downloads": {key: downloads[key] for key in\n            ("source_jobs", "max_workers", "peak_active_source_jobs", "fetch_seconds", "raw_cache_reused")},\n        "initial_backbone_sha256": backbone_digest, "input_transfer_seconds": input_transfer_seconds,\n        "resident_training_bytes": input_bytes, "initialization_seconds": initialization_seconds,\n        "kernel_verification_seconds": protocol_report["seconds"], "compilation_seconds": compilation_seconds,\n        "compilation_timing_scope": "fixed 3x1024, fixed 1x2048 and fixed-context evaluator, all before training; no training-time compilation",\n        "compilation_events": compilation_events,\n        "adaptive_compilation_seconds_inside_training": sum(e["seconds"] for e in compilation_events if e["scope"] == "inside_training_budget"),\n        "dispatch_preparation": read_json(output / "dispatch_preparation.json"),\n        "initial_eval_seconds": initial_eval_seconds, "final_eval_seconds": final_eval_seconds,\n        "training_seconds": training_seconds,\n        "training_overrun_seconds": max(0., training_seconds - config["training_seconds"]),\n        "last_dispatch_start_seconds": last_start, "host_dispatches": len(log_rows),\n        "updates": updates, "scored_tokens": tokens_processed,\n        "tokens_per_second": tokens_processed / training_seconds,\n        "processed_original_bytes_attributed": bytes_processed,\n        "original_bytes_per_second_attributed": bytes_processed / training_seconds,\n        "byte_accounting_note": "CPU batch-prefix sums using tokenizer offset attribution; repeated passes included, not unique bytes",\n        "training_domain_byte_shares": {name: int(domain_bytes[index]) / bytes_processed for index, name in enumerate(DOMAINS)},\n        "training_language_byte_shares": {name: int(language_bytes[index]) / bytes_processed for index, name in enumerate(LANGUAGES)},\n        "training_occupancy": tokens_processed / adaptive["positions"],\n        "prepared_occupancy": tokenization["splits"]["train"]["occupancy"],\n        "completed_data_passes_by_context": {mode: rows / training[mode]["tokens"].shape[0] for mode, rows in adaptive["rows_used"].items()},\n        "final_short_batch": adaptive["short_batch"], "final_long_batch": adaptive["long_batch"],\n        "batch_policy": adaptive["batch_policy"],\n        "maximum_short_batch_executed": max((row["batch"] for row in log_rows if row["mode"] == "short"), default=0),\n        "maximum_long_batch_executed": max((row["batch"] for row in log_rows if row["mode"] == "long"), default=0),\n        "context_rows_consumed": adaptive["rows_used"], "batch_checks": adaptive["checks"],\n        "context_transitions": adaptive["transitions"], "data_order_sha256": orders,\n        "tokens_by_context": {mode: sum(row["tokens"] for row in log_rows if row["mode"] == mode) for mode in training},\n        "updates_by_context": {mode: sum(row["updates"] for row in log_rows if row["mode"] == mode) for mode in training},\n        "data_manifest_sha256": sha256_file(data_dir / "manifest.json"),\n        "initial_eval": before, "final_eval": after,\n        "primary_metric": "FineWiki mean of 16 per-language BPB values at fixed 1024-token context; lower is better",\n        "legacy_12_language_macro_bpb": {"initial": before["wiki"]["legacy_12_language_macro_bpb"], "final": after["wiki"]["legacy_12_language_macro_bpb"]},\n        "checkpoint_kind": "weights_only", "optimizer_resume_supported": False,\n        "checkpoint_save_verify_seconds": checkpoint_seconds, "checkpoint_bytes": checkpoint.stat().st_size,\n        "checkpoint_sha256": digest, "checkpoint_array_leaves": leaves,\n        "memory_stats": memory_stats, "compiled_memory_estimate": compiled_memory,\n        "training_memory": training_memory,\n        "head": {"kind": "native_dense_exact_softmax", "chunked": False},\n        "learning_rate_schedule": {"kind": "five_update_warmup_then_constant", "cooldown": False,\n            "warmup_updates": config["warmup_updates"], "peak": config["learning_rate"]},\n        "checkpoint_restoration_check": "disabled_by_user", "report_finalized": False,\n        "memory_scope": "Training allocator samples and high-water counters; compiler estimate separately",\n        "notebook_elapsed_seconds": elapsed,\n        "compute_units_estimate": {"rate_per_hour": rate, "training": training_seconds * rate / 3600,\n            "through_report": elapsed * rate / 3600, "overhead_through_report": (elapsed - training_seconds) * rate / 3600},\n        "cost_note": "2.92 CU/hour supplied by user; connected runtime after report adds cost; no USD conversion assumed",\n        "historical_v2_tokens_per_second": 45559.448,\n        "historical_comparison_note": "Native o200k, expanded corpus and 16-language holdout, fixed 3x1024/1x2048 and 1200 seconds differ from V3.5. No controlled speed/quality claim.",\n        "historical_comparison": comparison, "performance_goals": goals,\n        "performance_goal_status": improvement,\n        "requirements": {key: "PASS" if passed else "FAIL" for key, passed in checks.items()},\n        "engineering_status": "PASS" if all(checks.values()) else "FAIL",\n        "learning_status": "PASS" if after["wiki"]["language_macro_bpb"] < before["wiki"]["language_macro_bpb"] else "FAIL"}\n    write_json(output / "result.json", result)\n    write_json(output / "attempt_status.json", {"version": version, "stage": "result saved",\n        "timed_training_started": True, "optimizer_updates": updates})\n    print(json.dumps({name: result[name] for name in ("engineering_status", "learning_status",\n        "performance_goal_status", "updates", "training_seconds", "tokens_per_second",\n        "original_bytes_per_second_attributed")}, indent=2), flush=True)\n    print(f"FineWiki macro BPB: {before[\'wiki\'][\'language_macro_bpb\']:.6f} -> {after[\'wiki\'][\'language_macro_bpb\']:.6f}", flush=True)\n    if not all(checks.values()):\n        raise RuntimeError(version + " engineering requirement failed; result.json was saved.")\n    # Scores and saved weights are final before disposable diagnostic updates.\n    write_json(output / "attempt_status.json", {"version": version, "stage": "profiling",\n        "timed_training_started": True, "optimizer_updates": updates})\n    diagnostic_shapes = []\n    def profile_dispatch(current, ignored, requested):\n        mode = "short" if len(diagnostic_shapes) < 2 else "long"\n        batch = adaptive[mode + "_batch"]\n        diagnostic_shapes.append({"mode": mode, "batch": batch, "sequence_length": 1024 if mode == "short" else 2048})\n        return functions[(mode, batch)](current, training[mode], requested)\n    profile = record_diagnostic_profile(state, training, profile_dispatch, output, work,\n        config["profile_dispatches"], config["profile_updates_per_dispatch"])\n    profile["diagnostic_shapes"] = diagnostic_shapes\n    profile["operation_timing_note"] = (f"Mixed diagnostic: 16 updates at {adaptive[\'short_batch\']}x1024, then "\n        f"16 at {adaptive[\'long_batch\']}x2048. Category ms/update average these shapes; not per-context timings.")\n    write_json(output / "profiling.json", profile)\n    del state\n    result["profiling"] = profile\n    result["profiling_status"] = profile["requirements_status"]\n    result["telemetry_status"] = profile["telemetry_status"]\n    result["notebook_elapsed_before_profile_seconds"] = elapsed\n    elapsed = time.monotonic() - config["notebook_start_monotonic"]\n    result["notebook_elapsed_seconds"] = elapsed\n    result["compute_units_estimate"].update({"through_report": elapsed * rate / 3600,\n        "overhead_through_report": (elapsed - training_seconds) * rate / 3600,\n        "diagnostic_profile": profile["seconds_including_export_and_analysis"] * rate / 3600})\n    result["performance_goals"]["notebook_seconds_at_most_1800"] = elapsed <= 1800.\n    result["performance_goal_status"] = "PASS" if all(result["performance_goals"].values()) else "FAIL"\n    result["report_finalized"] = True\n    write_json(output / "result.json", result)\n    write_json(output / "attempt_status.json", {"version": version, "stage": "complete",\n        "timed_training_started": True, "optimizer_updates": updates,\n        "diagnostic_updates": profile["completed_updates"], "profiling_status": result["profiling_status"]})\n    print("Profiling requirements:", result["profiling_status"], "| telemetry:", result["telemetry_status"], flush=True)\n    for name, measurement in profile["measurements"].items():\n        if measurement["status"] != "available":\n            print("MISSING:", name, "|", measurement.get("reason"), flush=True)\n\n\n\n\ndef check_source_access(lock):\n    """Check gated code access before launching expensive parallel corpus preparation."""\n    from huggingface_hub import HfApi, get_hf_file_metadata, hf_hub_url\n    files = HfApi().list_repo_files(STACK, repo_type="dataset", revision=lock[STACK])\n    selected = {}\n    for language in ("java", "c++", "javascript", "rust", "html"):\n        paths = [path for path in files if path.startswith("data/" + language + "/") and path.endswith(".json")]\n        if not paths:\n            raise ValueError("Pinned Stack-smol has no JSON files for " + language)\n        # HEAD, not a full language-file download. Never accept dataset terms on the user\'s behalf.\n        try:\n            get_hf_file_metadata(hf_hub_url(STACK, paths[0], repo_type="dataset", revision=lock[STACK]),\n                token=os.environ.get("HF_TOKEN"))\n        except Exception as error:\n            raise RuntimeError("The expanded corpus requires access to https://huggingface.co/datasets/bigcode/the-stack-smol . "\n                "Accept its access conditions with your Hugging Face account and use a permitted HF_TOKEN. "\n                "No corpus substitution or model training occurred.") from error\n        selected[language] = paths\n    return selected\n\n\ndef context_phase(tokens, short_batch):\n    # Both fixed shapes are compiled before training; schedule starts at token zero.\n    return "long" if tokens % 4000000 >= 3000000 else "short"\n\ndef tokens_to_boundary(tokens, short_batch):\n    position = tokens % 4000000\n    return (3000000 if position < 3000000 else 4000000) - position\n\ndef bounded_dispatch(prefix, rows, batch, proposed, tokens, short_batch):\n    """Switch on the first complete optimizer update crossing a token boundary."""\n    remaining = tokens_to_boundary(tokens, short_batch)\n    if remaining is None:\n        return proposed\n    start = int(completed_prefix(prefix, rows))\n    for count in range(1, proposed + 1):\n        if int(completed_prefix(prefix, rows + count * batch)) - start >= remaining:\n            return count\n    return proposed\n\ndef compiler_memory(executable):\n    try:\n        analysis = executable.memory_analysis()\n        values = {key: int(getattr(analysis, key)) for key in\n            ("argument_size_in_bytes", "output_size_in_bytes", "alias_size_in_bytes", "temp_size_in_bytes")}\n        if any(value < 0 for value in values.values()):\n            return None\n        return values\n    except (AttributeError, TypeError, NotImplementedError):\n        return None\n\ndef compiler_peak(values):\n    if values is None:\n        return None\n    return (values["argument_size_in_bytes"] + values["output_size_in_bytes"]\n        - values["alias_size_in_bytes"] + values["temp_size_in_bytes"])\n\n\n\ndef row_training_arrays(arrays, seed):\n    order = np.random.default_rng(seed).permutation(arrays["tokens"].shape[0])\n    return {key: value[order] for key, value in arrays.items()}, hashlib.sha256(order.tobytes()).hexdigest()\n\ndef row_prefix_counts(arrays):\n    return training_prefix_counts({key: value[:, None, :] for key, value in arrays.items()})\n\n\n\n\n\n\n"""Bound useful candidate storage independently of the upstream source mix."""\n\n\ndef prefix_cache_limit(job):\n    # The 256 MiB corpus is final text. Cached JSON also contains eval, spare\n    # candidates, escaping and provenance; it needs an independent disk budget.\n    return max(PREFIX_JOB_LIMIT_BYTES, 4 * sum(job["targets"].values()))\n\n\ndef select_prefix_records(records, state, desired):\n    """Stage an entire input row without modifying the committed cursor/counts."""\n    counts = dict(state["candidate_bytes"])\n    documents = dict(state["candidate_documents"])\n    selected = []\n    for record in records:\n        stratum = record["split"] + ":" + record["bucket"]\n        if stratum not in desired:\n            continue\n        if counts.get(stratum, 0) >= desired[stratum] and documents.get(stratum, 0) >= 4:\n            continue\n        selected.append(record)\n        counts[stratum] = counts.get(stratum, 0) + record["utf8_bytes"]\n        documents[stratum] = documents.get(stratum, 0) + 1\n    return selected\n\n\n\ndef encoding_cache_key(tokenizer_sha256, raw_digest):\n    value = {"schema": ENCODING_SCHEMA, "tokenizer_sha256": tokenizer_sha256,\n        "raw_manifest_sha256": raw_digest, "languages": LANGUAGES, "domains": DOMAINS,\n        "mapping": "sorted native IDs -> 1..N; virtual BOS/EOS added during packing"}\n    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()[:24]\n\ndef verify_encoding(directory, identity, raw_digest):\n    details = read_json(directory / "encoding.json")\n    if (details.get("schema") != ENCODING_SCHEMA\n            or details.get("cache_key") != encoding_cache_key(identity["tokenizer_sha256"], raw_digest)\n            or details.get("tokenizer_sha256") != identity["tokenizer_sha256"]\n            or details.get("raw_manifest_sha256") != raw_digest\n            or details.get("languages") != LANGUAGES or details.get("domains") != DOMAINS):\n        raise ValueError("Document encoding cache identity differs; cache preserved.")\n    for name, digest in details["files"].items():\n        if sha256_file(directory / name) != digest:\n            raise ValueError("Document encoding checksum differs: " + name)\n    for split, counts in details["splits"].items():\n        for name in ("ids.bin", "byte_weights.bin"):\n            if (directory / split / name).stat().st_size != counts["text_tokens"] * 4:\n                raise ValueError("Encoded binary size differs: " + split + "/" + name)\n    return details\n\ndef encode_corpus_once(raw, destination, tokenizer, mapping, identity, raw_digest):\n    destination.mkdir(parents=True, exist_ok=False)\n    started = time.monotonic()\n    details = {"schema": ENCODING_SCHEMA,\n        "cache_key": encoding_cache_key(identity["tokenizer_sha256"], raw_digest),\n        "tokenizer_sha256": identity["tokenizer_sha256"], "raw_manifest_sha256": raw_digest,\n        "text_encoding_policy": identity["text_encoding_policy"], "byte_accounting": identity["byte_accounting"],\n        "languages": LANGUAGES, "domains": DOMAINS, "splits": {}, "files": {}}\n    for split in ("train", "in_mix", "wiki"):\n        folder = destination / split\n        folder.mkdir()\n        token_count = byte_count = document_count = 0\n        last_progress = time.monotonic()\n        with (raw / (split + ".jsonl")).open(encoding="utf-8") as source, \\\n                (folder / "ids.bin").open("wb", buffering=8 * 1024**2) as ids_file, \\\n                (folder / "byte_weights.bin").open("wb", buffering=8 * 1024**2) as weights_file, \\\n                (folder / "documents.jsonl").open("w", encoding="utf-8") as documents_file:\n            for line in source:\n                doc = json.loads(line)\n                native_ids = tokenizer.encode_ordinary(doc["text"])\n                pieces = tokenizer.decode_tokens_bytes(native_ids)\n                original = doc["text"].encode("utf-8")\n                if b"".join(pieces) != original or len(original) != doc["utf8_bytes"]:\n                    raise ValueError("Native tokenizer is not byte-lossless on fixed input " + doc["sha256"])\n                ids = np.array([mapping[index] for index in native_ids], dtype="<i4")\n                weights = np.fromiter((len(piece) for piece in pieces), dtype="<i4", count=len(pieces))\n                if not len(ids) or len(ids) != len(weights) or int(weights.sum()) != doc["utf8_bytes"]:\n                    raise ValueError("Native token/byte attribution differs: " + doc["sha256"])\n                ids_file.write(ids.tobytes())\n                weights_file.write(weights.tobytes())\n                documents_file.write(json.dumps({"offset": token_count, "tokens": len(ids),\n                    "original_bytes": doc["utf8_bytes"], "sha256": doc["sha256"],\n                    "language_id": LANGUAGES.index(doc["language"]), "domain_id": DOMAINS.index(doc["domain"])}) + "\\n")\n                token_count += len(ids)\n                byte_count += doc["utf8_bytes"]\n                document_count += 1\n                if time.monotonic() - last_progress >= 5.:\n                    print(f"Native encoding once: {split}, {document_count:,} documents, {token_count:,} tokens", flush=True)\n                    last_progress = time.monotonic()\n        details["splits"][split] = {"text_tokens": token_count, "original_bytes": byte_count,\n            "documents": document_count, "encode_passes": 1, "roundtrip_verification_passes": 1}\n        for name in ("ids.bin", "byte_weights.bin", "documents.jsonl"):\n            relative = split + "/" + name\n            details["files"][relative] = sha256_file(destination / relative)\n        print(f"Encoded {split} once: {token_count:,} tokens; {byte_count:,} original bytes", flush=True)\n    details["encoding_seconds"] = time.monotonic() - started\n    write_json(destination / "encoding.json", details)\n    return details\n\ndef prepared_encoding(shared, raw, work, tokenizer, mapping, identity, raw_digest):\n    key = encoding_cache_key(identity["tokenizer_sha256"], raw_digest)\n    cache = shared / "encoded" / key\n    if cache.exists():\n        details = verify_encoding(cache, identity, raw_digest)\n        return cache, details, True\n    local = work / ("encoding-" + key)\n    details = encode_corpus_once(raw, local, tokenizer, mapping, identity, raw_digest)\n    cache.parent.mkdir(parents=True, exist_ok=True)\n    temporary = cache.with_name(key + ".partial-" + str(time.time_ns()))\n    shutil.copytree(local, temporary)\n    verify_encoding(temporary, identity, raw_digest)\n    temporary.rename(cache)\n    # Cold packing reads the fast local files; later runs can use the verified cache.\n    return local, details, False\n\ndef pack_encoded_split(directory, split, identity, sequence_length):\n    started = time.monotonic()\n    details = read_json(directory / "encoding.json")["splits"][split]\n    documents = read_jsonl(directory / split / "documents.jsonl")\n    if len(documents) != details["documents"] or not documents:\n        raise ValueError("Encoded document count differs: " + split)\n    ids = np.memmap(directory / split / "ids.bin", mode="r", dtype="<i4")\n    weights = np.memmap(directory / split / "byte_weights.bin", mode="r", dtype="<i4")\n    chunks = sum((doc["tokens"] + sequence_length) // sequence_length for doc in documents)\n    shape = (chunks, sequence_length)\n    result = {key: np.zeros(shape, np.bool_ if key in ("mask", "text_mask") else np.int32)\n        for key in ("tokens", "targets", "mask", "text_mask", "byte_weights", "language_ids", "domain_ids")}\n    row = expected_offset = 0\n    last_progress = time.monotonic()\n    for doc in documents:\n        offset, count = doc["offset"], doc["tokens"]\n        if offset != expected_offset or count < 1:\n            raise ValueError("Encoded document offsets are not contiguous.")\n        expected_offset += count\n        sequence = np.concatenate(([identity["bos_id"]], ids[offset:offset + count], [identity["eos_id"]]))\n        byte_weights = np.concatenate((weights[offset:offset + count], [0]))\n        for start in range(0, count + 1, sequence_length):\n            size = min(sequence_length, count + 1 - start)\n            result["tokens"][row, :size] = sequence[start:start + size]\n            result["targets"][row, :size] = sequence[start + 1:start + size + 1]\n            result["mask"][row, :size] = True\n            result["text_mask"][row, :size] = start + np.arange(size) < count\n            result["byte_weights"][row, :size] = byte_weights[start:start + size]\n            result["language_ids"][row, :size] = doc["language_id"]\n            result["domain_ids"][row, :size] = doc["domain_id"]\n            row += 1\n        if time.monotonic() - last_progress >= 5.:\n            print(f"Packing {sequence_length}: {split}, {row:,}/{chunks:,} chunks", flush=True)\n            last_progress = time.monotonic()\n    if (row != chunks or expected_offset != details["text_tokens"] or expected_offset != len(ids)\n            or len(weights) != len(ids) or int(result["byte_weights"].sum()) != details["original_bytes"]):\n        raise ValueError("Encoded chunks differ from original targets/bytes.")\n    packed, packing = pack_chunks(result, 42)\n    for key in EXTRA_KEYS:\n        packed[key] = np.zeros_like(packed["tokens"], dtype=result[key].dtype)\n        for source, target, start, length in packing["assignments_source_bin_start_length"]:\n            packed[key][target, start:start + length] = result[key][source, :length]\n    packing.update(text_tokens=details["text_tokens"], original_bytes=int(packed["byte_weights"].sum()),\n        packing_seconds=time.monotonic() - started, encoding_schema=ENCODING_SCHEMA)\n    return packed, packing\n\n\n\n\nO200K_ASSET_SHA256 = "97973057009b00298ee5c575a23b895c912518ded47f9dc1559032e9f2532187"\n\nO200K_RANKS_SHA256 = "446a9538cb6c348e3516120d7c08b09f57c36495e2acfffe59a5bf8b0cfb1a2d"\n\ndef fixed_train(state, training, prefixes, functions, config, output):\n    batches = {"short": 3, "long": 1}\n    policy = {"mode": "fixed", "selected_batches": batches, "discovery_enabled": False,\n        "candidate_compilation": False, "training_time_compilation": False,\n        "context_schedule": "Repeat 3M short / 1M long cumulative scored tokens from token zero."}\n    write_json(output / "batch_policy.json", policy)\n    print("FIXED BATCHES: 3 x 1024 / 1 x 2048; no discovery or batch changes.", flush=True)\n    log_rows, transitions, samples = [], [], []\n    rows_used = {"short": 0, "long": 0}\n    updates = tokens = positions = 0\n    maximum_step_seconds = last_progress = last_start = sampling_seconds = 0.\n    last_context = None\n    memory_before = memory_snapshot(jax.devices()[0].memory_stats(), 0.)\n    with (output / "training.jsonl").open("x") as stream:\n        started = time.monotonic()\n        try:\n            while True:\n                elapsed = time.monotonic() - started\n                remaining = config["training_seconds"] - elapsed\n                if remaining <= 0:\n                    break\n                mode = context_phase(tokens, batches["short"])\n                used_batch = batches[mode]\n                length = 1024 if mode == "short" else 2048\n                requested = config["updates_per_dispatch"]\n                if mode != last_context:\n                    transitions.append({"phase": "fixed_batches", "mode": mode, "batch": used_batch,\n                        "tokens": tokens, "update": updates, "elapsed_seconds": elapsed,\n                        "short_batch": 3, "long_batch": 1, "cycle": tokens // 4000000})\n                    print(f"Context: fixed {used_batch} x {length}; tokens={tokens:,}", flush=True)\n                    last_context = mode\n                if updates == 0 or remaining < max(2., 2 * requested * maximum_step_seconds):\n                    requested = 1\n                requested = bounded_dispatch(prefixes[mode]["tokens"], rows_used[mode], used_batch,\n                    requested, tokens, batches["short"])\n                last_start = time.monotonic() - started\n                if last_start >= config["training_seconds"]:\n                    break\n                begin = time.monotonic()\n                pending = functions[(mode, used_batch)](state, training[mode], jnp.array(requested, jnp.int32))\n                enqueued = time.monotonic()\n                state, values = jax.block_until_ready(pending)\n                end_call = time.monotonic()\n                metrics = {key: np.asarray(value).tolist() for key, value in values.items()}\n                completed = int(metrics["updates"])\n                previous_rows = rows_used[mode]\n                rows_used[mode] += completed * used_batch\n                expected = int(completed_prefix(prefixes[mode]["tokens"], rows_used[mode])\n                    - completed_prefix(prefixes[mode]["tokens"], previous_rows))\n                updates += completed\n                tokens += int(metrics["tokens"])\n                positions += completed * used_batch * length\n                elapsed = end_call - started\n                row = {**metrics, "update": updates, "total_tokens": tokens, "phase": "fixed_batches",\n                    "requested_updates": requested, "mode": mode, "batch": used_batch,\n                    "short_batch": 3, "long_batch": 1, "sequence_length": length,\n                    "start_seconds": last_start, "end_seconds": elapsed, "dispatch_seconds": end_call - begin,\n                    "enqueue_seconds": enqueued - begin, "synchronization_call_seconds": end_call - enqueued,\n                    "loss": metrics["nll_sum"] / max(metrics["tokens"], 1),\n                    "short_rows": rows_used["short"], "long_rows": rows_used["long"]}\n                log_rows.append(row)\n                stream.write(json.dumps({key: None if isinstance(value, float) and not math.isfinite(value) else value\n                    for key, value in row.items()}, allow_nan=False) + "\\n")\n                if not metrics["finite"] or completed != requested or expected != int(metrics["tokens"]):\n                    raise FloatingPointError("Nonfinite/incomplete update or CPU/device accounting mismatch. No retry.")\n                maximum_step_seconds = max(maximum_step_seconds, (end_call - begin) / completed)\n                if len(log_rows) == 1 or elapsed - last_progress >= 5.:\n                    stream.flush()\n                    sample_started = time.monotonic()\n                    samples.append(memory_snapshot(jax.devices()[0].memory_stats(), elapsed))\n                    sampling_seconds += time.monotonic() - sample_started\n                    print(f"updates={updates:,} tokens={tokens:,} batch={used_batch}x{length} loss={row[\'loss\']:.5f} elapsed={elapsed:.2f}s", flush=True)\n                    last_progress = elapsed\n            jax.block_until_ready(state)\n        except Exception:\n            stream.flush()\n            write_json(output / "attempt_status.json", {"version": "V4.2.1", "stage": "training failed",\n                "timed_training_started": True, "optimizer_updates": updates, "tokens": tokens,\n                "completed_short_rows": rows_used["short"], "completed_long_rows": rows_used["long"]})\n            raise\n        seconds = time.monotonic() - started\n    after = memory_snapshot(jax.devices()[0].memory_stats(), seconds)\n    write_json(output / "context_transitions.json", transitions)\n    return state, {"rows": log_rows, "checks": [], "transitions": transitions,\n        "updates": updates, "tokens": tokens, "positions": positions, "rows_used": rows_used,\n        "short_batch": 3, "long_batch": 1, "batch_policy": policy,\n        "training_seconds": seconds, "last_start": last_start, "memory_before": memory_before,\n        "memory_after": after, "samples": samples, "sampling_seconds": sampling_seconds}\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--phase", choices=["preflight", "prepare", "train"], required=True)\n    parser.add_argument("--work", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    args = parser.parse_args()\n    args.output.mkdir(parents=True, exist_ok=True)\n    try:\n        if args.phase == "preflight":\n            info = preflight()\n            write_json(args.output / "preflight.json", info)\n            print(json.dumps({name: info[name] for name in ("platform", "device_kind", "device_count", "jax")}, indent=2))\n        elif args.phase == "prepare":\n            prepare_cached(read_json(args.work / "config.json"), args.work, args.output)\n        else:\n            cache = args.work.parent / "jax-cache-V3"\n            cache.mkdir(parents=True, exist_ok=True)\n            os.environ["JAX_COMPILATION_CACHE_DIR"] = str(cache)\n            os.environ["JAX_PERSISTENT_CACHE_MIN_COMPILE_TIME_SECS"] = "1"\n            try:\n                train_pilot(args.work, args.output)\n            finally:\n                persistent = args.output.parent.parent / "V3-cache" / "jax"\n                try:\n                    persistent.mkdir(parents=True, exist_ok=True)\n                    shutil.copytree(cache, persistent, dirs_exist_ok=True)\n                except OSError as error:\n                    write_json(args.output / "cache_warning.json", {"warning": str(error), "training_retried": False})\n    except Exception:\n        if (args.output / "failure.json").exists():\n            write_json(args.output / ("failure-secondary-" + str(time.time_ns()) + ".json"),\n                {"phase": args.phase, "error": traceback.format_exc(), "automatic_retry": False})\n            raise\n        write_json(args.output / "failure.json", {"phase": args.phase, "error": traceback.format_exc(), "automatic_retry": False})\n        raise\n\n\nif __name__ == "__main__":\n    main()\n'
CONFIG = {'experiment': 'GEM-COLAB-V4.2.1', 'version': 'V4.2.1', 'implementation_revision': 'V4.2.1-native-o200k-fixed-3x1024-1x2048-1', 'requirement_prefix': 'V4.2.1', 'model_id': 'google/gemma-3-270m', 'tokenizer_reference': 'o200k_base', 'tokenizer_revision': 'tiktoken-0.14.0', 'sequence_length': 1024, 'local_batch_per_chip': 3, 'updates_per_dispatch': 16, 'training_seconds': 1200.0, 'minimum_updates': 5, 'seed': 42, 'expected_devices': 1, 'optimizer_label': 'AdamW', 'learning_rate': 0.0003, 'warmup_updates': 5, 'gradient_clip': 1.0, 'weight_decay': 0.01, 'compute_units_per_hour': 2.92, 'checkpoint_kind': 'weights_only', 'profile_dispatches': 4, 'profile_updates_per_dispatch': 8, 'long_sequence_length': 2048, 'memory_budget_fraction': 0.85, 'short_phase_tokens': 3000000, 'long_phase_tokens': 1000000, 'long_batch_per_chip': 1, 'batch_discovery_enabled': False}
CONFIG["tokenizer_reference"] = TOKENIZER_REFERENCE.strip()
CONFIG["tokenizer_revision"] = "tiktoken-0.14.0"
CONFIG["tokenizer_cache_directory"] = str(OUTPUT.parent.parent / "V3-tokenizers" / "tiktoken")
CONFIG["notebook_start_monotonic"] = NOTEBOOK_START_MONOTONIC
CONFIG["setup_seconds"] = SETUP_SECONDS
PROGRAM_PATH = WORK / "pilot_V4_2_1.py"
PROGRAM_PATH.write_text(PROGRAM_SOURCE)
(WORK / "config.json").write_text(json.dumps(CONFIG, indent=2))
shutil.copyfile(PROGRAM_PATH, OUTPUT / "pilot_source.py")
shutil.copyfile(WORK / "config.json", OUTPUT / "requested_config.json")

def run_phase(phase, token=None):
    if phase == "train" and (OUTPUT / "training.started").exists():
        if (OUTPUT / "result.json").exists():
            print("Existing completed attempt; displaying result without retraining.")
            print((OUTPUT / "result.json").read_text())
            return
        raise RuntimeError("This folder already owns an attempt. Prior logs preserved. Inspect attempt_status.json and train.log; do not delete the marker blindly.")
    if phase == "prepare" and (WORK / "prepared" / "resolved_config.json").exists():
        print("Preparation already completed for this run; reusing it.")
        return
    environment = dict(os.environ)
    environment.pop("PYTHONPATH", None)
    environment.pop("HF_TOKEN", None)
    if token:
        environment["HF_TOKEN"] = token
    environment["TOKENIZERS_PARALLELISM"] = "false"
    environment["JAX_TRACEBACK_FILTERING"] = "off"
    environment["HF_HUB_DOWNLOAD_TIMEOUT"] = "30"
    environment["HF_HUB_ETAG_TIMEOUT"] = "30"
    environment["PYTHONUNBUFFERED"] = "1"
    environment["MPLCONFIGDIR"] = str(BASE / "matplotlib-cache")
    command = [str(PYTHON), str(PROGRAM_PATH), "--phase", phase, "--work", str(WORK), "--output", str(OUTPUT)]
    log_path = OUTPUT / (phase + ".log")
    if log_path.exists():
        raise RuntimeError(f"Preserving existing {phase} log; this phase is not retried automatically.")
    with log_path.open("x") as log:
        process = subprocess.Popen(command, env=environment, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        status = process.wait()
    if status:
        raise RuntimeError(f"V4_2_1 {phase} failed. See {log_path} and failure.json. No automatic retry occurred.")


## 4. Encode native o200k once; pack both contexts
Reuse the frozen 256 MiB corpus and holdouts. Verify each document by exact native token-byte reconstruction, then pack 1,024 and 2,048 sequentially. o200k encoding/packing identities are separate from the Gemma caches.


In [ ]:
PREPARATION_START = time.monotonic()
run_phase("prepare", token=HF_TOKEN)
HF_TOKEN = None
PREPARATION_SECONDS = time.monotonic() - PREPARATION_START
print((OUTPUT / "preparation.json").read_text())
print((OUTPUT / "parallel_downloads.json").read_text())


## 5. One 1,200-second run with fixed batches
Compile only 3 ×1,024 and 1 ×2,048 before the timer. Start at 3 ×1,024; repeat 3M short /1M long cumulative scored-token intervals. No discovery, ramp, batch checks or candidate compilation. All updates train the same model and optimizer.


In [ ]:
run_phase("train")


## 6. Read training results and measurement completeness
Historical runs used different batch sizes and training durations.


In [ ]:
from IPython.display import display, Markdown
result = json.loads((OUTPUT / "result.json").read_text())
def show(rows):
    display(Markdown("| Measurement | Result |\n|---|---|\n" + "\n".join(f"| {name} | {value} |" for name, value in rows)))
def fmt(value, digits=3):
    return f"{value:,.{digits}f}" if isinstance(value, (float, int)) else "UNAVAILABLE"

show([
    ("Version / model parameters", f'{result["version"]} / {result["model_parameters"]:,}'),
    ("Tokenizer / native vocabulary IDs", f'{result["tokenizer_identity"]["reference"]} / {result["tokenizer_identity"]["vocabulary_ids"]:,}'),
    ("Tied embedding/head rows × width", f'{result["tokenizer_identity"]["model_rows"]:,} × 640'),
    ("Parameter change from V4", result["vocabulary_adaptation"]["parameter_delta_from_v4"]),
    ("Fixed batches: short / long", f'{result["final_short_batch"]} × 1,024 / {result["final_long_batch"]} × 2,048'),
    ("Batch discovery", "Disabled; fixed batches from update 1"),
    ("Short / long scored tokens", str(result["tokens_by_context"])),
    ("Compilation seconds inside training", fmt(result["adaptive_compilation_seconds_inside_training"])),
    ("Training / learning / performance", f'{result["engineering_status"]} / {result["learning_status"]} / {result["performance_goal_status"]}'),
    ("Profiling requirements / telemetry", f'{result["profiling_status"]} / {result["telemetry_status"]}'),
    ("Actual training seconds", fmt(result["training_seconds"])),
    ("Updates / useful tokens", f'{result["updates"]:,} / {result["scored_tokens"]:,}'),
    ("Useful tokens/s", fmt(result["tokens_per_second"], 1)),
    ("Attributed original bytes/s", fmt(result["original_bytes_per_second_attributed"], 1)),
    ("Useful occupancy", f'{result["training_occupancy"]:.2%}'),
    ("Initial → final FineWiki macro BPB", f'{result["initial_eval"]["wiki"]["language_macro_bpb"]:.6f} → {result["final_eval"]["wiki"]["language_macro_bpb"]:.6f}'),
    ("Initial → final in-mixture pooled BPB", f'{result["initial_eval"]["in_mix"]["pooled_bpb"]:.6f} → {result["final_eval"]["in_mix"]["pooled_bpb"]:.6f}'),
    ("Compilation / initial eval / final eval seconds", f'{result["compilation_seconds"]:.2f} / {result["initial_eval_seconds"]:.2f} / {result["final_eval_seconds"]:.2f}'),
    ("Weights save and file-check seconds", fmt(result["checkpoint_save_verify_seconds"])),
    ("Checkpoint restoration check", "Disabled as requested"),
    ("Whole notebook through report seconds", fmt(result["notebook_elapsed_seconds"])),
    ("Estimated training / total CU at 2.92/hour", f'{result["compute_units_estimate"]["training"]:.5f} / {result["compute_units_estimate"]["through_report"]:.5f}'),
])
memory = result["training_memory"]
show([
    ("Reported device capacity GiB", fmt(memory["device_limit_gib"])),
    ("Sampled maximum active GiB / % capacity", f'{fmt(memory["sampled_peak_active_gib"])} / {fmt(memory["sampled_peak_active_percent"])}%'),
    ("Training allocator peak % / scope", f'{fmt(memory["training_allocator_peak_percent"])}% / {memory["training_allocator_peak_status"]}'),
    ("Compiler + inactive context/evaluation estimate % capacity", fmt(memory["compiled_peak_plus_other_resident_percent"]) + "% (estimate)"),
    ("Memory sampling overhead ms", fmt(memory["sampling_seconds_inside_training"] * 1000)),
])
print("Memory limits:", memory["limitations"])
profile = result["profiling"]
rows = []
for name, measurement in profile["measurements"].items():
    if name == "operation_timings":
        continue
    value = fmt(measurement["value"]) + "%" if measurement["status"] == "available" else "UNAVAILABLE: " + str(measurement["reason"])
    rows.append((name, value))
show(rows)
operations = profile["measurements"]["operation_timings"]
display(Markdown("| Operation category | ms/update | Accounted operation time % |\n|---|---:|---:|\n" + "\n".join(
    f'| {name} | {fmt(values.get("milliseconds_per_update"))} | {fmt(values.get("percent_of_accounted_op_time"))} |'
    for name, values in operations["categories"].items())))
print("Operation timing scope:", operations["scope"])
print("Host enqueue/wait/gaps:", json.dumps(profile["host_timing"], indent=2))
estimates = profile["estimated_utilization"]
if estimates["status"] == "available":
    show([
        ("ESTIMATED compute TFLOP/s / % peak", f'{estimates["estimated_compute_tflops_per_second"]:.2f} / {estimates["estimated_compute_percent_of_peak"]:.2f}%'),
        ("ESTIMATED HBM GiB/s / % peak", f'{estimates["estimated_hbm_gib_per_second"]:.2f} / {estimates["estimated_hbm_percent_of_peak"]:.2f}%'),
        ("Diagnostic updates / total diagnostic seconds", f'{profile["completed_updates"]} / {profile["seconds_including_export_and_analysis"]:.2f}'),
    ])
    print(estimates["note"])
else:
    print("Compiler estimates unavailable:", estimates.get("reason"))
language_rows = []
for language, final in result["final_eval"]["wiki"]["languages"].items():
    initial = result["initial_eval"]["wiki"]["languages"][language]["bpb"]
    language_rows.append(f'| {language} | {initial:.6f} | {final["bpb"]:.6f} | {final["bpb"] - initial:+.6f} |')
display(Markdown("| FineWiki language | Initial BPB | Final BPB | Change |\n|---|---:|---:|---:|\n" + "\n".join(language_rows)))
print("Historical comparison:", json.dumps(result["historical_comparison"], indent=2))
print("Required measurements:", json.dumps(profile["checks"], indent=2))
print("Training requirements:", json.dumps(result["requirements"], indent=2))
print("Performance targets:", json.dumps(result["performance_goals"], indent=2))
print("Timed source byte shares:", json.dumps(result["training_domain_byte_shares"], indent=2))
print("Timed language byte shares:", json.dumps(result["training_language_byte_shares"], indent=2))
print("Artifacts:", OUTPUT)

print("Batch decisions:", json.dumps(result["batch_checks"], indent=2))
print("Context transitions:", json.dumps(result["context_transitions"], indent=2))
print(result["profiling"]["operation_timing_note"])

print("Fixed batch policy:", json.dumps(result["batch_policy"], indent=2))
print("Encoding preparation:", (OUTPUT / "encoding_preparation.json").read_text())


## 7. Download reports and disconnect
The report ZIP excludes the large weights artifact, which remains in Drive with the tokenizer.


In [ ]:
import zipfile
from google.colab import files
REPORT_ZIP = WORK / "gemma_colab_pilot_V4_2_1_reports.zip"
with zipfile.ZipFile(REPORT_ZIP, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUTPUT.iterdir()):
        if path.is_file() and path.suffix in (".json", ".jsonl", ".log", ".py", ".in", ".lock"):
            bundle.write(path, path.name)
    for directory in ("profile", "profile_tools", "tokenizer"):
        for path in sorted((OUTPUT / directory).rglob("*")):
            if path.is_file():
                bundle.write(path, str(path.relative_to(OUTPUT)))
shutil.copyfile(REPORT_ZIP, OUTPUT / REPORT_ZIP.name)
files.download(str(REPORT_ZIP))
print("FP32 weights:", OUTPUT / "weights.npz")
print("Tokenizer files:", OUTPUT / "tokenizer")
print("Finished. Select Runtime > Disconnect and delete runtime when done.")


# V4.2.1 corpus: 256 MiB

All percentages are **original UTF-8 bytes**, not tokens. Quotas can finish
up to **3 bytes short per bucket** to preserve UTF-8 boundaries.

| Source/category | Share | Target MiB |
|---|---:|---:|
| FinePDFs-Edu via shuffled Smol-Data | 25% | 64.00 |
| DCLM via shuffled Smol-Data | 15% | 38.40 |
| FineWeb-Edu via shuffled Smol-Data | 10% | 25.60 |
| FineWeb2 multilingual web | 30% | 76.80 |
| FineMath 3+ | 5% | 12.80 |
| Python (CodeParrot clean) | 3% | 7.68 |
| Java (Stack-smol) | 2% | 5.12 |
| C++ (Stack-smol) | 2% | 5.12 |
| JavaScript (Stack-smol) | 1.5% | 3.84 |
| Rust (Stack-smol) | 1% | 2.56 |
| HTML (Stack-smol) | 0.5% | 1.28 |
| FineTranslations, original + English | 5% | 12.80 |
| **Total** | **100%** | **256.00** |

FineWeb2 allocation, as percentages of the entire training pool:

| Language | Share |
|---|---:|
| French | 2.5% |
| Spanish | 2.0% |
| Portuguese | 1.5% |
| Chinese | 3.5% |
| Arabic | 3.5% |
| Hindi | 3.5% |
| Swahili | 2.0% |
| Zulu | 1.0% |
| Xhosa | 1.0% |
| Setswana | 1.0% |
| Sesotho | 0.8% |
| Russian | 2.5% |
| Japanese | 2.5% |
| German | 1.7% |
| Igbo | 1.0% |
| **Multilingual total** | **30.0%** |

French was already present. Russian, Japanese, German and Igbo expand the
human-language list from **12 to 16**, including English. Translation buckets
remain French/Arabic/Hindi/Swahili at **1.25% each**, containing both the original
and English side, with educational score ≥2 and no early-stop rows. Actual
language shares include English translations and are reported. Code uses six
distinct domain labels; its `eng_Latn` language ID is an accounting placeholder,
not a claim that comments or strings are all English. These are declared
engineering allocations, not measured optimal mixture weights.

Holdout: **64 KiB ×16 FineWiki languages =1 MiB**, plus **1 MiB** in-mixture
allocated by the source percentages. FineWiki configurations are checked at the
pinned dataset revision; unavailable configurations or byte shortages stop
preparation. No fabricated examples, language replacement or quota renormalizing.

Freeze source commits once per new shared recipe. Preserve exact raw byte
hashes, provenance, deterministic group splits and global exact source/span
deduplication. Code repository names from both source schemas are grouped
together. Near duplicates and unidentified cross-repository copies are not
exhaustively removed. Spans are exact UTF-8 prefixes up to **32 KiB**. Bounded
stream-prefix selection is not a uniform sample of the full source collection.

Use at most **6 concurrent source jobs**; persist source batches in Drive.
Initial candidate target is 125% of quota, at least four spans per stratum;
continue a pinned stream if deduplication consumes that buffer. The per-job
candidate JSONL cap is `max(256 MiB, 4 × total job text quotas)`: 514 MiB for the combined English job. Satisfied bucket/split strata are no longer cached while waiting for another stratum. Deduplication refills extend only deficient strata. Existing verified V3.9 prefixes remain reusable. Stack-smol contains a finite 10,000 samples
per language: quota fulfilment is checked, not assumed. Source access and
provenance terms are those of the upstream datasets.

Both 1,024 and 2,048 packings preserve all original text targets, independent
virtual EOS accounting, and document isolation. Each packing has its own cache
key and row order. Both reside on TPU; CPU-only byte/source prefixes are used
to account for the exact rows consumed, including repeat passes. A context
switch resumes its saved row cursor. It does not advance a single common raw
text frontier, so examples may recur across the two packings before either has
completed a full pass. Report context-specific exposure, not unique data seen.

BPB excludes virtual EOS and divides original-text negative log likelihood by
original UTF-8 bytes ×ln(2). Fixed 1,024-token evaluation scores the same held-out
text before/after training. Report 16-language macro BPB plus the original
12-language subset. Expanded recipe, potentially different source revisions,
and longer training prevent a controlled comparison with V3.5/V3.8.

References: [TPU recipes](https://github.com/AI-Hypercomputer/tpu-recipes/tree/main/training),
[MaxText setup guidance](https://github.com/AI-Hypercomputer/tpu-recipes/blob/main/training/MAXTEXT_README.md),
[Shinning-Tokens](https://github.com/godkdf/Shinning-Tokens/),
[FineWeb2](https://huggingface.co/datasets/HuggingFaceFW/fineweb-2),
[Stack-smol](https://huggingface.co/datasets/bigcode/the-stack-smol).
The TPU recipes are largely GKE/multichip setup guidance. V4.2.1 keeps the pinned
single-chip Colab environment; it does not claim to transplant a multichip recipe.

V4.2.1 retains the frozen raw corpus and source identities. Native tiktoken encode_ordinary preserves special-token spellings as ordinary text. Exact per-token byte lengths replace character-offset attribution. Its encoding/packing schema and tokenizer hash keep caches separate. Both training batches are fixed from update 1: 3 ×1,024 and 1 ×2,048, with the repeating 3M/1M token schedule.
